In [97]:
from pathlib import Path
from zipfile import ZipFile
import json
import pandas as pd

# Update this path if your ZIP is stored elsewhere
ZIP_PATH = Path("raw.zip")

EXTRACT_DIR = Path("data/raw/real_estate")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract the archive
with ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extraction completed.")
print("Extraction directory:", EXTRACT_DIR.resolve())

# Inventory all JSON files
json_files = list(EXTRACT_DIR.rglob("*.json"))

inventory = []

for file_path in json_files:
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        if isinstance(data, list):
            record_count = len(data)
            first_record = next(
                (item for item in data if isinstance(item, dict)), {}
            )
            columns = list(first_record.keys())
            root_type = "list"

        elif isinstance(data, dict):
            record_count = None
            columns = list(data.keys())
            root_type = "dict"

        else:
            record_count = None
            columns = []
            root_type = type(data).__name__

        inventory.append({
            "file": str(file_path.relative_to(EXTRACT_DIR)),
            "root_type": root_type,
            "records": record_count,
            "columns": columns
        })

    except (json.JSONDecodeError, UnicodeDecodeError) as error:
        inventory.append({
            "file": str(file_path.relative_to(EXTRACT_DIR)),
            "root_type": "invalid",
            "records": None,
            "columns": [],
            "error": str(error)
        })

inventory_df = pd.DataFrame(inventory)

print("Number of JSON files:", len(inventory_df))
display(inventory_df[["file", "root_type", "records"]])

Extraction completed.
Extraction directory: /home/ruth/projects/house-rental-price-prediction-in-addis/data/raw/real_estate
Number of JSON files: 29


,file,root_type,records
0,raw/engocha/engocha_real-estate-categories_202...,list,11.0
1,raw/engocha/engocha_real-estate_2023-12-28.json,list,1793.0
2,raw/livingethio/livingethio.json,list,813.0
3,raw/zegebeya/residential_2024-02-08.json,list,547.0
4,raw/zegebeya/commercial_2024-02-08.json,list,99.0
5,raw/ethiopiapropertycentre/product_urls.json,list,3341.0
6,raw/ethiopiapropertycentre/product_data_from_f...,list,3342.0
7,raw/qefira/apartments-for-rent_2023_05_29.json,list,1398.0
8,raw/qefira/houses-for-rent_2023_05_30.json,list,714.0
9,raw/qefira/land-for-sale_2023_05_30.json,list,209.0


In [98]:
import json
from pathlib import Path

DATA_DIR = Path("data/raw/real_estate/raw")

files_to_inspect = {
    "jiji": "jiji/houses-apartments-for-rent_2024-03-08.json",
    "qefira_apartments": "qefira/apartments-for-rent_2023_05_29.json",
    "qefira_houses": "qefira/houses-for-rent_2023_05_30.json",
    "realethio": "realethio/realethio_data.json",
    "livingethio": "livingethio/livingethio.json",
    "ethiopian_properties": "ethiopianproperties/any_any_2024_02_04.json",
    "zegebeya": "zegebeya/residential_2024-02-08.json",
}

for source, relative_path in files_to_inspect.items():

    file_path = DATA_DIR / relative_path

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    print(f"\n{'=' * 60}")
    print(f"SOURCE: {source}")
    print(f"TOTAL RECORDS: {len(data)}")

    if isinstance(data, list):
        for index, record in enumerate(data[:2]):
            print(f"\nRECORD {index + 1}")
            print(json.dumps(record, indent=2, ensure_ascii=False)[:2000])


SOURCE: jiji
TOTAL RECORDS: 2296

RECORD 1
{
  "advert": {
    "abuse_reported": false,
    "admin_info": {},
    "attrs": [
      {
        "name": "Property Address",
        "value": "Semit",
        "data_type": "str",
        "unit": null,
        "group_type": "value"
      },
      {
        "name": "Property Size",
        "value": 128,
        "data_type": "int",
        "unit": "sqm",
        "group_type": "value"
      },
      {
        "name": "Property Type",
        "value": "Apartment",
        "data_type": "str",
        "unit": null,
        "group_type": "value"
      },
      {
        "name": "Condition",
        "value": "Newly-Built",
        "data_type": "str",
        "unit": null,
        "group_type": "value"
      },
      {
        "name": "Furnishing",
        "value": "Furnished",
        "data_type": "str",
        "unit": null,
        "group_type": "value"
      },
      {
        "name": "Bedrooms",
        "value": 2,
        "data_type": "int",
   

In [99]:
import json
from pathlib import Path

DATA_DIR = Path("data/raw/real_estate/raw")

files_to_audit = {
    "jiji": "jiji/houses-apartments-for-rent_2024-03-08.json",
    "qefira_apartments": "qefira/apartments-for-rent_2023_05_29.json",
    "qefira_houses": "qefira/houses-for-rent_2023_05_30.json",
    "realethio": "realethio/realethio_data.json",
    "livingethio": "livingethio/livingethio.json",
    "ethiopian_properties": "ethiopianproperties/any_any_2024_02_04.json",
    "zegebeya": "zegebeya/residential_2024-02-08.json",
}

def find_price_fields(obj, path=""):
    """Recursively find fields whose names contain price/currency/rent."""

    results = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{path}.{key}" if path else key

            if any(word in key.lower() for word in [
                "price", "currency", "rent", "period", "rate"
            ]):
                results[current_path] = value

            if isinstance(value, (dict, list)):
                results.update(find_price_fields(value, current_path))

    elif isinstance(obj, list):
        for index, value in enumerate(obj[:5]):
            results.update(find_price_fields(
                value, f"{path}[{index}]"
            ))

    return results


for source, relative_path in files_to_audit.items():

    file_path = DATA_DIR / relative_path

    with open(file_path, "r", encoding="utf-8") as file:
        records = json.load(file)

    print(f"\n{'=' * 70}")
    print(f"SOURCE: {source}")
    print(f"TOTAL RECORDS: {len(records)}")

    for index, record in enumerate(records[:5]):

        print(f"\nRECORD {index + 1}")

        price_fields = find_price_fields(record)

        for key, value in price_fields.items():
            print(f"{key}: {str(value)[:300]}")

        # Show listing category/status fields
        for key in [
            "category",
            "listing_type",
            "property_status",
            "property_type",
            "status",
            "address",
            "title"
        ]:
            if key in record:
                print(f"{key}: {record[key]}")

        # For Jiji, inspect its nested advert fields
        if "advert" in record:
            advert = record["advert"]

            print("Advert keys:", list(advert.keys()))

            print("Category:", advert.get("category_name"))

            print("Attributes:")
            for attr in advert.get("attrs", []):
                print(
                    f"  {attr.get('name')}: "
                    f"{attr.get('value')} "
                    f"{attr.get('unit') or ''}"
                )


SOURCE: jiji
TOTAL RECORDS: 2296

RECORD 1
advert.date_moderated: Thu, 07 Mar 2024 16:35:00 GMT
advert.fb_view_content_data.currency: USD
advert.price: {'value': 95000, 'type': 'Negotiable', 'title': 'ETB 95,000 per month, Negotiable', 'period': 'per month', 'is_closed': False}
advert.price.period: per month
advert.price_history: True
advert.price_obj: {'value': 95000, 'view': 'ETB 95,000, Negotiable', 'period': 'per month', 'bulk': None}
advert.price_obj.period: per month
seller.advert_price: 95000
Advert keys: ['abuse_reported', 'admin_info', 'attrs', 'available_tops_count', 'badge_info', 'can_make_an_offer', 'can_view_contacts', 'category_id', 'category_name', 'category_slug', 'count_images', 'count_views', 'create_advert_like_this_url', 'date', 'date_created', 'date_moderated', 'description', 'discount', 'employer_took_cv', 'fav_count', 'fb_view_content_data', 'guid', 'icon_attributes', 'id', 'images', 'images_data', 'is_active', 'is_closed', 'is_cv', 'is_declined', 'is_fav', 'is_

In [100]:

import json
from pathlib import Path

DATA_DIR = Path("data/raw/real_estate/raw")

FILES = {
    "qefira_apartments": "qefira/apartments-for-rent_2023_05_29.json",
    "qefira_houses": "qefira/houses-for-rent_2023_05_30.json",
    "livingethio": "livingethio/livingethio.json",
    "ethiopian_properties": "ethiopianproperties/any_any_2024_02_04.json",
    "zegebeya": "zegebeya/residential_2024-02-08.json",
}

def get_description(record):
    for key in ["description", "parsed_description", "content"]:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.replace("\n", " ")[:500]
    return "No description available"


def get_status(record):
    # Ignore null or non-dictionary records
    if not isinstance(record, dict):
        return None

    status = record.get("status")

    if isinstance(status, dict):
        return status.get("name")

    return record.get("property_status") or status


for source, relative_path in FILES.items():
    path = DATA_DIR / relative_path

    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    print("\n" + "=" * 80)
    print("SOURCE:", source)

    shown = 0

    invalid_records = 0

    for record in records:

        if not isinstance(record, dict):
            invalid_records += 1
            continue

        status = str(get_status(record) or "").lower()

        if "rent" not in status:
            continue

        # Keep the remaining inspection code here

        # Keep records that appear to be rentals.
        if "rent" not in status:
            continue

        offers = record.get("offers", {})
        if not isinstance(offers, dict):
            offers = {}

        print("\nTITLE:", record.get("title"))
        print("STATUS:", get_status(record))
        print("PROPERTY TYPE:", record.get("property_type"))
        print("CATEGORY:", record.get("category"))
        print("PRICE:", record.get("price"))
        print("CURRENCY:", offers.get("priceCurrency") or record.get("price_prefix"))
        print("DESCRIPTION:", get_description(record))

        shown += 1
        if shown == 10:
            break

    print("\nRental records inspected:", shown)
    print("Invalid records skipped:", invalid_records)



SOURCE: qefira_apartments

Rental records inspected: 0
Invalid records skipped: 1

SOURCE: qefira_houses

Rental records inspected: 0
Invalid records skipped: 0

SOURCE: livingethio

TITLE: FURNISHED APARTMENT FOR RENT
STATUS: For Rent
PROPERTY TYPE: None
CATEGORY: None
PRICE: 166205.71
CURRENCY: USD
DESCRIPTION: <p>Three bedroom fully furnished apartment available for rent around Olimpia </p><p><br></p><p>3 bedroom </p><p>3 bathroom </p><p>1 separate kitchen </p><p>1 salon</p><p>Dining area </p><p>WiFi </p><p>Laundry </p><p>Elevator </p><p>Generator </p><p>Sauna and steam </p><p>Gym</p><p>Parking </p><p>Guards </p><p><br></p><p>Price 3000$ or bank rate </p>

TITLE: House for rent
STATUS: For Rent
PROPERTY TYPE: None
CATEGORY: None
PRICE: 110000
CURRENCY: ETB
DESCRIPTION: <p>&nbsp;G+2 unfurnished house.</p>

TITLE: House for rent
STATUS: For Rent
PROPERTY TYPE: None
CATEGORY: None
PRICE: 80000
CURRENCY: ETB
DESCRIPTION: <p>G+1 unfurnished house.</p>

TITLE: Furnished Villa for rent
ST

In [101]:

import json
import re
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw/real_estate/raw")
FILE_PATH = DATA_DIR / "jiji/houses-apartments-for-rent_2024-03-08.json"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)


def get_attributes(advert):
    """Convert Jiji's attribute list into a dictionary."""
    attributes = {}

    for attr in advert.get("attrs", []):
        if not isinstance(attr, dict):
            continue

        name = attr.get("name")
        value = attr.get("value")

        if name:
            attributes[name.strip().lower()] = value

    return attributes


def extract_number(value):
    """Extract the first numeric value from a field."""
    if value is None:
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(value).replace(",", ""))

    return float(match.group()) if match else None


jiji_listings = []

for record in records:

    if not isinstance(record, dict):
        continue

    advert = record.get("advert")

    if not isinstance(advert, dict):
        continue

    # Verify listing category
    category = str(advert.get("category_name", "")).strip().lower()

    if category != "houses & apartments for rent":
        continue

    # Verify price and rental period
    price_info = advert.get("price")

    if not isinstance(price_info, dict):
        continue

    price = price_info.get("value")
    price_title = str(price_info.get("title", "")).strip()
    period = str(price_info.get("period", "")).strip().lower()

    if "ETB" not in price_title.upper():
        continue

    if period != "per month":
        continue

    if not isinstance(price, (int, float)) or price <= 0:
        continue

    # Extract property attributes
    attrs = get_attributes(advert)

    listing = {
        "source": "jiji",
        "listing_id": advert.get("id") or advert.get("guid"),
        "title": advert.get("title"),
        "property_type": attrs.get("property type"),
        "address": attrs.get("property address"),
        "size_sqm": extract_number(attrs.get("property size")),
        "bedrooms": extract_number(attrs.get("bedrooms")),
        "bathrooms": extract_number(attrs.get("bathrooms")),
        "condition": attrs.get("condition"),
        "furnishing": attrs.get("furnishing"),

        # Preserve source price information
        "raw_price": float(price),
        "raw_price_title": price_title,

        # Provisional target
        "monthly_rent": float(price),
        "currency": "ETB",
        "price_period": "monthly",

        "description": advert.get("description"),
        "date_published": advert.get("date_created"),
    }

    jiji_listings.append(listing)


jiji_df = pd.DataFrame(jiji_listings)

def detect_currency_mentions(description):
    text = str(description).upper()

    usd = re.findall(
        r'(?:USD\s*)?\$\s*[\d,]+(?:\.\d+)?|[\d,]+(?:\.\d+)?\s*USD',
        text
    )

    etb = re.findall(
        r'(?:ETB|BIRR)\s*[\d,]+(?:\.\d+)?',
        text
    )

    return pd.Series({
        "description_mentions_usd": bool(usd),
        "description_mentions_etb": bool(etb),
        "usd_mentions": usd,
        "etb_mentions": etb,
    })

currency_audit = jiji_df["description"].apply(
    detect_currency_mentions
)

jiji_df = pd.concat(
    [jiji_df, currency_audit],
    axis=1
)

print("Description currency audit")
print(
    jiji_df[
        [
            "description_mentions_usd",
            "description_mentions_etb"
        ]
    ].sum()
)

print("Jiji extracted records:", len(jiji_df))
print("\nDataset shape:", jiji_df.shape)

print("\nFirst five records:")
display(jiji_df.head())

print("\nMissing values:")
display(jiji_df.isnull().sum())

print("\nDuplicate listing IDs:")
print(jiji_df["listing_id"].duplicated().sum())

print("\nPrice summary:")
display(jiji_df["monthly_rent"].describe())


Description currency audit
description_mentions_usd    67
description_mentions_etb    41
dtype: int64
Jiji extracted records: 2265

Dataset shape: (2265, 21)

First five records:


,source,listing_id,title,property_type,address,size_sqm,bedrooms,bathrooms,condition,furnishing,...,raw_price_title,monthly_rent,currency,price_period,description,date_published,description_mentions_usd,description_mentions_etb,usd_mentions,etb_mentions
0,jiji,590504,Furnished 2bdrm Apartment in Akaky Kaliti for ...,Apartment,Semit,128.0,2.0,2.0,Newly-Built,Furnished,...,"ETB 95,000 per month, Negotiable",95000.0,ETB,monthly,Furnished House for rent \n.2 bedrooms \n.128 ...,"Fri, 05 May 2023 01:55:56 GMT",False,False,[],[]
1,jiji,1210593,Furnished 2bdrm Apartment in Bole Bolbula for ...,Apartment,Bole Bolbula,90.0,2.0,2.0,Newly-Built,Furnished,...,"ETB 65,000 per month, Negotiable",65000.0,ETB,monthly,Location bole Bolbula\nFor all information cal...,"Thu, 07 Mar 2024 18:36:47 GMT",False,False,[],[]
2,jiji,716385,Furnished 3bdrm Apartment in Bole for rent,Apartment,Bole alem cinema,130.0,3.0,2.0,Newly-Built,Furnished,...,"ETB 75,000 per month",75000.0,ETB,monthly,Nearly alem cinema month 3bed room. It's have ...,"Thu, 20 Jul 2023 03:54:39 GMT",False,False,[],[]
3,jiji,672313,Furnished 2bdrm Apartment in Bole for rent,Apartment,Bole alem cinema,130.0,2.0,2.0,Fairly Used,Furnished,...,"ETB 110,000 per month, Negotiable",110000.0,ETB,monthly,2 bedrooms apartments around alem cinema \n110...,"Sun, 25 Jun 2023 07:07:49 GMT",False,False,[],[]
4,jiji,791213,Furnished 2bdrm Apartment in Bole for rent,Apartment,Summit Georges,130.0,2.0,2.0,Fairly Used,Furnished,...,"ETB 80,000 per month, Negotiable",80000.0,ETB,monthly,Area Summit Georges around saudi embassy{phone...,"Sun, 27 Aug 2023 12:52:18 GMT",False,False,[],[]



Missing values:


source                      0
listing_id                  0
title                       0
property_type               0
address                     0
size_sqm                    0
bedrooms                    0
bathrooms                   0
condition                   0
furnishing                  0
raw_price                   0
raw_price_title             0
monthly_rent                0
currency                    0
price_period                0
description                 0
date_published              0
description_mentions_usd    0
description_mentions_etb    0
usd_mentions                0
etb_mentions                0
dtype: int64


Duplicate listing IDs:
213

Price summary:


count    2.265000e+03
mean     1.355389e+06
std      2.721686e+07
min      9.000000e+02
25%      4.000000e+04
50%      7.000000e+04
75%      1.200000e+05
max      1.200000e+09
Name: monthly_rent, dtype: float64

In [102]:
print("Before deduplication:", len(jiji_df))

jiji_df = jiji_df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
).copy()

print("After deduplication:", len(jiji_df))
print(
    "Remaining duplicate IDs:",
    jiji_df["listing_id"].duplicated().sum()
)

Before deduplication: 2265
After deduplication: 2052
Remaining duplicate IDs: 0


In [103]:
currency_review = jiji_df[
    jiji_df["description_mentions_usd"]
    | jiji_df["description_mentions_etb"]
].copy()

print("Listings mentioning USD or ETB:",
      len(currency_review))

display(
    currency_review[
        [
            "listing_id",
            "title",
            "raw_price",
            "raw_price_title",
            "usd_mentions",
            "etb_mentions",
            "description",
        ]
    ].to_string(index=False)
)

Listings mentioning USD or ETB: 103


" listing_id                                                              title  raw_price                   raw_price_title     usd_mentions                 etb_mentions                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [104]:
import re
import numpy as np
import pandas as pd

def extract_first_usd(description):
    if not isinstance(description, str):
        return np.nan

    patterns = [
        r'\$\s*([\d,]+(?:\.\d+)?)',
        r'([\d,]+(?:\.\d+)?)\s*USD',
    ]

    for pattern in patterns:
        match = re.search(pattern, description, re.IGNORECASE)

        if match:
            return float(match.group(1).replace(",", ""))

    return np.nan


currency_review = jiji_df.copy()

currency_review["description_usd_amount"] = (
    currency_review["description"]
    .apply(extract_first_usd)
)

currency_review["etb_usd_ratio"] = (
    currency_review["raw_price"]
    / currency_review["description_usd_amount"]
)

print("USD mentions:", currency_review["description_mentions_usd"].sum())
print("ETB mentions:", currency_review["description_mentions_etb"].sum())

print("\nUSD amount extraction:")
print(currency_review["description_usd_amount"].describe())

print("\nETB/USD ratio distribution:")
print(currency_review["etb_usd_ratio"].describe())

display(
    currency_review.loc[
        currency_review["description_usd_amount"].notna(),
        [
            "listing_id",
            "raw_price",
            "description_usd_amount",
            "etb_usd_ratio",
            "description",
        ],
    ].sort_values("etb_usd_ratio").to_string(index=False)
)

USD mentions: 63
ETB mentions: 40

USD amount extraction:
count       63.000000
mean      4761.158730
std       9758.958551
min          3.000000
25%       1625.000000
50%       2300.000000
75%       3250.000000
max      58000.000000
Name: description_usd_amount, dtype: float64

ETB/USD ratio distribution:
count     63.000000
mean      62.687575
std       80.094074
min        1.000000
25%       54.798712
50%       56.000000
75%       56.833333
max      666.666667
Name: etb_usd_ratio, dtype: float64


" listing_id  raw_price  description_usd_amount  etb_usd_ratio                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    description\n     642422    58000.0                 58000.0       1.000000            

In [105]:
print("Before:", len(jiji_df))

jiji_df = jiji_df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
).copy()

print("After:", len(jiji_df))
print("Remaining duplicate IDs:", jiji_df["listing_id"].duplicated().sum())

Before: 2052
After: 2052
Remaining duplicate IDs: 0


In [106]:

# --------------------------------------------------
# A. Investigate duplicate listing IDs
# --------------------------------------------------

duplicate_rows = jiji_df[
    jiji_df["listing_id"].duplicated(keep=False)
].sort_values("listing_id")

print("Rows associated with duplicate IDs:", len(duplicate_rows))

print("\nNumber of duplicated listing IDs:")
print(duplicate_rows["listing_id"].nunique())

display(
    duplicate_rows[
        [
            "listing_id",
            "title",
            "property_type",
            "address",
            "size_sqm",
            "bedrooms",
            "monthly_rent",
            "date_published",
        ]
    ].head(30)
)

# Check whether duplicated IDs have conflicting prices
duplicate_summary = (
    duplicate_rows
    .groupby("listing_id")
    .agg(
        occurrences=("listing_id", "size"),
        unique_prices=("monthly_rent", "nunique"),
        unique_addresses=("address", "nunique"),
        unique_sizes=("size_sqm", "nunique"),
        unique_dates=("date_published", "nunique"),
    )
    .sort_values("occurrences", ascending=False)
)

print("\nDuplicate ID summary:")
display(duplicate_summary.head(20))


# --------------------------------------------------
# B. Investigate unusually high rental prices
# --------------------------------------------------

high_price_listings = (
    jiji_df[jiji_df["monthly_rent"] > 1_000_000]
    .sort_values("monthly_rent", ascending=False)
)

print("\nListings above 1,000,000 ETB/month:")
print("Count:", len(high_price_listings))

display(
    high_price_listings[
        [
            "listing_id",
            "title",
            "property_type",
            "address",
            "size_sqm",
            "bedrooms",
            "bathrooms",
            "monthly_rent",
            "date_published",
        ]
    ].head(30)
)


# --------------------------------------------------
# C. Inspect the 20 highest prices
# --------------------------------------------------

print("\nTop 20 rental prices:")
display(
    jiji_df.nlargest(20, "monthly_rent")[
        [
            "listing_id",
            "title",
            "address",
            "size_sqm",
            "bedrooms",
            "monthly_rent",
        ]
    ]
)


Rows associated with duplicate IDs: 0

Number of duplicated listing IDs:
0


,listing_id,title,property_type,address,size_sqm,bedrooms,monthly_rent,date_published



Duplicate ID summary:


,occurrences,unique_prices,unique_addresses,unique_sizes,unique_dates
listing_id,,,,,



Listings above 1,000,000 ETB/month:
Count: 57


,listing_id,title,property_type,address,size_sqm,bedrooms,bathrooms,monthly_rent,date_published
130,532210,"10bdrm Duplex in Addis Abba, Bole for rent",Duplex,bole area,500.0,10.0,10.0,1.200000e+09,"Thu, 23 Mar 2023 19:24:42 GMT"
167,684343,"Furnished 4bdrm Penthouse in Adids Ababa, Bole...",Penthouse,Bole area,350.0,4.0,5.0,4.300000e+08,"Sun, 02 Jul 2023 12:20:10 GMT"
1773,631966,"Furnished 10bdrm House in Addis Abeba, Bole fo...",House,Bole shala,500.0,10.0,10.0,1.100000e+08,"Thu, 01 Jun 2023 12:33:06 GMT"
1825,1080111,2bdrm Shared Apartment in South Wollo for rent,Shared Apartment,Dessie Piyasa,700.0,2.0,2.0,1.000000e+08,"Thu, 04 Jan 2024 10:13:22 GMT"
1732,765915,"Furnished 4bdrm Apartment in አዲሱ ገበያ, Gullele ...",Apartment,አዲሱ ገበያ,744.0,4.0,4.0,6.800000e+07,"Mon, 14 Aug 2023 21:50:05 GMT"
1307,1066113,6bdrm House in Addis Ketema for rent,House,Ccd tafo,1000.0,6.0,5.0,6.300000e+07,"Thu, 28 Dec 2023 17:41:33 GMT"
1731,520393,"Furnished 6bdrm Apartment in Own State, Gullel...",Apartment,Markos Magazines,760.0,6.0,4.0,6.000000e+07,"Tue, 14 Mar 2023 09:13:13 GMT"
1581,940776,"Furnished 3bdrm House in Siyuom, Bole for rent",House,bole ads abeba,250.0,3.0,3.0,4.800000e+07,"Fri, 03 Nov 2023 16:47:12 GMT"
227,1166266,"Furnished 4bdrm House in Ayat Area, Bole for rent",House,For sales,300.0,4.0,3.0,4.600000e+07,"Fri, 16 Feb 2024 20:30:20 GMT"
1372,1056737,"4bdrm House in Ccd, Oromia-Finfinne for rent",House,Tafo,1000.0,4.0,5.0,4.500000e+07,"Sun, 24 Dec 2023 17:05:47 GMT"



Top 20 rental prices:


,listing_id,title,address,size_sqm,bedrooms,monthly_rent
130,532210,"10bdrm Duplex in Addis Abba, Bole for rent",bole area,500.0,10.0,1.200000e+09
167,684343,"Furnished 4bdrm Penthouse in Adids Ababa, Bole...",Bole area,350.0,4.0,4.300000e+08
1773,631966,"Furnished 10bdrm House in Addis Abeba, Bole fo...",Bole shala,500.0,10.0,1.100000e+08
1825,1080111,2bdrm Shared Apartment in South Wollo for rent,Dessie Piyasa,700.0,2.0,1.000000e+08
1732,765915,"Furnished 4bdrm Apartment in አዲሱ ገበያ, Gullele ...",አዲሱ ገበያ,744.0,4.0,6.800000e+07
1307,1066113,6bdrm House in Addis Ketema for rent,Ccd tafo,1000.0,6.0,6.300000e+07
1731,520393,"Furnished 6bdrm Apartment in Own State, Gullel...",Markos Magazines,760.0,6.0,6.000000e+07
1581,940776,"Furnished 3bdrm House in Siyuom, Bole for rent",bole ads abeba,250.0,3.0,4.800000e+07
227,1166266,"Furnished 4bdrm House in Ayat Area, Bole for rent",For sales,300.0,4.0,4.600000e+07
587,1136362,"Furnished 5bdrm Villa in Personal, Bole for rent",Bole bulbula,150.0,5.0,4.500000e+07


In [107]:
import json
import pandas as pd
from pathlib import Path

# Load the original Jiji JSON
jiji_path = Path(
    "data/raw/real_estate/raw/jiji/"
    "houses-apartments-for-rent_2024-03-08.json"
)

with open(jiji_path, "r", encoding="utf-8") as f:
    jiji_raw = json.load(f)

# Select the 20 highest-priced extracted listings
top_listings = (
    jiji_df.nlargest(20, "monthly_rent")
    [["listing_id", "title", "address", "monthly_rent"]]
)

# Index original records by listing ID
raw_by_id = {}

for record in jiji_raw:
    advert = record.get("advert", {})
    listing_id = str(advert.get("id", ""))

    raw_by_id.setdefault(listing_id, []).append(record)

# Inspect original information
for _, row in top_listings.iterrows():
    listing_id = str(row["listing_id"])

    print("\n" + "=" * 80)
    print("LISTING ID:", listing_id)
    print("EXTRACTED PRICE:", row["monthly_rent"])
    print("TITLE:", row["title"])
    print("ADDRESS:", row["address"])

    matching_records = raw_by_id.get(listing_id, [])

    if not matching_records:
        print("Original record not found. Check the ID field.")
        continue

    for record in matching_records[:2]:
        advert = record.get("advert", {})

        print("\nADVERT KEYS:", list(advert.keys()))
        print("CATEGORY:", advert.get("category_name"))
        print("PRICE-RELATED FIELDS:")

        for key, value in advert.items():
            if any(word in key.lower() for word in [
                "price", "currency", "rent", "period", "amount"
            ]):
                print(f"  {key}: {value}")

        print("ATTRIBUTES:")
        for attr in advert.get("attrs", []):
            print(
                f"  {attr.get('name')}: "
                f"{attr.get('value')} "
                f"{attr.get('unit') or ''}"
            )

        print("DESCRIPTION:")
        print(str(advert.get("description", ""))[:1000])


LISTING ID: 532210
EXTRACTED PRICE: 1200000000.0
TITLE: 10bdrm Duplex in Addis Abba, Bole for rent
ADDRESS: bole area

ADVERT KEYS: ['abuse_reported', 'admin_info', 'attrs', 'available_tops_count', 'badge_info', 'can_make_an_offer', 'can_view_contacts', 'category_id', 'category_name', 'category_slug', 'count_images', 'count_views', 'create_advert_like_this_url', 'date', 'date_created', 'date_moderated', 'description', 'discount', 'employer_took_cv', 'fav_count', 'fb_view_content_data', 'guid', 'icon_attributes', 'id', 'images', 'images_data', 'is_active', 'is_closed', 'is_cv', 'is_declined', 'is_fav', 'is_job', 'is_on_moderation', 'is_sent_cv', 'labels', 'labels_data', 'make_an_offer_auth_url', 'message_url', 'price', 'price_history', 'price_obj', 'region_name', 'region_slug', 'region_text', 'safety_tips', 'safety_tips_title', 'services_info', 'shared_data', 'show_apply_discount_button', 'show_edit_discount_button', 'similar_ads_href', 'sold_reported', 'status', 'status_color', 'title

In [108]:

import json
import re
import pandas as pd
from pathlib import Path

jiji_path = Path(
    "data/raw/real_estate/raw/jiji/"
    "houses-apartments-for-rent_2024-03-08.json"
)

with open(jiji_path, "r", encoding="utf-8") as f:
    jiji_raw = json.load(f)

sale_patterns = [
    r"\bfor sale\b",
    r"\bfor sell\b",
    r"\bhouse for sell\b",
    r"\btotal price\b",
    r"\bdown.payment\b",
    r"\bprice\s*[:\-]?\s*\d[\d,]*\s*million\b",
    r"\bselling\b",
    r"\bto buy\b",
]

sale_regex = re.compile("|".join(sale_patterns), flags=re.IGNORECASE)

flagged_records = []

for record in jiji_raw:
    advert = record.get("advert", {})
    description = advert.get("description", "") or ""

    if sale_regex.search(description):
        price_info = advert.get("price", {})

        flagged_records.append({
            "listing_id": advert.get("id"),
            "title": advert.get("title"),
            "address": next(
                (
                    attr.get("value")
                    for attr in advert.get("attrs", [])
                    if attr.get("name") == "Property Address"
                ),
                None
            ),
            "price": price_info.get("value"),
            "description": description[:500],
        })

sale_review_df = pd.DataFrame(flagged_records)

print("Flagged listings:", len(sale_review_df))
print(sale_review_df[
    ["listing_id", "title", "address", "price", "description"]
].head(30).to_string(index=False))


Flagged listings: 28
 listing_id                                                title                 address      price                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           description
     532210           10bdrm Duplex in Addis Abba, Bole for rent               bole area 1200000000                                                                                                                                                                                                                                                                      

In [109]:

import pandas as pd

raw_details = []

for record in jiji_raw:
    advert = record.get("advert", {})

    attrs = {
        attr.get("name"): attr.get("value")
        for attr in advert.get("attrs", [])
    }

    raw_details.append({
        "listing_id": str(advert.get("id", "")),
        "description": advert.get("description", ""),
        "region_name": advert.get("region_name", ""),
        "region_text": advert.get("region_text", ""),
        "estate_name": attrs.get("Estate Name", ""),
    })

raw_details_df = pd.DataFrame(raw_details)

jiji_clean = jiji_df.copy()
jiji_clean["listing_id"] = jiji_clean["listing_id"].astype(str)

jiji_clean = jiji_clean.merge(
    raw_details_df.drop_duplicates("listing_id"),
    on="listing_id",
    how="left"
)

print("Enriched shape:", jiji_clean.shape)



Enriched shape: (2052, 25)


In [110]:

confirmed_sale_ids = {
    "532210", "1136362", "874334", "1197153",
    "761239", "747838", "1016969", "829855",
    "980321", "621886", "788234", "1057061",
    "1092515", "1080111", "774992"
}

manual_review_ids = {
    "1027143", "1051915", "936323",
    "1027611", "562677", "1002604"
}

# Exclude confirmed sale listings
jiji_clean = jiji_clean[
    ~jiji_clean["listing_id"].isin(confirmed_sale_ids)
].copy()

# Exclude records clearly located outside Addis Ababa.
# We use multiple location fields, not address alone.
outside_addis_patterns = [
    "bishoftu",
    "bishf",
    "dukem",
    "adama",
    "dessie",
    "south wollo",
    "east shewa",
]

location_text = (
    jiji_clean["title"].fillna("") + " " +
    jiji_clean["address"].fillna("") + " " +
    jiji_clean["region_name"].fillna("") + " " +
    jiji_clean["region_text"].fillna("")
).str.lower()

outside_addis_mask = location_text.apply(
    lambda value: any(
        location in value for location in outside_addis_patterns
    )
)

jiji_clean = jiji_clean[~outside_addis_mask].copy()

print("Records after initial exclusions:", len(jiji_clean))
print(
    "Manual-review records still present:",
    jiji_clean["listing_id"].isin(manual_review_ids).sum()
)


Records after initial exclusions: 1986
Manual-review records still present: 3


In [111]:

additional_exclusions = {
    "1027611",
    "562677",
    "1002604",
}

jiji_clean = jiji_clean[
    ~jiji_clean["listing_id"].isin(additional_exclusions)
].copy()

print("Records after sale/ambiguous exclusions:", len(jiji_clean))


Records after sale/ambiguous exclusions: 1983


In [112]:

additional_exclusions = {
    "1027611",
    "562677",
    "1002604",
}

jiji_clean = jiji_clean[
    ~jiji_clean["listing_id"].isin(additional_exclusions)
].copy()

print("Records after sale/ambiguous exclusions:", len(jiji_clean))


Records after sale/ambiguous exclusions: 1983


In [113]:

print(
    jiji_clean["monthly_rent"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\nHighest 20 remaining rents:")

print(
    jiji_clean.nlargest(20, "monthly_rent")[
        ["listing_id", "title", "address", "monthly_rent",
         "size_sqm", "bedrooms"]
    ].to_string(index=False)
)


count    1.983000e+03
mean     7.643467e+05
std      1.064216e+07
min      9.000000e+02
1%       6.410000e+03
5%       1.600000e+04
25%      4.000000e+04
50%      7.500000e+04
75%      1.210000e+05
95%      2.800000e+05
99%      1.351170e+07
max      4.300000e+08
Name: monthly_rent, dtype: float64

Highest 20 remaining rents:
listing_id                                                             title                     address  monthly_rent  size_sqm  bedrooms
    684343           Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent                   Bole area   430000000.0     350.0       4.0
    631966              Furnished 10bdrm House in Addis Abeba, Bole for rent                  Bole shala   110000000.0     500.0      10.0
    765915            Furnished 4bdrm Apartment in አዲሱ ገበያ, Gullele for rent                     አዲሱ ገበያ    68000000.0     744.0       4.0
   1066113                              6bdrm House in Addis Ketema for rent                    Ccd tafo    63000000

In [114]:

import pandas as pd

# Build an audit table directly from the original Jiji records
audit_records = []

for record in jiji_raw:
    advert = record.get("advert", {})
    price = advert.get("price") or {}
    price_obj = advert.get("price_obj") or {}

    audit_records.append({
        "listing_id": str(advert.get("id", "")),
        "raw_price_value": price.get("value"),
        "raw_price_title": price.get("title"),
        "raw_price_period": price.get("period"),
        "price_obj_title": price_obj.get("title"),
        "price_obj_period": price_obj.get("period"),
        "region_name": advert.get("region_name"),
        "region_text": advert.get("region_text"),
        "description": advert.get("description", ""),
    })

raw_audit_df = pd.DataFrame(audit_records)

# Join raw metadata to the cleaned dataset

price_audit = jiji_clean.merge(
    raw_audit_df.drop_duplicates("listing_id"),
    on="listing_id",
    how="left",
    suffixes=("_clean", "_raw")
)

high_price_audit = price_audit[
    price_audit["monthly_rent"] > 1_000_000
].copy()

print("Listings above 1 million ETB:", len(high_price_audit))

columns_to_show = [
    "listing_id",
    "title",
    "address",
    "region_name_raw",
    "region_text_raw",
    "monthly_rent",
    "raw_price_title",
    "raw_price_period",
    "description",
]

for _, row in high_price_audit.sort_values(
    "monthly_rent", ascending=False
).iterrows():

    print("\n" + "=" * 90)

    for column in columns_to_show:
        print(f"{column}: {row.get(column, 'NOT AVAILABLE')}")


Listings above 1 million ETB: 44

listing_id: 684343
title: Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent
address: Bole area
region_name_raw: Bole
region_text_raw: Addis Ababa, Bole
monthly_rent: 430000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description: Luxury 4 bedroom penthouse for rent in bole area

listing_id: 631966
title: Furnished 10bdrm House in Addis Abeba, Bole for rent
address: Bole shala
region_name_raw: Bole
region_text_raw: Addis Ababa, Bole
monthly_rent: 110000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description: G+5 fully furnished huose rental 20 bedrooms 25 bathrooms parikg space and elevator available bole atlas area one floor 5bedrooms and 5bathrooms

listing_id: 765915
title: Furnished 4bdrm Apartment in አዲሱ ገበያ, Gullele for rent
address: አዲሱ ገበያ
region_name_raw: Gullele
region_text_raw: Addis Ababa, Gullele
monthly_rent: 68000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description: አዲ

In [115]:

print("Region name distribution:")
print(jiji_clean["region_name"].value_counts(dropna=False).head(30))

print("\nRegion text distribution:")
print(jiji_clean["region_text"].value_counts(dropna=False).head(30))

print("\nListings with suspicious addresses:")
print(
    jiji_clean[
        jiji_clean["address"].fillna("").str.lower().str.contains(
            r"for sale|for sales|home|^2$|legetafo|tafo",
            regex=True
        )
    ][
        ["listing_id", "title", "address", "region_name", "region_text"]
    ].to_string(index=False)
)


Region name distribution:
region_name
Bole                1391
Yeka                 295
Kirkos                99
Nifas Silk-Lafto      70
Lideta                38
Akaky Kaliti          32
Kolfe Keranio         17
Arada                 17
Oromia-Finfinne        8
Gullele                7
Addis Ketema           4
Sidama                 2
Bahir Dar              1
Anuak                  1
North Shewa            1
Name: count, dtype: int64

Region text distribution:
region_text
Addis Ababa, Bole                                        1391
Addis Ababa, Yeka                                         295
Addis Ababa, Kirkos                                        99
Addis Ababa, Nifas Silk-Lafto                              70
Addis Ababa, Lideta                                        38
Addis Ababa, Akaky Kaliti                                  32
Addis Ababa, Kolfe Keranio                                 17
Addis Ababa, Arada                                         17
Oromia, Oromia-Finfinne   

In [116]:

print("Available description columns:")
print([
    col for col in price_audit.columns
    if "description" in col.lower()
])

# Prefer the original description from the raw audit data
description_column = (
    "description_raw"
    if "description_raw" in price_audit.columns
    else "description_clean"
)

high_price_audit = price_audit[
    price_audit["monthly_rent"] > 1_000_000
].copy()

print("\nHigh-price rows:", len(high_price_audit))
print(
    "Unique high-price listing IDs:",
    high_price_audit["listing_id"].nunique()
)

columns_to_show = [
    "listing_id",
    "title",
    "address",
    "region_name_raw",
    "monthly_rent",
    "raw_price_title",
    "raw_price_period",
    description_column,
]

for _, row in high_price_audit.sort_values(
    "monthly_rent", ascending=False
).iterrows():

    print("\n" + "=" * 100)

    for column in columns_to_show:
        print(f"{column}: {row.get(column, 'NOT AVAILABLE')}")


Available description columns:
['description_x', 'description_mentions_usd', 'description_mentions_etb', 'description_y', 'description']

High-price rows: 44
Unique high-price listing IDs: 44

listing_id: 684343
title: Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent
address: Bole area
region_name_raw: Bole
monthly_rent: 430000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description_clean: NOT AVAILABLE

listing_id: 631966
title: Furnished 10bdrm House in Addis Abeba, Bole for rent
address: Bole shala
region_name_raw: Bole
monthly_rent: 110000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description_clean: NOT AVAILABLE

listing_id: 765915
title: Furnished 4bdrm Apartment in አዲሱ ገበያ, Gullele for rent
address: አዲሱ ገበያ
region_name_raw: Gullele
monthly_rent: 68000000.0
raw_price_title: NOT AVAILABLE
raw_price_period: per month
description_clean: NOT AVAILABLE

listing_id: 1066113
title: 6bdrm House in Addis Ketema for rent
address: Ccd tafo
re

In [117]:

# Keep records explicitly labelled as Addis Ababa
addis_mask = (
    jiji_clean["region_text"]
    .fillna("")
    .str.startswith("Addis Ababa,")
)

jiji_addis = jiji_clean[addis_mask].copy()

print("Before geographic filtering:", len(jiji_clean))
print("After geographic filtering:", len(jiji_addis))
print("Excluded:", len(jiji_clean) - len(jiji_addis))

print("\nRemaining regions:")
print(jiji_addis["region_text"].value_counts())


Before geographic filtering: 1983
After geographic filtering: 1970
Excluded: 13

Remaining regions:
region_text
Addis Ababa, Bole                1391
Addis Ababa, Yeka                 295
Addis Ababa, Kirkos                99
Addis Ababa, Nifas Silk-Lafto      70
Addis Ababa, Lideta                38
Addis Ababa, Akaky Kaliti          32
Addis Ababa, Kolfe Keranio         17
Addis Ababa, Arada                 17
Addis Ababa, Gullele                7
Addis Ababa, Addis Ketema           4
Name: count, dtype: int64


In [118]:
confirmed_sale_ids = {
    "765915",
    "940776",
    "1166266",
    "728148",
    "636570",
    "1136395",
    "1001405",
    "997341",
    "888070",
    "562696",
    "1041109",
    "868059",
    "839983",
    "818569",
    "936306",
    "569646",
    "1166259",
    "1125132",
    "1059458",
    "1108604",
    "983712",
    "1182763",
    "740939",
    "1145023",
    "1023094",
    "1021191",
    "600978",
    "1149032",
    "1126924",
    "663491",
    "655681",
    "922438",
}

In [119]:
jiji_clean["listing_id"] = jiji_clean["listing_id"].astype(str)

before = len(jiji_clean)

jiji_clean = jiji_clean[
    ~jiji_clean["listing_id"].isin(confirmed_sale_ids)
].copy()

print("Records before:", before)
print("Records after:", len(jiji_clean))
print("Excluded:", before - len(jiji_clean))

Records before: 1983
Records after: 1951
Excluded: 32


In [120]:
high_price = (
    jiji_clean[jiji_clean["monthly_rent"] > 1_000_000]
    .sort_values("monthly_rent", ascending=False)
)

print("Remaining high-price records:", len(high_price))

columns_to_inspect = [
    "listing_id",
    "title",
    "address",
    "monthly_rent",
    "currency",
    "price_period",
    "description_clean",
    "description_raw",
]

available_columns = [
    col for col in columns_to_inspect
    if col in high_price.columns
]

print(
    high_price[available_columns].to_string(index=False)
)

Remaining high-price records: 12
listing_id                                                             title                address  monthly_rent currency price_period
    684343           Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent              Bole area   430000000.0      ETB      monthly
    631966              Furnished 10bdrm House in Addis Abeba, Bole for rent             Bole shala   110000000.0      ETB      monthly
   1066113                              6bdrm House in Addis Ketema for rent               Ccd tafo    63000000.0      ETB      monthly
    520393          Furnished 6bdrm Apartment in Own State, Gullele for rent       Markos Magazines    60000000.0      ETB      monthly
   1056737                      4bdrm House in Ccd, Oromia-Finfinne for rent                   Tafo    45000000.0      ETB      monthly
    895571                                      8bdrm House in Bole for rent                Hayat 5    40000000.0      ETB      monthly
    934714 Furn

In [121]:
confirmed_exclusions = {
    "1066113",  # Likely sale
    "1056737",  # Likely sale / outside study boundary
    "947419",   # Sululta, outside Addis boundary
    "1202878",  # Commercial hotel/office rental
}

jiji_clean["listing_id"] = jiji_clean["listing_id"].astype(str)

before = len(jiji_clean)

# Remove confirmed exclusions
jiji_clean = jiji_clean[
    ~jiji_clean["listing_id"].isin(confirmed_exclusions)
].copy()

# Remove exact duplicate rows
jiji_clean = jiji_clean.drop_duplicates()

print("Records before:", before)
print("Records after:", len(jiji_clean))
print("Records removed:", before - len(jiji_clean))
print("Remaining high-price records:",
      (jiji_clean["monthly_rent"] > 1_000_000).sum())

TypeError: unhashable type: 'list'

In [122]:
review_ids = {
    "684343",
    "631966",
    "520393",
    "895571",
    "934714",
    "945021",
    "1062056",
    "884272",
}

print(
    jiji_clean[
        jiji_clean["listing_id"].isin(review_ids)
    ][["listing_id", "title", "address", "monthly_rent"]]
    .sort_values("monthly_rent", ascending=False)
    .to_string(index=False)
)

listing_id                                                             title          address  monthly_rent
    684343           Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent        Bole area   430000000.0
    631966              Furnished 10bdrm House in Addis Abeba, Bole for rent       Bole shala   110000000.0
    520393          Furnished 6bdrm Apartment in Own State, Gullele for rent Markos Magazines    60000000.0
    895571                                      8bdrm House in Bole for rent          Hayat 5    40000000.0
    934714 Furnished 3bdrm House in Betel Sen Selasa, Kolfe Keranio for rent Betel sen selasa    25000000.0
    945021                        Furnished 4bdrm Apartment in Bole for rent           Samiti    13000000.0
   1062056              3bdrm Apartment in Phison Real Estate, Bole for rent      Ayat zone 5    11200000.0
    884272                         4bdrm House in Condominium, Bole for rent            Samit    11000000.0


In [123]:
import json
from pathlib import Path

review_ids = {
    "684343", "631966", "520393", "895571",
    "934714", "945021", "1062056", "884272"
}

jiji_dir = Path("data/raw/real_estate/raw/jiji")

files = list(
    jiji_dir.glob("houses-apartments-for-rent_2024-03-08*.json")
)

print("Matching files:", files)

Matching files: [PosixPath('data/raw/real_estate/raw/jiji/houses-apartments-for-rent_2024-03-08.json')]


In [124]:
with open(files[0], "r", encoding="utf-8") as f:
    raw_records = json.load(f)

print("Raw records:", len(raw_records))
print("Sample record keys:", raw_records[0].keys())
print("Sample advert keys:", raw_records[0].get("advert", {}).keys())

Raw records: 2296
Sample record keys: dict_keys(['advert', 'seller'])
Sample advert keys: dict_keys(['abuse_reported', 'admin_info', 'attrs', 'available_tops_count', 'badge_info', 'can_make_an_offer', 'can_view_contacts', 'category_id', 'category_name', 'category_slug', 'count_images', 'count_views', 'create_advert_like_this_url', 'date', 'date_created', 'date_moderated', 'description', 'discount', 'employer_took_cv', 'fav_count', 'fb_view_content_data', 'guid', 'icon_attributes', 'id', 'images', 'images_data', 'is_active', 'is_closed', 'is_cv', 'is_declined', 'is_fav', 'is_job', 'is_on_moderation', 'is_sent_cv', 'labels', 'labels_data', 'make_an_offer_auth_url', 'message_url', 'price', 'price_history', 'price_obj', 'region_name', 'region_slug', 'region_text', 'safety_tips', 'safety_tips_title', 'services_info', 'shared_data', 'show_apply_discount_button', 'show_edit_discount_button', 'similar_ads_href', 'sold_reported', 'status', 'status_color', 'title', 'title_labels', 'url'])


In [125]:
for record in raw_records:
    advert = record.get("advert", record)

    listing_id = str(
        advert.get("id", advert.get("listing_id", ""))
    )

    if listing_id in review_ids:
        print("\n" + "=" * 80)
        print("LISTING ID:", listing_id)
        print("TITLE:", advert.get("title"))
        print("DESCRIPTION:", advert.get("description"))


LISTING ID: 684343
TITLE: Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent
DESCRIPTION: Luxury 4 bedroom penthouse for rent in bole area

LISTING ID: 1062056
TITLE: 3bdrm Apartment in Phison Real Estate, Bole for rent
DESCRIPTION: Welcome to our stunning 3-bedroom apartments, a true embodiment of modern luxury and sophistication. Nestled in a prime location, these apartments offer the perfect balance between convenience and tranquility.
Don't miss the opportunity to make one of these stunning 3-bedroom apartments your own. Contact us today to schedule a viewing and experience the epitome of luxury living.

LISTING ID: 1062056
TITLE: 3bdrm Apartment in Phison Real Estate, Bole for rent
DESCRIPTION: Welcome to our stunning 3-bedroom apartments, a true embodiment of modern luxury and sophistication. Nestled in a prime location, these apartments offer the perfect balance between convenience and tranquility.
Don't miss the opportunity to make one of these stunning 3-bedroom apartmen

In [126]:
review_ids = {
    "684343",
    "631966",
    "520393",
    "895571",
    "934714",
    "945021",
    "1062056",
    "884272",
}

for record in raw_records:
    advert = record.get("advert", {})
    listing_id = str(advert.get("id", ""))

    if listing_id in review_ids:
        print("\n" + "=" * 80)
        print("LISTING ID:", listing_id)
        print("TITLE:", advert.get("title"))
        print("REGION:", advert.get("region_text"))
        print("PRICE:", advert.get("price"))
        print("PRICE OBJECT:", advert.get("price_obj"))
        print("DATE:", advert.get("date_created"))


LISTING ID: 684343
TITLE: Furnished 4bdrm Penthouse in Adids Ababa, Bole for rent
REGION: Addis Ababa, Bole
PRICE: {'value': 430000000, 'type': 'Negotiable', 'title': 'ETB 430,000,000 per month, Negotiable', 'period': 'per month', 'is_closed': False}
PRICE OBJECT: {'value': 430000000, 'view': 'ETB 430,000,000, Negotiable', 'period': 'per month', 'bulk': None}
DATE: Sun, 02 Jul 2023 12:20:10 GMT

LISTING ID: 1062056
TITLE: 3bdrm Apartment in Phison Real Estate, Bole for rent
REGION: Addis Ababa, Bole
PRICE: {'value': 11200000, 'type': 'Amount', 'title': 'ETB 11,200,000 per month', 'period': 'per month', 'is_closed': False}
PRICE OBJECT: {'value': 11200000, 'view': 'ETB 11,200,000', 'period': 'per month', 'bulk': None}
DATE: Wed, 27 Dec 2023 04:57:20 GMT

LISTING ID: 1062056
TITLE: 3bdrm Apartment in Phison Real Estate, Bole for rent
REGION: Addis Ababa, Bole
PRICE: {'value': 11200000, 'type': 'Amount', 'title': 'ETB 11,200,000 per month', 'period': 'per month', 'is_closed': False}
PRIC

In [127]:
# Identify suspicious high-price records
high_price_mask = jiji_clean["monthly_rent"] > 1_000_000

# Preserve them separately for auditing
jiji_review = jiji_clean[high_price_mask].copy()

# Keep only records that pass the current price screening
jiji_model = jiji_clean[~high_price_mask].copy()

print("Records before price screening:", len(jiji_clean))
print("Records quarantined:", len(jiji_review))
print("Records retained:", len(jiji_model))

print(
    "Maximum retained monthly rent:",
    jiji_model["monthly_rent"].max()
)

Records before price screening: 1947
Records quarantined: 8
Records retained: 1939
Maximum retained monthly rent: 840000.0


In [128]:
duplicate_ids = (
    jiji_model[
        jiji_model.duplicated(
            subset=["listing_id"],
            keep=False
        )
    ]
    .sort_values("listing_id")
)

print("Rows with duplicated listing IDs:", len(duplicate_ids))
print(
    "Number of duplicated listing IDs:",
    duplicate_ids["listing_id"].nunique()
)

print(duplicate_ids.to_string(index=False))

Rows with duplicated listing IDs: 0
Number of duplicated listing IDs: 0
Empty DataFrame
Columns: [source, listing_id, title, property_type, address, size_sqm, bedrooms, bathrooms, condition, furnishing, raw_price, raw_price_title, monthly_rent, currency, price_period, description_x, date_published, description_mentions_usd, description_mentions_etb, usd_mentions, etb_mentions, description_y, region_name, region_text, estate_name]
Index: []


In [129]:
jiji_summary = {
    "Source": "Jiji",
    "Records after geographic and sale filtering": len(jiji_clean),
    "Records quarantined for suspicious prices": len(jiji_review),
    "Records retained for modeling": len(jiji_model),
    "Duplicate listing IDs": jiji_model["listing_id"].duplicated().sum(),
    "Missing monthly rent": jiji_model["monthly_rent"].isna().sum(),
    "Missing currency": jiji_model["currency"].isna().sum(),
    "Missing price period": jiji_model["price_period"].isna().sum(),
}

for key, value in jiji_summary.items():
    print(f"{key}: {value}")

Source: Jiji
Records after geographic and sale filtering: 1947
Records quarantined for suspicious prices: 8
Records retained for modeling: 1939
Duplicate listing IDs: 0
Missing monthly rent: 0
Missing currency: 0
Missing price period: 0


In [130]:
print(
    jiji_model["monthly_rent"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

count      1939.000000
mean      90893.765859
std       74559.658135
min         900.000000
1%         6190.000000
5%        16000.000000
25%       40000.000000
50%       70000.000000
75%      120000.000000
95%      247500.000000
99%      350000.000000
max      840000.000000
Name: monthly_rent, dtype: float64


In [131]:
# Lowest 20 monthly rents
print("LOWEST 20 LISTINGS")

print(
    jiji_model.nsmallest(20, "monthly_rent")[
        [
            "listing_id",
            "title",
            "address",
            "property_type",
            "size_sqm",
            "bedrooms",
            "monthly_rent",
        ]
    ].to_string(index=False)
)

# Highest 20 monthly rents
print("\nHIGHEST 20 LISTINGS")

print(
    jiji_model.nlargest(20, "monthly_rent")[
        [
            "listing_id",
            "title",
            "address",
            "property_type",
            "size_sqm",
            "bedrooms",
            "monthly_rent",
        ]
    ].to_string(index=False)
)

LOWEST 20 LISTINGS
listing_id                                                                 title                     address       property_type  size_sqm  bedrooms  monthly_rent
   1105007        Furnished 3bdrm House in Africa Cdc, Nifas Silk-Lafto for rent               haile garment               House     150.0       3.0         900.0
   1193970                                Furnished 3bdrm House in Bole for rent Bole Michael nedaji maheber               House     500.0       3.0        1000.0
   1115558 Furnished 2bdrm Apartment in Boren R/State, Nifas Silk-Lafto for rent             Bisrate Gabriel           Apartment     150.0       2.0        1500.0
    266618                            Furnished 3bdrm Apartment in Yeka for Rent                   kazanchis           Apartment     115.0       3.0        1500.0
   1034902                                4bdrm House in Ferensay, Yeka for rent                    ferensay               House     400.0       4.0        1800.0
   

In [133]:
print("CURRENCY VALUES")
print(jiji_model["currency"].value_counts(dropna=False))

print("\nPRICE PERIOD VALUES")
print(jiji_model["price_period"].value_counts(dropna=False))

print("\nPROPERTY TYPES")
print(jiji_model["property_type"].value_counts(dropna=False))

CURRENCY VALUES
currency
ETB    1939
Name: count, dtype: int64

PRICE PERIOD VALUES
price_period
monthly    1939
Name: count, dtype: int64

PROPERTY TYPES
property_type
Apartment              1310
House                   356
Condo                    95
Villa                    65
Studio Apartment         33
Penthouse                29
Townhouse / Terrace      15
Duplex                   15
Mansion                   9
Block of Flats            4
Shared Apartment          2
Bedsitter                 2
Bungalow                  2
Mini Flat                 1
Room & Parlour            1
Name: count, dtype: int64


In [134]:
# Collect suspicious listing IDs
lowest_ids = (
    jiji_model.nsmallest(10, "monthly_rent")["listing_id"]
    .astype(str)
    .tolist()
)

highest_ids = (
    jiji_model.nlargest(20, "monthly_rent")["listing_id"]
    .astype(str)
    .tolist()
)

inspect_ids = set(lowest_ids + highest_ids)

# Inspect original JSON records
for record in raw_records:
    advert = record.get("advert", {})
    listing_id = str(advert.get("id", ""))

    if listing_id in inspect_ids:
        print("\n" + "=" * 90)
        print("LISTING ID:", listing_id)
        print("TITLE:", advert.get("title"))
        print("REGION:", advert.get("region_text"))
        print("PRICE:", advert.get("price"))
        print("DESCRIPTION:", advert.get("description"))

        print("\nATTRIBUTES:")
        for attr in advert.get("attrs", []):
            print(
                f"{attr.get('name')}: "
                f"{attr.get('value')} "
                f"{attr.get('unit') or ''}"
            )


LISTING ID: 1078603
TITLE: Furnished 4bdrm Townhouse / Terrace in Bole for rent
REGION: Addis Ababa, Bole
PRICE: {'value': 420000, 'type': 'Amount', 'title': 'ETB 420,000 per month', 'period': 'per month', 'is_closed': False}
DESCRIPTION: ።g+3 home Rent imperial ።4beds and 6baths ።5living room ።heating fire extinguisher wifi ።dedicated work space ።kitchen ።cinima ።bathtup ።steam and sauna ።7500USD

ATTRIBUTES:
Property Address: imperial 
Property Size: 450 sqm
Property Type: Townhouse / Terrace 
Condition: Newly-Built 
Furnishing: Furnished 
Bedrooms: 4 
Bathrooms: 6 
Facilities: ['24-hour Electricity', 'Balcony', 'Chandelier', 'Dining Area', 'Dishwasher', 'Hot Water', 'Kitchen Cabinets', 'Kitchen Shelf', 'Microwave', 'Pop Ceiling', 'Refrigerator', 'Tiled Floor', 'TV', 'Wardrobe', 'Wi-Fi'] 

LISTING ID: 1055032
TITLE: Furnished 9bdrm House in Adis Abba, Bole for rent
REGION: Addis Ababa, Bole
PRICE: {'value': 400000, 'type': 'Negotiable', 'title': 'ETB 400,000 per month, Negotiable', 

In [135]:
# Records requiring price or attribute verification
review_ids = {
    # Low-price records
    "1105007",
    "1193970",
    "1115558",
    "266618",
    "1034902",
    "1040000",
    "1174252",
    "876089",
    "992063",
    "992492",

    # High-price records
    "1055032",
    "1129386",
    "422772",
    "1046653",
}

# Records outside our residential rental scope
excluded_ids = {
    "1118520",
    "666526",
    "905625",
    "799943",
}

In [136]:
jiji_audit = jiji_model.copy()

jiji_audit["listing_id"] = (
    jiji_audit["listing_id"].astype(str)
)

jiji_review = jiji_audit[
    jiji_audit["listing_id"].isin(review_ids)
].copy()

jiji_excluded = jiji_audit[
    jiji_audit["listing_id"].isin(excluded_ids)
].copy()

jiji_final = jiji_audit[
    ~jiji_audit["listing_id"].isin(review_ids | excluded_ids)
].copy()

print("Initial modeling candidates:", len(jiji_audit))
print("Quarantined:", len(jiji_review))
print("Excluded by property scope:", len(jiji_excluded))
print("Remaining candidates:", len(jiji_final))

Initial modeling candidates: 1939
Quarantined: 14
Excluded by property scope: 4
Remaining candidates: 1921


In [137]:
audit_columns = [
    "listing_id",
    "title",
    "address",
    "property_type",
    "size_sqm",
    "bedrooms",
    "monthly_rent",
    "currency",
    "price_period",
    "date_published",
    "description",
]

print("QUARANTINED LISTINGS")
print(
    jiji_review[audit_columns]
    .to_string(index=False, max_colwidth=250)
)

print("\nEXCLUDED BY PROPERTY SCOPE")
print(
    jiji_excluded[audit_columns]
    .to_string(index=False, max_colwidth=250)
)

QUARANTINED LISTINGS


KeyError: "['description'] not in index"

In [138]:
inspect_ids = {
    "1034902",
    "1174252",
    "992492",
    "422772",
    "1078603",
    "1102230",
}

for record in raw_records:
    advert = record.get("advert", {})
    listing_id = str(advert.get("id"))

    if listing_id in inspect_ids:
        print("\n" + "=" * 80)
        print("LISTING ID:", listing_id)
        print("TITLE:", advert.get("title"))
        print("PRICE:")
        print(advert.get("price"))
        print("PRICE OBJECT:")
        print(advert.get("price_obj"))
        print("DESCRIPTION:")
        print(advert.get("description"))


LISTING ID: 1078603
TITLE: Furnished 4bdrm Townhouse / Terrace in Bole for rent
PRICE:
{'value': 420000, 'type': 'Amount', 'title': 'ETB 420,000 per month', 'period': 'per month', 'is_closed': False}
PRICE OBJECT:
{'value': 420000, 'view': 'ETB 420,000', 'period': 'per month', 'bulk': None}
DESCRIPTION:
።g+3 home Rent imperial ።4beds and 6baths ።5living room ።heating fire extinguisher wifi ።dedicated work space ።kitchen ።cinima ።bathtup ።steam and sauna ።7500USD

LISTING ID: 1102230
TITLE: 8bdrm House in Bole for rent
PRICE:
{'value': 840000, 'type': 'Amount', 'title': 'ETB 840,000 per month', 'period': 'per month', 'is_closed': False}
PRICE OBJECT:
{'value': 840000, 'view': 'ETB 840,000', 'period': 'per month', 'bulk': None}
DESCRIPTION:
House rent in bole 8bedroom 8bathroom 10car parking 15000 USD

LISTING ID: 1102230
TITLE: 8bdrm House in Bole for rent
PRICE:
{'value': 840000, 'type': 'Amount', 'title': 'ETB 840,000 per month', 'period': 'per month', 'is_closed': False}
PRICE OBJEC

In [139]:
from collections import Counter

listing_ids = [
    str(record.get("advert", {}).get("id"))
    for record in raw_records
    if record.get("advert", {}).get("id") is not None
]

id_counts = Counter(listing_ids)

duplicate_ids = {
    listing_id: count
    for listing_id, count in id_counts.items()
    if count > 1
}

print("Unique listing IDs:", len(set(listing_ids)))
print("Records with IDs:", len(listing_ids))
print("Number of duplicated IDs:", len(duplicate_ids))

print("\nDuplicate listing IDs:")
for listing_id, count in duplicate_ids.items():
    print(f"{listing_id}: {count} occurrences")

Unique listing IDs: 2082
Records with IDs: 2296
Number of duplicated IDs: 200

Duplicate listing IDs:
687029: 2 occurrences
707659: 2 occurrences
1175879: 2 occurrences
753524: 2 occurrences
692851: 2 occurrences
558425: 2 occurrences
724632: 3 occurrences
907859: 2 occurrences
618557: 2 occurrences
680200: 3 occurrences
684328: 2 occurrences
776722: 2 occurrences
582788: 2 occurrences
614090: 2 occurrences
706375: 3 occurrences
978561: 2 occurrences
1050373: 2 occurrences
1101375: 2 occurrences
1088240: 2 occurrences
976062: 3 occurrences
615090: 2 occurrences
1025805: 2 occurrences
907815: 2 occurrences
1027508: 2 occurrences
994799: 2 occurrences
1084143: 2 occurrences
711102: 2 occurrences
1055032: 3 occurrences
1162146: 2 occurrences
1131137: 2 occurrences
603104: 2 occurrences
1117784: 2 occurrences
1105624: 2 occurrences
1049486: 2 occurrences
975218: 2 occurrences
1043544: 2 occurrences
1133209: 2 occurrences
432615: 2 occurrences
1072986: 2 occurrences
538178: 2 occurrences
44

In [140]:
import json
from collections import defaultdict, Counter

duplicate_groups = defaultdict(list)

for record in raw_records:
    advert = record.get("advert", {})
    listing_id = str(advert.get("id", ""))

    if listing_id:
        duplicate_groups[listing_id].append(advert)

# Fields that define the listing's content
comparison_fields = [
    "title",
    "description",
    "price",
    "price_obj",
    "attrs",
    "region_text",
    "region_name",
]

exact_duplicate_groups = []
conflicting_duplicate_groups = []

for listing_id, adverts in duplicate_groups.items():
    if len(adverts) <= 1:
        continue

    signatures = {
        json.dumps(
            {field: advert.get(field) for field in comparison_fields},
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        for advert in adverts
    }

    if len(signatures) == 1:
        exact_duplicate_groups.append(listing_id)
    else:
        conflicting_duplicate_groups.append(listing_id)

print("DUPLICATE AUDIT")
print("=" * 50)
print("Duplicate listing IDs:", len(exact_duplicate_groups) + len(conflicting_duplicate_groups))
print("Exact duplicate groups:", len(exact_duplicate_groups))
print("Conflicting duplicate groups:", len(conflicting_duplicate_groups))

print("\nOccurrences per duplicate ID:")
print(Counter(
    len(adverts)
    for adverts in duplicate_groups.values()
    if len(adverts) > 1
))

print("\nCONFLICTING DUPLICATE LISTINGS")
print("=" * 50)

for listing_id in conflicting_duplicate_groups[:15]:
    adverts = duplicate_groups[listing_id]

    print(f"\nListing ID: {listing_id} ({len(adverts)} occurrences)")

    for index, advert in enumerate(adverts, start=1):
        print(f"\nVersion {index}")
        print("Title:", advert.get("title"))
        print("Description:", str(advert.get("description", ""))[:300])
        print("Price:", advert.get("price"))
        print("Region:", advert.get("region_text"))

DUPLICATE AUDIT
Duplicate listing IDs: 200
Exact duplicate groups: 200
Conflicting duplicate groups: 0

Occurrences per duplicate ID:
Counter({2: 186, 3: 14})

CONFLICTING DUPLICATE LISTINGS


In [141]:
# Convert raw JSON records into a DataFrame
raw_df = pd.json_normalize(raw_records)

# Convert listing IDs to strings
raw_df["advert.id"] = raw_df["advert.id"].astype(str)

print("Records before deduplication:", len(raw_df))

# Retain one occurrence of each listing ID
raw_df = raw_df.drop_duplicates(
    subset=["advert.id"],
    keep="first"
).copy()

print("Records after deduplication:", len(raw_df))
print("Remaining duplicate IDs:", raw_df["advert.id"].duplicated().sum())

Records before deduplication: 2296
Records after deduplication: 2082
Remaining duplicate IDs: 0


In [143]:
import pandas as pd
from pathlib import Path

# Identify listings whose descriptions mention USD or dollar values
usd_mask = jiji_df["description"].str.contains(
    r"USD|US dollar|dollar|\$",
    case=False,
    na=False
)

usd_review = jiji_df.loc[usd_mask].copy()

# Add columns for manual review
usd_review["currency_decision"] = ""
usd_review["review_note"] = ""

# Keep only useful audit columns
review_columns = [
    "listing_id",
    "title",
    "address",
    "raw_price",
    "raw_price_title",
    "description",
    "currency_decision",
    "review_note"
]

usd_review = usd_review[review_columns].copy()

print("Listings requiring currency review:", len(usd_review))

display(usd_review.head(10))

Listings requiring currency review: 181


,listing_id,title,address,raw_price,raw_price_title,description,currency_decision,review_note
12,1099286,5bdrm House in Yeka for rent,"Megenagna, close to Bellevue Hotel, Addis Abab...",200000.0,"ETB 200,000 per month, Negotiable",House with Good View: Rent \n__Unfurnished 5 b...,,
22,949777,"3bdrm Townhouse / Terrace in Top View, Yeka fo...",megenagna,120000.0,"ETB 120,000 per month",Cozy G+1 House up for rent \n* \n* Spacious li...,,
31,1078603,Furnished 4bdrm Townhouse / Terrace in Bole fo...,imperial,420000.0,"ETB 420,000 per month",።g+3 home Rent imperial ።4beds and 6baths ።5li...,,
39,1012514,Furnished 3bdrm Apartment in Bole for rent,Meskel flower area,140000.0,"ETB 140,000 per month",Fully Furnished Apartment \n• Location Meskel ...,,
50,1011310,Furnished 3bdrm Apartment in Bole for rent,Meskel flower ( Gazebo square),84000.0,"ETB 84,000 per month",Fully Furnished Apartment \n• Location Meskel ...,,
56,1024873,Furnished 2bdrm Apartment in Kirkos for rent,Bole Peakock,84000.0,"ETB 84,000 per month, Negotiable",Two bedroom furnished apartment available for ...,,
67,776722,Furnished 6bdrm Apartment in Bole for rent,BOLE,250000.0,"ETB 250,000 per month",Location Bole \nBad rooms 6\nbathroom 6\nprice...,,
71,1080176,Furnished 3bdrm Apartment in Kirkos for rent,Around dembel,112000.0,"ETB 112,000 per month",Three bedroom fully furnished apartment availa...,,
84,1158302,Furnished 2bdrm Apartment in Bole for rent,Bole Atlas,150000.0,"ETB 150,000 per month",Apartment + Natural Light: Rent \n__Furnished...,,
91,1195104,Furnished 2bdrm Apartment in Bole for rent,bambis,95200.0,"ETB 95,200 per month",Two bedroom full furnished available for rent ...,,


In [144]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

review_path = output_dir / "jiji_currency_review.csv"

usd_review.to_csv(review_path, index=False)

print(f"Saved review file to: {review_path}")

Saved review file to: data/processed/jiji_currency_review.csv


In [161]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. REGISTER YOUR EXTRACTED DATASETS
# --------------------------------------------------

# Add each website's extracted DataFrame here as it becomes available.
# For now, we only have the Jiji DataFrame.

source_frames = {
    "jiji": jiji_df,
    "qefira_apartments": qefira_apartments_df,
    "qefira_houses": qefira_houses_df,
    "realethio": realethio_df,
    "livingethio": livingethio_df,
    "ethiopianproperties": ethiopianproperties_df,
    "zegebeya": zegebeya_df,
}

# --------------------------------------------------
# 2. STANDARDIZE AND AUDIT EACH DATASET
# --------------------------------------------------

audit_datasets = []

for source_name, df in source_frames.items():

    df = df.copy()

    # Support different column names across extracted datasets
    def get_column(possible_names, default=None):
        for name in possible_names:
            if name in df.columns:
                return df[name]
        return pd.Series(default, index=df.index)

    audit = pd.DataFrame(index=df.index)

    audit["source"] = source_name
    audit["listing_id"] = get_column(["listing_id", "id"])
    audit["title"] = get_column(["title"])
    audit["address"] = get_column(["address", "location"])

    audit["raw_price"] = get_column(
        ["raw_price", "monthly_rent", "price"]
    )

    audit["raw_price_title"] = get_column(
        ["raw_price_title", "price_title"]
    )

    audit["currency"] = get_column(["currency"])
    audit["price_period"] = get_column(["price_period", "period"])
    audit["description"] = get_column(["description"], "").fillna("").astype(str)

    # Detect currency mentions in descriptions and structured metadata
    audit["description_mentions_usd"] = audit["description"].str.contains(
        r"\bUSD\b|\bUS dollars?\b|\bdollars?\b|\$",
        case=False,
        na=False
    )

    audit["description_mentions_etb"] = audit["description"].str.contains(
        r"\bETB\b|\bBirr\b|\bBirr\b",
        case=False,
        na=False
    )

    # Flag descriptions that mention both currencies
    audit["mentions_both_currencies"] = (
        audit["description_mentions_usd"]
        & audit["description_mentions_etb"]
    )

    # Identify missing structured currency or period
    audit["missing_currency"] = (
        audit["currency"].isna()
        | audit["currency"].astype(str).str.strip().eq("")
    )

    audit["missing_price_period"] = (
        audit["price_period"].isna()
        | audit["price_period"].astype(str).str.strip().eq("")
    )

    # Conservative review flag
    audit["needs_currency_review"] = (
        audit["description_mentions_usd"]
        | audit["mentions_both_currencies"]
        | audit["missing_currency"]
        | audit["missing_price_period"]
    )

    # Manual review fields
    audit["currency_decision"] = ""
    audit["review_note"] = ""

    audit_datasets.append(audit.reset_index(drop=True))


# --------------------------------------------------
# 3. COMBINE AUDIT RESULTS
# --------------------------------------------------

if not audit_datasets:
    raise ValueError("No source datasets were registered.")

currency_audit = pd.concat(
    audit_datasets,
    ignore_index=True
)

currency_review = currency_audit[
    currency_audit["needs_currency_review"]
].copy()


# --------------------------------------------------
# 4. SAVE RESULTS
# --------------------------------------------------

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

currency_audit.to_csv(
    output_dir / "all_sources_currency_audit.csv",
    index=False
)

currency_review.to_csv(
    output_dir / "all_sources_currency_review.csv",
    index=False
)


# --------------------------------------------------
# 5. PRINT SUMMARY
# --------------------------------------------------

print("CURRENCY AUDIT SUMMARY")
print("=" * 50)

print("Total listings audited:", len(currency_audit))
print("Listings requiring review:", len(currency_review))

print("\nListings per source:")
print(currency_audit.groupby("source").size())

print("\nReview candidates per source:")
print(
    currency_review.groupby("source").size()
)

print("\nCurrency mentions:")
print(
    currency_audit.groupby("source")[
        ["description_mentions_usd", "description_mentions_etb"]
    ].sum()
)

print("\nSaved files:")
print(output_dir / "all_sources_currency_audit.csv")
print(output_dir / "all_sources_currency_review.csv")

display(currency_review.head(10))

NameError: name 'qefira_apartments_df' is not defined

In [148]:
import json
from pathlib import Path

QEFIRA_DIR = Path("data/raw/real_estate/raw/qefira")

qefira_files = {
    "apartments": QEFIRA_DIR / "apartments-for-rent_2023_05_29.json",
    "houses": QEFIRA_DIR / "houses-for-rent_2023_05_30.json",
    "bedsitters_rooms": QEFIRA_DIR / "bedsitters-rooms-for-rent_2023_05_30.json",
    "commercial": QEFIRA_DIR / "commercial-property-for-rent_2023_05_30.json",
}

for category, file_path in qefira_files.items():

    print("\n" + "=" * 70)
    print(f"SOURCE: QEFIRA - {category.upper()}")
    print("=" * 70)

    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print("Total records:", len(records))

    if not records:
        continue

    print("\nFirst record keys:")
    print(list(records[0].keys()))

    print("\nFirst record:")
    print(json.dumps(records[0], indent=2, ensure_ascii=False)[:5000])

    # Inspect price information
    print("\nPRICE SCHEMA:")

    for i, record in enumerate(records[:5]):
        print(f"\nRecord {i + 1}")

        offers = record.get("offers")

        if isinstance(offers, dict):
            print("Offers:", offers)
        else:
            print("Offers:", offers)

        print("Category:", record.get("category"))
        print("Address:", record.get("address"))


SOURCE: QEFIRA - APARTMENTS
Total records: 1398

First record keys:
['id', 'name', 'description', 'category', 'image', 'offers', 'parsed_description', 'address']

First record:
{
  "id": "5734933",
  "name": "Apartment for rent in kasanches",
  "description": "",
  "category": "Apartments for Rent",
  "image": "https://i.roamcdn.net/hz/qe/listing-gallery-full-1920w/f5dfc76d6bf6f475655d8a43afb1cba2/-/horizon-files-prod/qe/picture/qxj6xndv/b4b9e30984e9c8df8313eb4bf1d5d4ca0625dd88.jpg",
  "offers": {
    "@type": "Offer",
    "url": "https://www.qefira.com/listings/apartment-for-rent-in-kasanches-5734933",
    "priceCurrency": "ETB",
    "price": 84500,
    "itemCondition": "http://schema.org/NewCondition",
    "availability": "http://schema.org/InStock",
    "validFrom": "2023-05-29 12:42:10",
    "priceValidUntil": "2023-08-27",
    "seller": {
      "@type": "Person",
      "name": "leke"
    }
  },
  "parsed_description": {
    "No of bedrooms": "2",
    "Bathrooms": "2"
  },
  "addr

In [150]:
import json
import re
import pandas as pd

from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

QEFIRA_DIR = Path("data/raw/real_estate/raw/qefira")

FILES = {
    "apartments": "apartments-for-rent_2023_05_29.json",
    "houses": "houses-for-rent_2023_05_30.json",
    "bedsitters_rooms": "bedsitters-rooms-for-rent_2023_05_30.json",
}

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def mentions_usd(text):
    return bool(re.search(
        r"\$|\bUSD\b|\bUS dollars?\b|\bdollars?\b",
        str(text),
        flags=re.IGNORECASE
    ))


def mentions_etb(text):
    return bool(re.search(
        r"\bETB\b|\bBirr\b",
        str(text),
        flags=re.IGNORECASE
    ))


def detect_price_period(text):
    """
    Detect period only when the description provides evidence.
    Do not assume all Qefira prices are monthly.
    """

    text = str(text).lower()

    patterns = {
        "monthly": [
            r"per\s+month",
            r"/\s*month",
            r"/\s*mo\b",
            r"\bmonthly\b",
            r"\bper\s+mo\b",
        ],
        "weekly": [
            r"per\s+week",
            r"/\s*week",
            r"\bweekly\b",
        ],
        "daily": [
            r"per\s+day",
            r"/\s*day",
            r"\bdaily\b",
        ],
        "yearly": [
            r"per\s+year",
            r"/\s*year",
            r"\byearly\b",
            r"\bannually\b",
        ],
    }

    for period, regexes in patterns.items():
        if any(re.search(pattern, text) for pattern in regexes):
            return period

    return "unspecified"


def extract_number(value):
    """Extract a numeric value without changing its unit."""
    if value is None:
        return None

    match = re.search(r"\d+(?:,\d{3})*(?:\.\d+)?", str(value))

    if match:
        return float(match.group(0).replace(",", ""))

    return None


def is_addis_ababa(address):
    return "addis ababa" in str(address).lower()


# ============================================================
# 3. EXTRACT EACH QEFIRA RENTAL DATASET
# ============================================================

all_listings = []

for source_category, filename in FILES.items():

    file_path = QEFIRA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print(f"\nProcessing {source_category}: {len(records)} records")

    for record in records:
        
        if not isinstance(record, dict):
            print(f"Skipping invalid record: {record!r}")
            continue

        offers = record.get("offers") or {}
        parsed = record.get("parsed_description") or {}

        if not isinstance(offers, dict):
            offers = {}

        if not isinstance(parsed, dict):
            parsed = {}

        description = str(record.get("description") or "")
        title = str(record.get("name") or "")
        category = str(record.get("category") or "")
        address = str(record.get("address") or "")

        # Preserve original structured price
        raw_price = extract_number(offers.get("price"))

        currency = offers.get("priceCurrency")

        # Search both title and description for rental period evidence
        searchable_text = f"{title} {description}"
        price_period = detect_price_period(searchable_text)

        usd_mentioned = mentions_usd(searchable_text)
        etb_mentioned = mentions_etb(searchable_text)

        # Extract property attributes without assuming units
        bedrooms = extract_number(
            parsed.get("No of bedrooms")
        )

        bathrooms = extract_number(
            parsed.get("Bathrooms")
        )

        size_value = None
        size_unit = None

        if parsed.get("Square feet") is not None:
            size_value = extract_number(parsed.get("Square feet"))
            size_unit = "sqft"

        elif parsed.get("Square meters") is not None:
            size_value = extract_number(parsed.get("Square meters"))
            size_unit = "sqm"

        # Only accept records explicitly categorized as rentals
        is_rental = "for rent" in category.lower()

        # Residential categories only
        is_residential_category = any(
            keyword in category.lower()
            for keyword in [
                "apartment",
                "house",
                "bedsitter",
                "room",
            ]
        )

        # Require Addis Ababa in the address
        is_addis = is_addis_ababa(address)

        # Currency and price validation
        valid_price = (
            raw_price is not None
            and raw_price > 0
        )

        missing_currency = not currency

        # Do not assume the structured ETB price is valid
        currency_conflict = usd_mentioned

        needs_currency_review = (
            missing_currency
            or currency_conflict
            or price_period == "unspecified"
            or not valid_price
        )

        # Only create a monthly target when the period and
        # currency are sufficiently supported by the available data.
        monthly_rent = None

        if (
            valid_price
            and str(currency).upper() == "ETB"
            and price_period == "monthly"
            and not usd_mentioned
        ):
            monthly_rent = raw_price

        listing = {
            "source": "qefira",
            "source_category": source_category,

            "listing_id": record.get("id"),
            "title": title,
            "description": description,
            "category": category,
            "address": address,

            "listing_url": offers.get("url"),
            "date_published": offers.get("validFrom"),

            "property_type": category,
            "bedrooms": bedrooms,
            "bathrooms": bathrooms,

            "size_value": size_value,
            "size_unit": size_unit,

            "raw_price": raw_price,
            "raw_price_title": (
                f"{currency} {raw_price}"
                if currency and raw_price is not None
                else None
            ),

            "currency": currency,
            "price_period": price_period,
            "monthly_rent": monthly_rent,

            "description_mentions_usd": usd_mentioned,
            "description_mentions_etb": etb_mentioned,
            "currency_conflict": currency_conflict,
            "missing_currency": missing_currency,
            "valid_price": valid_price,
            "needs_currency_review": needs_currency_review,

            "is_rental": is_rental,
            "is_residential_category": is_residential_category,
            "is_addis_ababa": is_addis,

            "currency_decision": "",
            "review_note": "",
        }

        all_listings.append(listing)


# ============================================================
# 4. CREATE DATAFRAME
# ============================================================

qefira_df = pd.DataFrame(all_listings)

print("\n" + "=" * 65)
print("QEFIRA EXTRACTION SUMMARY")
print("=" * 65)

print("Total extracted records:", len(qefira_df))


# ============================================================
# 5. VALIDATION AND FILTERING
# ============================================================

# Keep the complete extracted data for auditing.
# Do not discard records just because they need review.

qefira_candidates = qefira_df[
    qefira_df["is_rental"]
    & qefira_df["is_residential_category"]
    & qefira_df["is_addis_ababa"]
    & qefira_df["valid_price"]
].copy()

qefira_review = qefira_candidates[
    qefira_candidates["needs_currency_review"]
].copy()

# Candidates with ETB, monthly evidence, and no USD mention
qefira_verified_candidates = qefira_candidates[
    (qefira_candidates["currency"].astype(str).str.upper() == "ETB")
    & (qefira_candidates["price_period"] == "monthly")
    & (~qefira_candidates["description_mentions_usd"])
].copy()


# ============================================================
# 6. DUPLICATE AUDIT
# ============================================================

duplicate_mask = qefira_df["listing_id"].duplicated(keep=False)

qefira_duplicates = qefira_df[
    duplicate_mask
].sort_values("listing_id").copy()


# ============================================================
# 7. SAVE DATASETS
# ============================================================

qefira_df.to_csv(
    OUTPUT_DIR / "qefira_all_extracted.csv",
    index=False
)

qefira_candidates.to_csv(
    OUTPUT_DIR / "qefira_residential_rental_candidates.csv",
    index=False
)

qefira_review.to_csv(
    OUTPUT_DIR / "qefira_currency_review.csv",
    index=False
)

qefira_verified_candidates.to_csv(
    OUTPUT_DIR / "qefira_monthly_etb_candidates.csv",
    index=False
)

qefira_duplicates.to_csv(
    OUTPUT_DIR / "qefira_duplicate_audit.csv",
    index=False
)


# ============================================================
# 8. PRINT AUDIT REPORT
# ============================================================

print("\nRECORD COUNTS")
print("-" * 50)

print("All extracted:", len(qefira_df))
print("Residential rental candidates:", len(qefira_candidates))
print("Currency review candidates:", len(qefira_review))
print("Monthly ETB candidates:", len(qefira_verified_candidates))
print("Duplicate listing records:", len(qefira_duplicates))

print("\nRECORDS BY ORIGINAL FILE")
print("-" * 50)

print(qefira_df.groupby("source_category").size())

print("\nCANDIDATES BY CATEGORY")
print("-" * 50)

print(qefira_candidates.groupby("source_category").size())

print("\nCURRENCY DISTRIBUTION")
print("-" * 50)

print(qefira_candidates["currency"].value_counts(dropna=False))

print("\nPRICE PERIOD DISTRIBUTION")
print("-" * 50)

print(qefira_candidates["price_period"].value_counts(dropna=False))

print("\nCURRENCY REVIEW REASONS")
print("-" * 50)

print(
    qefira_candidates[
        [
            "description_mentions_usd",
            "description_mentions_etb",
            "currency_conflict",
            "missing_currency",
            "needs_currency_review",
        ]
    ].sum()
)

print("\nMISSING VALUES")
print("-" * 50)

print(
    qefira_candidates[
        [
            "raw_price",
            "currency",
            "price_period",
            "bedrooms",
            "bathrooms",
            "size_value",
        ]
    ].isna().sum()
)

print("\nSAMPLE CURRENCY REVIEW RECORDS")
print("-" * 50)

display(
    qefira_review[
        [
            "listing_id",
            "title",
            "raw_price",
            "currency",
            "price_period",
            "description",
            "needs_currency_review",
        ]
    ].head(10)
)

print("\nSAVED FILES")
print("-" * 50)

for filename in [
    "qefira_all_extracted.csv",
    "qefira_residential_rental_candidates.csv",
    "qefira_currency_review.csv",
    "qefira_monthly_etb_candidates.csv",
    "qefira_duplicate_audit.csv",
]:
    print(OUTPUT_DIR / filename)


Processing apartments: 1398 records
Skipping invalid record: None

Processing houses: 714 records

Processing bedsitters_rooms: 17 records

QEFIRA EXTRACTION SUMMARY
Total extracted records: 2128

RECORD COUNTS
--------------------------------------------------
All extracted: 2128
Residential rental candidates: 2119
Currency review candidates: 2040
Monthly ETB candidates: 59
Duplicate listing records: 29

RECORDS BY ORIGINAL FILE
--------------------------------------------------
source_category
apartments          1397
bedsitters_rooms      17
houses               714
dtype: int64

CANDIDATES BY CATEGORY
--------------------------------------------------
source_category
apartments          1397
bedsitters_rooms       8
houses               714
dtype: int64

CURRENCY DISTRIBUTION
--------------------------------------------------
currency
ETB    2119
Name: count, dtype: int64

PRICE PERIOD DISTRIBUTION
--------------------------------------------------
price_period
unspecified    1704

,listing_id,title,raw_price,currency,price_period,description,needs_currency_review
0,5734933,Apartment for rent in kasanches,84500.0,ETB,unspecified,,True
1,5735250,Apartment for rent at Bisrate Gebriel(Old Airp...,2000.0,ETB,unspecified,For rentFurnished and/or Unfurnished options\...,True
2,5734584,"Distinctive Bungalow for rent in Old Airport, ...",216000.0,ETB,unspecified,Enter Ethiopia is pleased to present this bung...,True
3,5734908,Apartment for rent in aware close to German sc...,81000.0,ETB,unspecified,🍏 Nice Apartment For rent 🍏\n__Furnished 2 bed...,True
5,5734840,Apartment for rent in bole Atlas,60000.0,ETB,unspecified,🌻 One bedroom apartment: rent 🌻__Furnished 1 b...,True
6,5735195,Guest House for Rent in Kazanchis,60000.0,ETB,unspecified,Guest House for Rent in Kazanchis1 bed1 bath1 ...,True
7,5734886,New apartment for rent in bole,135000.0,ETB,unspecified,🌺 Brand new Apartment: Rent 🌺\n__Furnished 2 b...,True
8,5734417,CONDOMINIUM FOR SALE @ BOLE 24,650000.0,ETB,unspecified,"CONDOMINIUM FOR SALE @ BOLE 24, AROUND NEW STA...",True
10,5734218,Fully furnished apartment,350.0,ETB,unspecified,Apartment for Rent in USDFully furnished apart...,True
11,5694854,Apartment for rent close to Greek school/Bambis,70000.0,ETB,unspecified,FOR RENT :,True



SAVED FILES
--------------------------------------------------
data/processed/qefira_all_extracted.csv
data/processed/qefira_residential_rental_candidates.csv
data/processed/qefira_currency_review.csv
data/processed/qefira_monthly_etb_candidates.csv
data/processed/qefira_duplicate_audit.csv


In [151]:

import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/realethio/realethio_data.json"
)

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 65)
print("REALETHIO SCHEMA INSPECTION")
print("=" * 65)

# Inspect the top-level structure
print("\nTOP-LEVEL TYPE:")
print(type(data).__name__)

# Normalize the top-level structure for inspection only
if isinstance(data, list):
    records = data
elif isinstance(data, dict):
    print("\nTOP-LEVEL KEYS:")
    print(list(data.keys()))

    # Identify the list containing the listings
    records = next(
        (value for value in data.values() if isinstance(value, list)),
        None
    )

    if records is None:
        raise ValueError("Could not find a list of listing records.")
else:
    raise ValueError("Unexpected JSON structure.")

print("\nTOTAL RECORDS:", len(records))

# Count invalid records
invalid = sum(not isinstance(r, dict) for r in records)
print("INVALID RECORDS:", invalid)

valid_records = [
    r for r in records
    if isinstance(r, dict)
]

# Inspect the keys used across records
key_counts = Counter(
    key
    for record in valid_records
    for key in record.keys()
)

print("\nFIELD FREQUENCY:")
for key, count in key_counts.most_common():
    print(f"{key}: {count}/{len(valid_records)}")

# Inspect representative records
print("\n" + "=" * 65)
print("SAMPLE RECORDS")
print("=" * 65)

for i, record in enumerate(valid_records[:5], start=1):
    print(f"\nRECORD {i}")
    print(json.dumps(record, indent=2, ensure_ascii=False)[:5000])

# Inspect listing status distribution
print("\n" + "=" * 65)
print("LISTING STATUS DISTRIBUTION")
print("=" * 65)

status_counts = Counter(
    str(record.get("listing_type"))
    for record in valid_records
)

for status, count in status_counts.most_common():
    print(f"{status}: {count}")


REALETHIO SCHEMA INSPECTION

TOP-LEVEL TYPE:
list

TOTAL RECORDS: 1272
INVALID RECORDS: 0

FIELD FREQUENCY:
title: 1272/1272
price: 1272/1272
listing_type: 1272/1272
address: 1272/1272
description: 1272/1272
features: 1272/1272
images: 1272/1272
Property ID: 1272/1272
Price: 1272/1272
Property Type: 1272/1272
Property Status: 1272/1272
Furnished: 1272/1272
date_published: 1272/1272
date_modified: 1272/1272
author: 1272/1272
lng: 1272/1272
lat: 1272/1272
url: 1272/1272
Property Size: 1271/1272
Floor: 1259/1272
Land Area: 1143/1272
Bedrooms: 1003/1272
Bathrooms: 897/1272
Garage: 554/1272
Garages: 390/1272
Year Built: 348/1272
Bathroom: 96/1272
Bedroom: 84/1272
Rooms: 16/1272
Room: 2/1272
Garage Size: 1/1272

SAMPLE RECORDS

RECORD 1
{
  "title": "CMC Michael, 3 bedrooms Apartment for Rent, Addis Ababa.",
  "price": "50,000ETB /mo",
  "listing_type": "For Rent",
  "address": "Addis tesfa fisseha stadium bole, Al-tad cmc Michael Addis Ababa, 211219, YK_13_0605 Street, Yedejazmach Alula Irs

In [153]:

import json
import re
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------

file_path = Path(
    "data/raw/real_estate/raw/realethio/realethio_data.json"
)

with open(file_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Raw records:", len(records))


# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------

def extract_number(value):
    """Extract the first numeric value from a string."""
    if value is None:
        return None

    match = re.search(
        r"\d[\d,]*(?:\.\d+)?",
        str(value)
    )

    if not match:
        return None

    try:
        return float(match.group().replace(",", ""))
    except ValueError:
        return None


def detect_currency(price_text):
    """Detect currency from the structured price field."""
    text = str(price_text).upper()

    if "ETB" in text or "BIRR" in text:
        return "ETB"

    if "USD" in text or "$" in text:
        return "USD"

    return None


def detect_period(price_text, description):
    """Detect an explicitly stated billing period."""
    text = f"{price_text} {description}".lower()

    if re.search(r"/\s*mo\b|per\s+month|monthly", text):
        return "monthly"

    if re.search(r"/\s*week\b|per\s+week|weekly", text):
        return "weekly"

    if re.search(r"/\s*day\b|per\s+day|daily", text):
        return "daily"

    if re.search(r"/\s*year\b|per\s+year|yearly|annually", text):
        return "yearly"

    return "unspecified"


def extract_measurement(value):
    """Extract a numeric measurement."""
    return extract_number(value)


def is_residential(property_type, title):
    """Identify residential property types."""
    text = f"{property_type} {title}".lower()

    # Explicitly exclude commercial and non-residential uses
    excluded = [
        "commercial",
        "office",
        "warehouse",
        "shop",
        "retail",
        "industrial",
        "land",
        "plot",
        "guest house",
        "guesthouse",
        "hotel",
    ]

    if any(term in text for term in excluded):
        return False

    residential = [
        "apartment",
        "house",
        "villa",
        "condominium",
        "residential",
        "townhouse",
        "bungalow",
        "studio",
        "bedsitter",
        "room",
    ]

    return any(term in text for term in residential)


# ---------------------------------------------------------
# 3. EXTRACT LISTINGS
# ---------------------------------------------------------

extracted = []

for record in records:

    if not isinstance(record, dict):
        continue

    title = str(record.get("title") or "")
    description = str(record.get("description") or "")
    address = str(record.get("address") or "")

    listing_type = str(
        record.get("listing_type") or ""
    ).strip()

    property_type = str(
        record.get("Property Type") or ""
    ).strip()

    price_text = str(
        record.get("price") or record.get("Price") or ""
    ).strip()

    # Keep only rental listings
    if listing_type.lower() != "for rent":
        continue

    # Keep only Addis Ababa listings
    if "addis ababa" not in address.lower():
        continue

    # Keep residential properties only
    if not is_residential(property_type, title):
        continue

    raw_price = extract_number(price_text)
    currency = detect_currency(price_text)
    period = detect_period(price_text, description)

    description_lower = description.lower()

    description_mentions_usd = bool(
        re.search(r"\busd\b|\$", description_lower)
    )

    description_mentions_etb = bool(
        re.search(r"\betb\b|\bbirr\b", description_lower)
    )

    # Identify disagreement between structured price and description
    currency_conflict = (
        currency == "ETB" and description_mentions_usd
    ) or (
        currency == "USD" and description_mentions_etb
    )

    review_reasons = []

    if raw_price is None or raw_price <= 0:
        review_reasons.append("invalid_price")

    if currency is None:
        review_reasons.append("missing_currency")

    if period == "unspecified":
        review_reasons.append("unspecified_period")

    if description_mentions_usd:
        review_reasons.append("description_mentions_usd")

    if currency_conflict:
        review_reasons.append("currency_conflict")

    needs_review = len(review_reasons) > 0

    # Strict eligibility rule:
    # Only explicitly monthly ETB prices without USD mentions.
    monthly_rent = None

    if (
        raw_price is not None
        and raw_price > 0
        and currency == "ETB"
        and period == "monthly"
        and not description_mentions_usd
        and not currency_conflict
    ):
        monthly_rent = raw_price

    extracted.append({
        "source": "realethio",
        "listing_id": record.get("Property ID"),
        "title": title,
        "property_type": property_type,
        "listing_type": listing_type,
        "address": address,
        "description": description,

        "raw_price": raw_price,
        "raw_price_text": price_text,
        "currency": currency,
        "price_period": period,
        "monthly_rent": monthly_rent,

        "description_mentions_usd": description_mentions_usd,
        "description_mentions_etb": description_mentions_etb,
        "currency_conflict": currency_conflict,
        "needs_currency_review": needs_review,
        "currency_review_reason": "; ".join(review_reasons),

        "size_sqm": extract_measurement(
            record.get("Property Size")
        ),
        "land_area_sqm": extract_measurement(
            record.get("Land Area")
        ),
        "bedrooms": extract_measurement(
            record.get("Bedrooms") or record.get("Bedroom")
        ),
        "bathrooms": extract_measurement(
            record.get("Bathrooms") or record.get("Bathroom")
        ),
        "floor": extract_measurement(record.get("Floor")),
        "furnished": record.get("Furnished"),

        "date_published": record.get("date_published"),
        "date_modified": record.get("date_modified"),
        "latitude": record.get("lat"),
        "longitude": record.get("lng"),
        "url": record.get("url"),
    })


# ---------------------------------------------------------
# 4. CREATE DATAFRAME
# ---------------------------------------------------------

realethio_df = pd.DataFrame(extracted)

print("\nExtracted records:", len(realethio_df))


# ---------------------------------------------------------
# 5. DUPLICATE AUDIT
# ---------------------------------------------------------

duplicates = realethio_df[
    realethio_df["listing_id"].notna()
    & realethio_df["listing_id"].duplicated(keep=False)
].copy()

print("\nDuplicate rows:", len(duplicates))
print(
    "Duplicate listing IDs:",
    duplicates["listing_id"].nunique()
)

# Keep the first occurrence of each Property ID.
# Preserve records with missing IDs for separate review.
realethio_df = realethio_df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
)

print("Records after deduplication:", len(realethio_df))


# ---------------------------------------------------------
# 6. CREATE DATASET SUBSETS
# ---------------------------------------------------------

realethio_review = realethio_df[
    realethio_df["needs_currency_review"]
].copy()

realethio_monthly_etb = realethio_df[
    realethio_df["monthly_rent"].notna()
].copy()


# ---------------------------------------------------------
# 7. SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("REALETHIO EXTRACTION SUMMARY")
print("=" * 65)

print("\nRECORD COUNTS")
print("-" * 50)
print("Extracted:", len(realethio_df))
print("Currency review:", len(realethio_review))
print("Monthly ETB candidates:", len(realethio_monthly_etb))

print("\nPROPERTY TYPES")
print(realethio_df["property_type"].value_counts().head(20))

print("\nCURRENCY DISTRIBUTION")
print(realethio_df["currency"].value_counts(dropna=False))

print("\nPRICE PERIOD DISTRIBUTION")
print(realethio_df["price_period"].value_counts(dropna=False))

print("\nCURRENCY REVIEW REASONS")

review_reasons = [
    "description_mentions_usd",
    "description_mentions_etb",
    "currency_conflict",
    "needs_currency_review",
]

for reason in review_reasons:
    print(
        f"{reason}: "
        f"{realethio_df[reason].fillna(False).sum()}"
    )

print("\nMISSING VALUES")
print(
    realethio_df[
        [
            "raw_price",
            "currency",
            "price_period",
            "bedrooms",
            "bathrooms",
            "size_sqm",
        ]
    ].isna().sum()
)

print("\nMONTHLY RENT DISTRIBUTION")
if len(realethio_monthly_etb) > 0:
    print(
        realethio_monthly_etb["monthly_rent"].describe(
            percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
    )

print("\nSAMPLE CURRENCY REVIEW RECORDS")
display(
    realethio_review[
        [
            "listing_id",
            "title",
            "raw_price_text",
            "currency",
            "price_period",
            "description",
            "currency_review_reason",
        ]
    ].head(15)
)


# ---------------------------------------------------------
# 8. SAVE OUTPUTS
# ---------------------------------------------------------

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

realethio_df.to_csv(
    output_dir / "realethio_all_extracted.csv",
    index=False
)

realethio_review.to_csv(
    output_dir / "realethio_currency_review.csv",
    index=False
)

realethio_monthly_etb.to_csv(
    output_dir / "realethio_monthly_etb_candidates.csv",
    index=False
)

duplicates.to_csv(
    output_dir / "realethio_duplicate_audit.csv",
    index=False
)

print("\nSAVED FILES")
print("data/processed/realethio_all_extracted.csv")
print("data/processed/realethio_currency_review.csv")
print("data/processed/realethio_monthly_etb_candidates.csv")
print("data/processed/realethio_duplicate_audit.csv")


Raw records: 1272

Extracted records: 553

Duplicate rows: 2
Duplicate listing IDs: 1
Records after deduplication: 552

REALETHIO EXTRACTION SUMMARY

RECORD COUNTS
--------------------------------------------------
Extracted: 552
Currency review: 410
Monthly ETB candidates: 142

PROPERTY TYPES
property_type
Apartment for rent                   397
House for rent                       138
Building for rent                     12
Building for rent, House for rent      3
Apartment for sale                     1
House for sale                         1
Name: count, dtype: int64

CURRENCY DISTRIBUTION
currency
ETB    552
Name: count, dtype: int64

PRICE PERIOD DISTRIBUTION
price_period
monthly        547
unspecified      5
Name: count, dtype: int64

CURRENCY REVIEW REASONS
description_mentions_usd: 408
description_mentions_etb: 192
currency_conflict: 408
needs_currency_review: 410

MISSING VALUES
raw_price        0
currency         0
price_period     0
bedrooms         1
bathrooms       29


,listing_id,title,raw_price_text,currency,price_period,description,currency_review_reason
1,RE6832,"Three bedrooms Apartment for Rent, Lancha Or M...","320,000ETB /mo",ETB,monthly,"Lancha Or Meskel Flower, Three bedrooms Apartm...",description_mentions_usd; currency_conflict
2,RE6831,"Two bedrooms Apartment for Rent, Lancha Or Mes...","127,600ETB /mo",ETB,monthly,"Lancha Or Meskel Flower, Two bedrooms Apartmen...",description_mentions_usd; currency_conflict
4,RE6829,"G+2 Furnished Three Bedroom House for Rent, Ay...","250,000ETB /mo",ETB,monthly,"Ayat Beshale, G+2 Furnished Three Bedroom Hous...",description_mentions_usd; currency_conflict
5,RE6824,Furnished 4 bedrooms Penthouse Apartment for R...,"232,000ETB /mo",ETB,monthly,"Aware, Furnished 4 bedrooms Penthouse Apartmen...",description_mentions_usd; currency_conflict
6,RE6828,"Furnished One bedroom apartment for Rent, Bela...","69,600ETB /mo",ETB,monthly,"Bela, Furnished One bedroom apartment for Rent...",description_mentions_usd; currency_conflict
7,RE6827,"Furnished Two bedroom Apartment for Rent, Bela...","87,000ETB /mo",ETB,monthly,"Bela, Furnished Two bedroom Apartment for Rent...",description_mentions_usd; currency_conflict
9,RE6818,"Furnished Two bedrooms Apartment for Rent, Bol...","250,000ETB",ETB,monthly,"Bole Rwanda, Furnished Two bedrooms Apartment ...",description_mentions_usd; currency_conflict
10,RE2767,"Three bedrooms house for rent, Bole, Addis Ababa.","163,000ETB /mo",ETB,monthly,"Three bedrooms house for rent, Bole, Addis Aba...",description_mentions_usd; currency_conflict
11,RE6384,"Megenagna, Furnished Duplex 3 bedrooms Apartme...","145,000ETB /mo",ETB,monthly,"Megenagna, Furnished Duplex 3 bedrooms Apartme...",description_mentions_usd; currency_conflict
12,RE3764,"Bole, furnished 9 bedrooms house for rent, Add...","350,000ETB /mo",ETB,monthly,"Bole, furnished 9 bedrooms house for rent, Add...",description_mentions_usd; currency_conflict



SAVED FILES
data/processed/realethio_all_extracted.csv
data/processed/realethio_currency_review.csv
data/processed/realethio_monthly_etb_candidates.csv
data/processed/realethio_duplicate_audit.csv


In [154]:

import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/livingethio/livingethio.json"
)

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 65)
print("LIVINGETHIO SCHEMA INSPECTION")
print("=" * 65)

print("\nTOP-LEVEL TYPE:", type(data).__name__)

# Identify the records list
if isinstance(data, list):
    records = data

elif isinstance(data, dict):
    print("\nTOP-LEVEL KEYS:", list(data.keys()))

    records = next(
        (value for value in data.values() if isinstance(value, list)),
        None
    )

    if records is None:
        raise ValueError("Could not find a list of records.")

else:
    raise ValueError("Unexpected JSON structure.")

print("\nTOTAL RECORDS:", len(records))

invalid = sum(
    not isinstance(record, dict)
    for record in records
)

valid_records = [
    record for record in records
    if isinstance(record, dict)
]

print("INVALID RECORDS:", invalid)

# Field frequency
key_counts = Counter(
    key
    for record in valid_records
    for key in record.keys()
)

print("\nFIELD FREQUENCY")

for key, count in key_counts.most_common():
    print(f"{key}: {count}/{len(valid_records)}")

# Sample records
print("\n" + "=" * 65)
print("SAMPLE RECORDS")
print("=" * 65)

for i, record in enumerate(valid_records[:5], start=1):
    print(f"\nRECORD {i}")
    print(json.dumps(
        record,
        indent=2,
        ensure_ascii=False
    )[:5000])

# Inspect nested status values
print("\n" + "=" * 65)
print("STATUS DISTRIBUTION")
print("=" * 65)

statuses = Counter()

for record in valid_records:
    status = record.get("status")

    if isinstance(status, dict):
        status = status.get("name")

    statuses[str(status)] += 1

for status, count in statuses.most_common():
    print(f"{status}: {count}")


LIVINGETHIO SCHEMA INSPECTION

TOP-LEVEL TYPE: list

TOTAL RECORDS: 813
INVALID RECORDS: 0

FIELD FREQUENCY
id: 813/813
title: 813/813
content: 813/813
price: 813/813
price_usd: 813/813
price_prefix: 813/813
price_suffix: 813/813
video_url: 813/813
bedrooms: 813/813
bathrooms: 813/813
area: 813/813
land_area: 813/813
garage: 813/813
address: 813/813
latitude: 813/813
longitude: 813/813
note: 813/813
isfeatured: 813/813
post_status: 813/813
contact: 813/813
contact_phone: 813/813
contact_name: 813/813
createdAt: 813/813
updatedAt: 813/813
statusId: 813/813
status_id: 813/813
labelId: 813/813
label_id: 813/813
locationId: 813/813
location_id: 813/813
userId: 813/813
user_id: 813/813
editor_id: 813/813
bldg_id: 813/813
status: 813/813
label: 813/813
location: 813/813
building: 813/813
author: 813/813
fans: 813/813
units: 813/813
type: 813/813
features: 813/813
images: 813/813
leads: 813/813
inquiries: 813/813
reviews: 813/813

SAMPLE RECORDS

RECORD 1
{
  "id": 1144,
  "title": "FURNISHED

In [155]:

import json
import re
import html
import pandas as pd
from pathlib import Path
from collections import Counter

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------

file_path = Path(
    "data/raw/real_estate/raw/livingethio/livingethio.json"
)

with open(file_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Raw records:", len(records))


# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------

def extract_number(value):
    if value is None:
        return None

    match = re.search(
        r"\d[\d,]*(?:\.\d+)?",
        str(value)
    )

    if not match:
        return None

    try:
        return float(match.group().replace(",", ""))
    except ValueError:
        return None


def clean_html(value):
    """Remove HTML tags and decode HTML entities."""
    if not value:
        return ""

    text = re.sub(r"<[^>]+>", " ", str(value))
    return html.unescape(text).strip()


def get_nested_name(value):
    """Safely extract a name from a nested object."""
    if isinstance(value, dict):
        return str(value.get("name") or "").strip()

    return ""


def get_property_types(record):
    """Extract property type names from the nested type list."""
    types = record.get("type")

    if not isinstance(types, list):
        return []

    return [
        str(item.get("name")).strip()
        for item in types
        if isinstance(item, dict) and item.get("name")
    ]


def detect_period(price_text, description):
    text = f"{price_text} {description}".lower()

    if re.search(r"/\s*mo\b|per\s+month|monthly", text):
        return "monthly"

    if re.search(r"/\s*week\b|per\s+week|weekly", text):
        return "weekly"

    if re.search(r"/\s*day\b|per\s+day|daily", text):
        return "daily"

    if re.search(r"/\s*year\b|per\s+year|yearly|annually", text):
        return "yearly"

    return "unspecified"


# ---------------------------------------------------------
# 3. EXTRACT RENTAL LISTINGS
# ---------------------------------------------------------

residential_types = {
    "apartment",
    "house",
    "condo",
    "villa",
    "townhouse",
    "studio",
    "bedsitter",
    "room",
    "bungalow",
    "duplex",
    "mansion",
}

extracted = []

for record in records:

    if not isinstance(record, dict):
        continue

    status = get_nested_name(record.get("status"))

    # Keep only rental listings
    if status.lower() != "for rent":
        continue

    property_types = get_property_types(record)

    # Keep residential property types
    matching_types = [
        ptype for ptype in property_types
        if ptype.lower() in residential_types
    ]

    if not matching_types:
        continue

    title = str(record.get("title") or "")
    description = clean_html(record.get("content"))
    address = str(record.get("address") or "")
    location = get_nested_name(record.get("location"))

    raw_price = extract_number(record.get("price"))
    raw_price_usd = extract_number(record.get("price_usd"))

    currency = str(
        record.get("price_prefix") or ""
    ).strip().upper()

    if currency not in {"ETB", "USD"}:
        currency = None

    price_text = " ".join([
        str(record.get("price") or ""),
        str(record.get("price_prefix") or ""),
        str(record.get("price_suffix") or ""),
    ])

    period = detect_period(price_text, description)

    description_lower = description.lower()

    description_mentions_usd = bool(
        re.search(r"\busd\b|\$", description_lower)
    )

    description_mentions_etb = bool(
        re.search(r"\betb\b|\bbirr\b", description_lower)
    )

    currency_conflict = (
        (currency == "ETB" and description_mentions_usd)
        or
        (currency == "USD" and description_mentions_etb)
    )

    review_reasons = []

    if raw_price is None or raw_price <= 0:
        review_reasons.append("invalid_price")

    if currency is None:
        review_reasons.append("missing_currency")

    if period == "unspecified":
        review_reasons.append("unspecified_period")

    if description_mentions_usd:
        review_reasons.append("description_mentions_usd")

    if currency_conflict:
        review_reasons.append("currency_conflict")

    # Strict monthly ETB eligibility
    monthly_rent = None

    if (
        raw_price is not None
        and raw_price > 0
        and currency == "ETB"
        and period == "monthly"
        and not description_mentions_usd
        and not currency_conflict
    ):
        monthly_rent = raw_price

    extracted.append({
        "source": "livingethio",
        "listing_id": record.get("id"),

        "title": title,
        "description": description,
        "property_type": "; ".join(matching_types),
        "address": address,
        "location": location,

        "raw_price": raw_price,
        "raw_price_usd": raw_price_usd,
        "price_prefix": record.get("price_prefix"),
        "price_suffix": record.get("price_suffix"),
        "currency": currency,
        "price_period": period,
        "monthly_rent": monthly_rent,

        "description_mentions_usd": description_mentions_usd,
        "description_mentions_etb": description_mentions_etb,
        "currency_conflict": currency_conflict,
        "needs_currency_review": bool(review_reasons),
        "currency_review_reason": "; ".join(review_reasons),

        "bedrooms": extract_number(record.get("bedrooms")),
        "bathrooms": extract_number(record.get("bathrooms")),
        "size_sqm": extract_number(record.get("area")),
        "land_area_sqm": extract_number(record.get("land_area")),
        "garage": extract_number(record.get("garage")),
        "furnished": get_nested_name(record.get("label")),

        "post_status": record.get("post_status"),
        "date_published": record.get("createdAt"),
        "date_modified": record.get("updatedAt"),

        "latitude": record.get("latitude"),
        "longitude": record.get("longitude"),
    })


# ---------------------------------------------------------
# 4. CREATE DATAFRAME
# ---------------------------------------------------------

livingethio_df = pd.DataFrame(extracted)

print("\nExtracted residential rental records:", len(livingethio_df))


# ---------------------------------------------------------
# 5. DUPLICATE AUDIT
# ---------------------------------------------------------

duplicates = livingethio_df[
    livingethio_df["listing_id"].duplicated(keep=False)
].copy()

print("\nDuplicate rows:", len(duplicates))
print(
    "Duplicate listing IDs:",
    duplicates["listing_id"].nunique()
)

livingethio_df = livingethio_df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
).copy()

print("Records after deduplication:", len(livingethio_df))


# ---------------------------------------------------------
# 6. CREATE SUBSETS
# ---------------------------------------------------------

livingethio_review = livingethio_df[
    livingethio_df["needs_currency_review"]
].copy()

livingethio_monthly_etb = livingethio_df[
    livingethio_df["monthly_rent"].notna()
].copy()


# ---------------------------------------------------------
# 7. SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("LIVINGETHIO EXTRACTION SUMMARY")
print("=" * 65)

print("\nRECORD COUNTS")
print("-" * 50)
print("Residential rental records:", len(livingethio_df))
print("Currency review candidates:", len(livingethio_review))
print("Monthly ETB candidates:", len(livingethio_monthly_etb))

print("\nPROPERTY TYPE DISTRIBUTION")
print(livingethio_df["property_type"].value_counts())

print("\nLOCATION DISTRIBUTION")
print(livingethio_df["location"].value_counts().head(30))

print("\nCURRENCY DISTRIBUTION")
print(livingethio_df["currency"].value_counts(dropna=False))

print("\nPRICE PERIOD DISTRIBUTION")
print(livingethio_df["price_period"].value_counts(dropna=False))

print("\nCURRENCY REVIEW REASONS")

for column in [
    "description_mentions_usd",
    "description_mentions_etb",
    "currency_conflict",
    "needs_currency_review",
]:
    print(
        f"{column}:",
        livingethio_df[column].fillna(False).sum()
    )

print("\nMISSING VALUES")
print(
    livingethio_df[
        [
            "raw_price",
            "raw_price_usd",
            "currency",
            "price_period",
            "bedrooms",
            "bathrooms",
            "size_sqm",
        ]
    ].isna().sum()
)

print("\nMONTHLY ETB PRICE DISTRIBUTION")

if not livingethio_monthly_etb.empty:
    print(
        livingethio_monthly_etb["monthly_rent"].describe(
            percentiles=[
                0.01, 0.05, 0.25,
                0.50, 0.75, 0.95, 0.99
            ]
        )
    )

print("\nSAMPLE CURRENCY REVIEW RECORDS")

display(
    livingethio_review[
        [
            "listing_id",
            "title",
            "raw_price",
            "raw_price_usd",
            "price_prefix",
            "currency",
            "price_period",
            "description",
            "currency_review_reason",
        ]
    ].head(15)
)


# ---------------------------------------------------------
# 8. SAVE OUTPUTS
# ---------------------------------------------------------

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

livingethio_df.to_csv(
    output_dir / "livingethio_all_extracted.csv",
    index=False
)

livingethio_review.to_csv(
    output_dir / "livingethio_currency_review.csv",
    index=False
)

livingethio_monthly_etb.to_csv(
    output_dir / "livingethio_monthly_etb_candidates.csv",
    index=False
)

duplicates.to_csv(
    output_dir / "livingethio_duplicate_audit.csv",
    index=False
)

print("\nSAVED FILES")
print("data/processed/livingethio_all_extracted.csv")
print("data/processed/livingethio_currency_review.csv")
print("data/processed/livingethio_monthly_etb_candidates.csv")
print("data/processed/livingethio_duplicate_audit.csv")


Raw records: 813

Extracted residential rental records: 283

Duplicate rows: 0
Duplicate listing IDs: 0
Records after deduplication: 283

LIVINGETHIO EXTRACTION SUMMARY

RECORD COUNTS
--------------------------------------------------
Residential rental records: 283
Currency review candidates: 267
Monthly ETB candidates: 14

PROPERTY TYPE DISTRIBUTION
property_type
Apartment    195
House         75
Villa         10
Condo          3
Name: count, dtype: int64

LOCATION DISTRIBUTION
location
Bole               39
Mexico             25
Bole Atlas         18
Bisrate Gebriel    16
Summit             15
Bole Bulbula       12
CMC                11
22                 11
Megenagna          11
Ayat               10
Gotera              9
Kazanchis           8
Dembel              7
Meskel Flower       7
Bole Rwanda         7
Arat Kilo           6
Wollo Sefer         6
Gurd Shola          4
Summit Figa         4
Sarbet              4
Bole Olympia        3
Kebena              3
Gerji               3


,listing_id,title,raw_price,raw_price_usd,price_prefix,currency,price_period,description,currency_review_reason
0,1144,FURNISHED APARTMENT FOR RENT,166205.71,3000.0,USD,USD,unspecified,Three bedroom fully furnished apartment availa...,unspecified_period; description_mentions_usd
4,1126,Fully Furnished Apartment,60000.00,NaN,ETB,ETB,unspecified,Fully Furnished Apartment with: Living and din...,unspecified_period
11,1098,Apartment for rent,54000.00,NaN,ETB,ETB,unspecified,,unspecified_period
13,1095,2 Bedroom Apartment For Rent,94539.66,1700.0,USD,USD,unspecified,,unspecified_period
14,1092,House For Rent,233568.57,4200.0,USD,USD,unspecified,Fully furnished house that has spacious living...,unspecified_period
16,1084,Apartment For Sale,167750.00,NaN,ETB,ETB,unspecified,Mixed used type of building ( G+18) Building ...,unspecified_period; description_mentions_usd; ...
18,1082,G+3 Furnished House,100000.00,NaN,ETB,ETB,unspecified,"G+3 furnished house that store room, laundry, ...",unspecified_period
20,1079,House For Rent,120000.00,NaN,ETB,ETB,unspecified,Suitable for hotel and guest house as well.,unspecified_period
21,1078,2 Bedroom Apartment,50000.00,NaN,ETB,ETB,unspecified,"Unfurnished apartment that has water tanker, s...",unspecified_period
23,1076,House For Rent,70000.00,NaN,ETB,ETB,unspecified,,unspecified_period



SAVED FILES
data/processed/livingethio_all_extracted.csv
data/processed/livingethio_currency_review.csv
data/processed/livingethio_monthly_etb_candidates.csv
data/processed/livingethio_duplicate_audit.csv


In [156]:
import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/ethiopianproperties/any_any_2024_02_04.json"
)

with file_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

print("TOP-LEVEL TYPE:", type(data).__name__)

if isinstance(data, dict):
    print("TOP-LEVEL KEYS:", list(data.keys()))

    # Identify the list containing the property records.
    records = next(
        (value for value in data.values() if isinstance(value, list)),
        []
    )
elif isinstance(data, list):
    records = data
else:
    records = []

print("\nTOTAL RECORDS:", len(records))

valid_records = [r for r in records if isinstance(r, dict)]
print("VALID RECORDS:", len(valid_records))
print("INVALID RECORDS:", len(records) - len(valid_records))

if valid_records:
    print("\nFIELD NAMES:")
    print(sorted(set().union(*(r.keys() for r in valid_records))))

    print("\nMISSING VALUE COUNTS:")
    all_keys = sorted(set().union(*(r.keys() for r in valid_records)))

    for key in all_keys:
        missing = sum(
            r.get(key) is None or str(r.get(key)).strip() == ""
            for r in valid_records
        )
        print(f"{key}: {missing}/{len(valid_records)}")

    # Inspect useful listing fields only.
    fields_to_inspect = [
        "id",
        "title",
        "property_type",
        "property_status",
        "listing_type",
        "price",
        "address",
        "description",
        "bedrooms",
        "bathrooms",
        "size",
        "area",
    ]

    print("\nSAMPLE RECORDS:")
    for record in valid_records[:5]:
        print("\n" + "=" * 60)
        for field in fields_to_inspect:
            if field in record:
                value = record[field]
                if isinstance(value, str):
                    value = value[:500]
                print(f"{field}: {value}")

    # Inspect the most common values in important fields.
    for field in ["property_type", "property_status", "listing_type"]:
        if field in valid_records[0]:
            print(f"\n{field.upper()} DISTRIBUTION:")
            print(Counter(
                str(r.get(field)) for r in valid_records
            ).most_common(20))

TOP-LEVEL TYPE: list

TOTAL RECORDS: 704
VALID RECORDS: 704
INVALID RECORDS: 0

FIELD NAMES:
['Bathroom', 'Bathrooms', 'Bedroom', 'Bedrooms', 'Garage', 'Garages', 'SQ M', 'Sq', 'Sq  M', 'Sq Ft', 'Sq M', 'Sq Meter', 'Sq Mt', 'Sq m', 'additional_details', 'address', 'agent-address', 'agent-name', 'date_modified', 'date_published', 'description', 'features', 'images', 'latitude', 'longitude', 'price', 'property_id', 'property_status', 'property_type', 'property_url', 'sq mts', 'title', 'unknown or sqm']

MISSING VALUE COUNTS:
Bathroom: 673/704
Bathrooms: 98/704
Bedroom: 690/704
Bedrooms: 85/704
Garage: 540/704
Garages: 260/704
SQ M: 703/704
Sq: 703/704
Sq  M: 698/704
Sq Ft: 698/704
Sq M: 148/704
Sq Meter: 703/704
Sq Mt: 702/704
Sq m: 703/704
additional_details: 699/704
address: 0/704
agent-address: 23/704
agent-name: 18/704
date_modified: 0/704
date_published: 0/704
description: 3/704
features: 16/704
images: 0/704
latitude: 653/704
longitude: 653/704
price: 44/704
property_id: 10/704
pro

In [157]:
import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/ethiopianproperties/any_any_2024_02_04.json"
)

with file_path.open("r", encoding="utf-8") as file:
    records = json.load(file)

valid_records = [
    r for r in records
    if isinstance(r, dict)
]

# 1. Price format distribution
print("=" * 70)
print("PRICE FORMAT DISTRIBUTION")
print("=" * 70)

price_formats = Counter()

for record in valid_records:
    price = str(record.get("price") or "").strip()

    if not price:
        label = "Missing"
    elif "$" in price or "USD" in price.upper():
        label = "USD"
    elif "ETB" in price.upper() or "BIRR" in price.upper():
        label = "ETB"
    else:
        label = "Unclear / Other"

    price_formats[label] += 1

print(price_formats)

# 2. Exact price examples
print("\n" + "=" * 70)
print("PRICE EXAMPLES")
print("=" * 70)

for label in ["USD", "ETB", "Unclear / Other"]:
    print(f"\n{label} examples:")

    examples = []
    for record in valid_records:
        price = str(record.get("price") or "").strip()

        if label == "USD" and ("$" in price or "USD" in price.upper()):
            examples.append(price)
        elif label == "ETB" and (
            "ETB" in price.upper() or "BIRR" in price.upper()
        ):
            examples.append(price)
        elif label == "Unclear / Other" and price and not (
            "$" in price
            or "USD" in price.upper()
            or "ETB" in price.upper()
            or "BIRR" in price.upper()
        ):
            examples.append(price)

    for example in list(dict.fromkeys(examples))[:15]:
        print(example)

# 3. Property type distribution for rental records
print("\n" + "=" * 70)
print("PROPERTY TYPES: RENTAL RECORDS")
print("=" * 70)

rental_records = [
    r for r in valid_records
    if "For Rent" in str(r.get("property_status", ""))
]

print("Rental records:", len(rental_records))

print(
    Counter(
        str(r.get("property_type") or "Missing")
        for r in rental_records
    ).most_common(40)
)

# 4. Ambiguous rental statuses
print("\n" + "=" * 70)
print("AMBIGUOUS RENTAL STATUSES")
print("=" * 70)

ambiguous = [
    r for r in valid_records
    if (
        "For Rent, For Sale" in str(r.get("property_status", ""))
        or "For Rent, Leased" in str(r.get("property_status", ""))
    )
]

for record in ambiguous[:10]:
    print({
        "property_id": record.get("property_id"),
        "title": record.get("title"),
        "property_type": record.get("property_type"),
        "property_status": record.get("property_status"),
        "price": record.get("price"),
        "address": record.get("address"),
    })

# 5. Look for explicit billing periods in rental descriptions
print("\n" + "=" * 70)
print("BILLING PERIOD INDICATORS")
print("=" * 70)

period_terms = [
    "per month", "monthly", "/month", "/mo", "month",
    "per week", "weekly", "/week",
    "per day", "daily", "/day",
    "per year", "yearly", "annually", "/year"
]

period_counts = Counter()

for record in rental_records:
    text = (
        str(record.get("title") or "") + " " +
        str(record.get("description") or "")
    ).lower()

    found = [term for term in period_terms if term in text]

    if found:
        period_counts[", ".join(found)] += 1
    else:
        period_counts["No explicit period"] += 1

print(period_counts)

PRICE FORMAT DISTRIBUTION
Counter({'USD': 660, 'Missing': 44})

PRICE EXAMPLES

USD examples:
$4,000.00
$7,000.00
$2,500.00
$8,000.00
$1,482.00 Per Sq M
$1,500.00
$1,000.00
$13,000.00 (Negotiable)
$6,000.00 (Negotiable)
$1.00
$3,500.00
$10,000,000.00 Birr
$90,000,000.00 Birr
$32,000,000.00 Birr
$2,000.00

ETB examples:
$10,000,000.00 Birr
$90,000,000.00 Birr
$32,000,000.00 Birr
$13,000,000.00 Birr
$15,000,000.00 ET Birr
$20,000,000.00 ET Birr
$14,000,000.00 Birr
$15,000,000.00 Birr
$2,100,000.00 Birr
$2,000,000.00 birr
$5,200,000.00 Birr
$26,000,000.00 Birr
$9,300,000.00 Birr
$8,500,000.00 Birr
$23.00 Birr

Unclear / Other examples:

PROPERTY TYPES: RENTAL RECORDS
Rental records: 586
[('House, Residential', 192), ('House', 182), ('Apartment, Residential', 73), ('Apartment', 51), ('Commercial, Office', 20), ('Commercial, House, Residential', 11), ('Office', 6), ('Commercial, House, Office, Residential', 5), ('Residential, Serviced Apartment', 5), ('Hotel', 4), ('Warehouse', 4), ('Vacati

In [158]:
import json
import re
import pandas as pd

from pathlib import Path


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

file_path = Path(
    "data/raw/real_estate/raw/ethiopianproperties/"
    "any_any_2024_02_04.json"
)

with file_path.open("r", encoding="utf-8") as file:
    records = json.load(file)


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def extract_number(value):
    if value is None:
        return None

    match = re.search(r"[\d,]+(?:\.\d+)?", str(value))

    if not match:
        return None

    return float(match.group().replace(",", ""))


def detect_currency(price):
    text = str(price or "").strip().upper()

    # Explicit Birr/ETB takes precedence over dollar symbols.
    if "BIRR" in text or "ETB" in text:
        return "ETB"

    if "$" in text or "USD" in text:
        return "USD"

    return "unknown"


def detect_period(price, description, title):
    text = " ".join([
        str(price or ""),
        str(description or ""),
        str(title or "")
    ]).lower()

    # Explicit period indicators only.
    if re.search(r"\b(per month|monthly|/month|/mo)\b", text):
        return "monthly"

    if re.search(r"\b(per week|weekly|/week)\b", text):
        return "weekly"

    if re.search(r"\b(per day|daily|/day)\b", text):
        return "daily"

    if re.search(r"\b(per year|yearly|annually|/year)\b", text):
        return "yearly"

    # Avoid inferring a billing period from generic "month" mentions.
    return "unspecified"


def is_residential(property_type):
    text = str(property_type or "").lower()

    residential_terms = [
        "house",
        "apartment",
        "residential",
        "condominium",
        "serviced apartment",
        "vacation rental",
    ]

    return any(term in text for term in residential_terms)


def has_mixed_use(property_type):
    text = str(property_type or "").lower()

    commercial_terms = [
        "commercial",
        "office",
        "hotel",
        "warehouse",
        "retail",
        "shop",
        "building/mall",
    ]

    return any(term in text for term in commercial_terms)


# --------------------------------------------------
# 3. Extract records
# --------------------------------------------------

extracted = []

for record in records:

    if not isinstance(record, dict):
        continue

    status = str(record.get("property_status") or "").strip()
    address = str(record.get("address") or "").strip()

    # Keep Addis Ababa listings only.
    if "addis ababa" not in address.lower():
        continue

    # Retain listings with rental intent.
    if "for rent" not in status.lower():
        continue

    title = record.get("title")
    description = record.get("description")
    property_type = record.get("property_type")
    raw_price = record.get("price")

    currency = detect_currency(raw_price)

    price_value = extract_number(raw_price)

    period = detect_period(
        raw_price,
        description,
        title
    )

    residential = is_residential(property_type)
    mixed_use = has_mixed_use(property_type)

    # Strict monthly ETB eligibility.
    monthly_rent = None

    if (
        residential
        and not mixed_use
        and status == "For Rent"
        and currency == "ETB"
        and period == "monthly"
        and price_value is not None
        and price_value > 0
    ):
        monthly_rent = price_value

    extracted.append({
        "source": "ethiopianproperties",

        "listing_id": record.get("property_id"),
        "title": title,
        "property_type": property_type,
        "property_status": status,

        "address": address,
        "description": description,

        "raw_price": raw_price,
        "raw_price_value": price_value,
        "currency": currency,
        "price_period": period,
        "monthly_rent": monthly_rent,

        "bedrooms": extract_number(
            record.get("Bedrooms") or record.get("Bedroom")
        ),

        "bathrooms": extract_number(
            record.get("Bathrooms") or record.get("Bathroom")
        ),

        # Preserve the original size fields.
        # Units are not assumed to be consistent yet.
        "size_raw": (
            record.get("Sq M")
            or record.get("SQ M")
            or record.get("Sq  M")
            or record.get("Sq m")
            or record.get("Sq Meter")
            or record.get("Sq Mt")
            or record.get("sq mts")
            or record.get("Sq Ft")
            or record.get("Sq")
        ),

        "garage": (
            record.get("Garages")
            or record.get("Garage")
        ),

        "date_published": record.get("date_published"),
        "date_modified": record.get("date_modified"),

        "latitude": record.get("latitude"),
        "longitude": record.get("longitude"),

        "is_residential": residential,
        "is_mixed_use": mixed_use,
    })


# --------------------------------------------------
# 4. Create DataFrame
# --------------------------------------------------

ethiopianproperties_df = pd.DataFrame(extracted)

# Remove duplicate listing IDs where available.
ethiopianproperties_df = (
    ethiopianproperties_df.drop_duplicates(
        subset=["listing_id"],
        keep="first"
    )
    .copy()
)


# --------------------------------------------------
# 5. Build review and modeling datasets
# --------------------------------------------------

ethiopianproperties_review = ethiopianproperties_df[
    (
        ethiopianproperties_df["currency"].isin(
            ["USD", "unknown"]
        )
    )
    | (
        ethiopianproperties_df["price_period"] == "unspecified"
    )
    | ethiopianproperties_df["is_mixed_use"]
    | (
        ethiopianproperties_df["property_status"] != "For Rent"
    )
].copy()


ethiopianproperties_monthly_etb = (
    ethiopianproperties_df[
        ethiopianproperties_df["monthly_rent"].notna()
    ]
    .copy()
)


# --------------------------------------------------
# 6. Save datasets
# --------------------------------------------------

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

ethiopianproperties_df.to_csv(
    output_dir / "ethiopianproperties_all_extracted.csv",
    index=False
)

ethiopianproperties_review.to_csv(
    output_dir / "ethiopianproperties_currency_review.csv",
    index=False
)

ethiopianproperties_monthly_etb.to_csv(
    output_dir / "ethiopianproperties_monthly_etb_candidates.csv",
    index=False
)


# --------------------------------------------------
# 7. Summary
# --------------------------------------------------

print("=" * 70)
print("ETHIOPIANPROPERTIES EXTRACTION SUMMARY")
print("=" * 70)

print("Extracted records:", len(ethiopianproperties_df))
print("Review candidates:", len(ethiopianproperties_review))
print("Monthly ETB candidates:", len(ethiopianproperties_monthly_etb))

print("\nCurrency distribution:")
print(ethiopianproperties_df["currency"].value_counts(dropna=False))

print("\nPrice period distribution:")
print(ethiopianproperties_df["price_period"].value_counts(dropna=False))

print("\nProperty types:")
print(ethiopianproperties_df["property_type"].value_counts(dropna=False).head(20))

print("\nProperty statuses:")
print(ethiopianproperties_df["property_status"].value_counts(dropna=False))

print("\nMissing values:")
print(
    ethiopianproperties_df[
        ["listing_id", "raw_price_value", "bedrooms", "bathrooms", "size_raw"]
    ].isna().sum()
)

print("\nMonthly ETB rent statistics:")
if not ethiopianproperties_monthly_etb.empty:
    print(
        ethiopianproperties_monthly_etb["monthly_rent"].describe()
    )
else:
    print("No confirmed monthly ETB candidates.")

print("\nSaved:")
print("- ethiopianproperties_all_extracted.csv")
print("- ethiopianproperties_currency_review.csv")
print("- ethiopianproperties_monthly_etb_candidates.csv")

ETHIOPIANPROPERTIES EXTRACTION SUMMARY
Extracted records: 561
Review candidates: 561
Monthly ETB candidates: 0

Currency distribution:
currency
USD        535
unknown     22
ETB          4
Name: count, dtype: int64

Price period distribution:
price_period
unspecified    317
monthly        241
daily            2
yearly           1
Name: count, dtype: int64

Property types:
property_type
House, Residential                            188
House                                         167
Apartment, Residential                         73
Apartment                                      49
Commercial, Office                             20
Commercial, House, Residential                 11
Office                                          6
Commercial, House, Office, Residential          5
Residential, Serviced Apartment                 5
Hotel                                           4
Building/Mall, Commercial                       3
Warehouse                                       3
NaN        

In [159]:
import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/zegebeya/residential_2024-02-08.json"
)

with file_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

print("TOP-LEVEL TYPE:", type(data).__name__)

if isinstance(data, list):
    records = data
elif isinstance(data, dict):
    print("TOP-LEVEL KEYS:", list(data.keys()))
    records = next(
        (value for value in data.values() if isinstance(value, list)),
        []
    )
else:
    records = []

valid_records = [
    record for record in records
    if isinstance(record, dict)
]

print("\nTOTAL RECORDS:", len(records))
print("VALID RECORDS:", len(valid_records))
print("INVALID RECORDS:", len(records) - len(valid_records))

if valid_records:
    all_keys = sorted(
        set().union(*(record.keys() for record in valid_records))
    )

    print("\nFIELD NAMES:")
    print(all_keys)

    print("\nMISSING VALUE COUNTS:")
    for key in all_keys:
        missing = sum(
            record.get(key) is None
            or str(record.get(key)).strip() == ""
            for record in valid_records
        )
        print(f"{key}: {missing}/{len(valid_records)}")

    print("\nSAMPLE RECORDS:")
    for record in valid_records[:5]:
        print("\n" + "=" * 60)
        for key, value in record.items():
            if key.lower() in {
                "contact", "phone", "email", "agent",
                "author", "password"
            }:
                continue

            if isinstance(value, str):
                value = value[:500]

            print(f"{key}: {value}")

    # Inspect common values for likely classification fields.
    for field in [
        "property_status",
        "status",
        "property_type",
        "type",
        "category",
        "price_currency",
        "currency",
    ]:
        if field in valid_records[0]:
            print(f"\n{field.upper()} DISTRIBUTION:")
            print(
                Counter(
                    str(record.get(field))
                    for record in valid_records
                ).most_common(30)
            )

TOP-LEVEL TYPE: list

TOTAL RECORDS: 547
VALID RECORDS: 547
INVALID RECORDS: 0

FIELD NAMES:
['Bathrooms', 'Bedrooms', 'Car Parking', 'Lot Size', 'Total Area', 'Year Built', 'address', 'agent-name', 'contacts-list', 'date_modified', 'date_published', 'description', 'features', 'latitude', 'longitude', 'num_images', 'price', 'property_id', 'property_status', 'property_type', 'property_url', 'title']

MISSING VALUE COUNTS:
Bathrooms: 15/547
Bedrooms: 17/547
Car Parking: 542/547
Lot Size: 546/547
Total Area: 34/547
Year Built: 237/547
address: 0/547
agent-name: 199/547
contacts-list: 199/547
date_modified: 0/547
date_published: 0/547
description: 8/547
features: 21/547
latitude: 0/547
longitude: 0/547
num_images: 0/547
price: 0/547
property_id: 0/547
property_status: 4/547
property_type: 0/547
property_url: 0/547
title: 0/547

SAMPLE RECORDS:

property_id: ZG-15701-property
property_url: https://zegebeya.com/property/apartments-for-sale-in-addis-ababa-around-kebena/
property_type: Apartme

In [160]:
import json
import re
import pandas as pd

from pathlib import Path


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

file_path = Path(
    "data/raw/real_estate/raw/zegebeya/"
    "residential_2024-02-08.json"
)

with file_path.open("r", encoding="utf-8") as file:
    records = json.load(file)


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def extract_number(value):
    if value is None:
        return None

    match = re.search(r"[\d,]+(?:\.\d+)?", str(value))

    if not match:
        return None

    return float(match.group().replace(",", ""))


def detect_currency(price, description):
    price_text = str(price or "").upper()
    description_text = str(description or "").upper()

    # Explicit ETB/Birr takes precedence.
    if re.search(r"\bETB\b|\bBIRR\b", price_text):
        return "ETB"

    if re.search(r"\bUSD\b", price_text) or "$" in price_text:
        return "USD"

    # Description is secondary evidence.
    if re.search(r"\bETB\b|\bBIRR\b", description_text):
        if re.search(r"\bUSD\b", description_text) or "$" in description_text:
            return "review"
        return "ETB"

    if re.search(r"\bUSD\b", description_text) or "$" in description_text:
        return "USD"

    return "unknown"


def detect_period(price, description, title):
    text = " ".join([
        str(price or ""),
        str(description or ""),
        str(title or "")
    ]).lower()

    if re.search(r"\b(per month|monthly|/month|/mo)\b", text):
        return "monthly"

    if re.search(r"\b(per week|weekly|/week)\b", text):
        return "weekly"

    if re.search(r"\b(per day|daily|/day)\b", text):
        return "daily"

    if re.search(r"\b(per year|yearly|annually|/year)\b", text):
        return "yearly"

    return "unspecified"


def is_residential(property_type):
    text = str(property_type or "").lower()

    residential_terms = [
        "house",
        "apartment",
        "residential",
        "villa",
        "condominium",
        "furnished apartment",
        "furnished house",
    ]

    return any(term in text for term in residential_terms)


def is_mixed_use(property_type):
    text = str(property_type or "").lower()

    commercial_terms = [
        "commercial",
        "office",
        "shop",
        "warehouse",
        "hotel",
        "land & plot",
        "apartment building",
    ]

    return any(term in text for term in commercial_terms)


def parse_area(value):
    if value is None:
        return None, None

    text = str(value).strip()

    number = extract_number(text)

    if number is None:
        return None, None

    text_lower = text.lower()

    if "square meter" in text_lower or "sqm" in text_lower:
        unit = "sqm"
    elif "square feet" in text_lower or "sqft" in text_lower:
        unit = "sqft"
    else:
        unit = "unknown"

    return number, unit


# --------------------------------------------------
# 3. Extract records
# --------------------------------------------------

extracted = []

for record in records:

    if not isinstance(record, dict):
        continue

    address = str(record.get("address") or "")
    status = str(record.get("property_status") or "").strip()
    property_type = record.get("property_type")

    if "addis ababa" not in address.lower():
        continue

    # Only records explicitly carrying rental intent.
    if "for rent" not in status.lower():
        continue

    title = record.get("title")
    description = record.get("description")
    raw_price = record.get("price")

    currency = detect_currency(raw_price, description)

    price_value = extract_number(raw_price)

    period = detect_period(
        raw_price,
        description,
        title
    )

    residential = is_residential(property_type)
    mixed_use = is_mixed_use(property_type)

    area_value, area_unit = parse_area(
        record.get("Total Area")
    )

    # Active rental status only.
    active_rental = (
        "for rent" in status.lower()
        and "rented out" not in status.lower()
        and "not available" not in status.lower()
    )

    monthly_rent = None

    if (
        residential
        and not mixed_use
        and active_rental
        and currency == "ETB"
        and period == "monthly"
        and price_value is not None
        and price_value > 0
    ):
        monthly_rent = price_value

    extracted.append({
        "source": "zegebeya",

        "listing_id": record.get("property_id"),
        "title": title,
        "property_type": property_type,
        "property_status": status,

        "address": address,
        "description": description,

        "raw_price": raw_price,
        "raw_price_value": price_value,
        "currency": currency,
        "price_period": period,
        "monthly_rent": monthly_rent,

        "bedrooms": extract_number(record.get("Bedrooms")),
        "bathrooms": extract_number(record.get("Bathrooms")),

        "size_value": area_value,
        "size_unit": area_unit,

        "year_built": extract_number(record.get("Year Built")),

        "features": record.get("features"),

        "date_published": record.get("date_published"),
        "date_modified": record.get("date_modified"),

        "latitude": record.get("latitude"),
        "longitude": record.get("longitude"),

        "is_residential": residential,
        "is_mixed_use": mixed_use,
        "active_rental": active_rental,
    })


# --------------------------------------------------
# 4. Create DataFrame and deduplicate
# --------------------------------------------------

zegebeya_df = pd.DataFrame(extracted)

zegebeya_df = zegebeya_df.drop_duplicates(
    subset=["listing_id"],
    keep="first"
).copy()


# --------------------------------------------------
# 5. Create review and modeling datasets
# --------------------------------------------------

zegebeya_review = zegebeya_df[
    (
        zegebeya_df["currency"].isin(["USD", "unknown", "review"])
    )
    | (
        zegebeya_df["price_period"] == "unspecified"
    )
    | zegebeya_df["is_mixed_use"]
    | (~zegebeya_df["active_rental"])
].copy()


zegebeya_monthly_etb = zegebeya_df[
    zegebeya_df["monthly_rent"].notna()
].copy()


# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

zegebeya_df.to_csv(
    output_dir / "zegebeya_all_extracted.csv",
    index=False
)

zegebeya_review.to_csv(
    output_dir / "zegebeya_currency_review.csv",
    index=False
)

zegebeya_monthly_etb.to_csv(
    output_dir / "zegebeya_monthly_etb_candidates.csv",
    index=False
)


# --------------------------------------------------
# 7. Summary
# --------------------------------------------------

print("=" * 70)
print("ZEGEBEYA EXTRACTION SUMMARY")
print("=" * 70)

print("Extracted records:", len(zegebeya_df))
print("Review candidates:", len(zegebeya_review))
print("Monthly ETB candidates:", len(zegebeya_monthly_etb))

print("\nCurrency distribution:")
print(zegebeya_df["currency"].value_counts(dropna=False))

print("\nPrice period distribution:")
print(zegebeya_df["price_period"].value_counts(dropna=False))

print("\nProperty statuses:")
print(zegebeya_df["property_status"].value_counts(dropna=False))

print("\nProperty types:")
print(zegebeya_df["property_type"].value_counts().head(20))

print("\nActive rental candidates:", zegebeya_df["active_rental"].sum())

print("\nMissing values:")
print(
    zegebeya_df[
        [
            "listing_id",
            "raw_price_value",
            "bedrooms",
            "bathrooms",
            "size_value",
        ]
    ].isna().sum()
)

print("\nMonthly ETB rent statistics:")

if not zegebeya_monthly_etb.empty:
    print(zegebeya_monthly_etb["monthly_rent"].describe())
else:
    print("No confirmed monthly ETB candidates.")

print("\nSaved:")
print("- zegebeya_all_extracted.csv")
print("- zegebeya_currency_review.csv")
print("- zegebeya_monthly_etb_candidates.csv")

ZEGEBEYA EXTRACTION SUMMARY
Extracted records: 241
Review candidates: 135
Monthly ETB candidates: 106

Currency distribution:
currency
ETB        188
USD         50
unknown      3
Name: count, dtype: int64

Price period distribution:
price_period
monthly        135
unspecified    106
Name: count, dtype: int64

Property statuses:
property_status
For Rent,Featured                             161
For Rent                                       61
For Rent,Rented Out,Featured                   11
For Rent,Not Available                          3
For Rent,Not Available,Rented Out,Featured      2
For Rent,Not Available,Featured                 1
For Rent,For Sale,Featured                      1
For Rent,Rented Out                             1
Name: count, dtype: int64

Property types:
property_type
Houses, Residential, Villa                                                 47
Houses, Residential                                                        37
Apartments, Furnished Apartment, Residen

In [163]:
import json
from pathlib import Path
from collections import Counter

file_path = Path(
    "data/raw/real_estate/raw/engocha/"
    "engocha_real-estate_2023-12-28.json"
)

with file_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

print("TOP-LEVEL TYPE:", type(data).__name__)

if isinstance(data, list):
    records = data
elif isinstance(data, dict):
    print("TOP-LEVEL KEYS:", list(data.keys()))
    records = next(
        (value for value in data.values() if isinstance(value, list)),
        []
    )
else:
    records = []

valid_records = [
    record for record in records
    if isinstance(record, dict)
]

print("\nTOTAL RECORDS:", len(records))
print("VALID RECORDS:", len(valid_records))
print("INVALID RECORDS:", len(records) - len(valid_records))

if valid_records:
    all_keys = sorted(
        set().union(*(record.keys() for record in valid_records))
    )

    print("\nFIELD NAMES:")
    print(all_keys)

    print("\nMISSING VALUE COUNTS:")
    for key in all_keys:
        missing = sum(
            record.get(key) is None
            or str(record.get(key)).strip() == ""
            for record in valid_records
        )
        print(f"{key}: {missing}/{len(valid_records)}")

    print("\nSAMPLE RECORDS:")
    for record in valid_records[:5]:
        print("\n" + "=" * 60)

        for key, value in record.items():
            if key.lower() in {
                "contact", "phone", "email", "agent",
                "author", "password"
            }:
                continue

            if isinstance(value, str):
                value = value[:500]

            print(f"{key}: {value}")

    for field in [
        "property_status",
        "status",
        "property_type",
        "type",
        "category",
        "price_currency",
        "currency",
    ]:
        if field in valid_records[0]:
            print(f"\n{field.upper()} DISTRIBUTION:")
            print(
                Counter(
                    str(record.get(field))
                    for record in valid_records
                ).most_common(30)
            )

TOP-LEVEL TYPE: list

TOTAL RECORDS: 1793
VALID RECORDS: 1793
INVALID RECORDS: 0

FIELD NAMES:
['Attrib1', 'Attrib2', 'Attrib3', 'Attrib4', 'Attrib5', 'Attrib6', 'Attrib7', 'Attrib8', 'Brand', 'BusinessID', 'BusinessName', 'City', 'Condition', 'Currency', 'Description', 'Featured', 'FeaturedTill', 'ImageCount', 'ListingDate', 'ListingID', 'ListingImage', 'Phone', 'Price', 'PriceType', 'PropertyType', 'RandomOrder', 'Status', 'TimeNow', 'Title', 'isFeatured', 'name']

MISSING VALUE COUNTS:
Attrib1: 180/1793
Attrib2: 264/1793
Attrib3: 441/1793
Attrib4: 795/1793
Attrib5: 1403/1793
Attrib6: 1751/1793
Attrib7: 1793/1793
Attrib8: 1793/1793
Brand: 1793/1793
BusinessID: 1563/1793
BusinessName: 1725/1793
City: 0/1793
Condition: 1793/1793
Currency: 0/1793
Description: 230/1793
Featured: 0/1793
FeaturedTill: 0/1793
ImageCount: 0/1793
ListingDate: 0/1793
ListingID: 0/1793
ListingImage: 1/1793
Phone: 0/1793
Price: 0/1793
PriceType: 707/1793
PropertyType: 0/1793
RandomOrder: 1563/1793
Status: 0/1793

In [165]:

import json
import re
from pathlib import Path

import pandas as pd

# --------------------------------------------------
# 1. Load raw data
# --------------------------------------------------

input_path = Path(
    "data/raw/real_estate/raw/engocha/"
    "engocha_real-estate_2023-12-28.json"
)

output_dir = Path("data/processed/engocha")
output_dir.mkdir(parents=True, exist_ok=True)

with open(input_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print("Raw records:", len(records))


# --------------------------------------------------
# 2. Helper functions
# --------------------------------------------------

def clean_text(value):
    if value is None:
        return ""
    return str(value).strip()


def parse_price(value):
    """Convert a numeric or formatted price into a number."""
    if value is None:
        return None

    if isinstance(value, (int, float)):
        return float(value)

    cleaned = re.sub(r"[^\d.]", "", str(value))

    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None



def parse_attribute(value):
    """
    Example:
    --1|-*|23|-*|Bedrooms--
    Returns: '1'

    Also handles empty and malformed values safely.
    """
    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    # Extract the value before the first separator.
    if value.startswith("--"):
        value = value[2:]

    if "|-*" in value:
        value = value.split("|-*", 1)[0]

    value = value.strip("- |")

    return value if value else None



def detect_period(text):
    """
    Only identify explicit rental periods.
    Do not assume that an unspecified price is monthly.
    """
    text = clean_text(text).lower()

    if re.search(
        r"per\s+month|monthly|/month\b|/mo\b|"
        r"month\s*ly|\bmonthly\s+rent\b",
        text
    ):
        return "monthly"

    if re.search(r"per\s+week|weekly|/week\b|/wk\b", text):
        return "weekly"

    if re.search(r"per\s+day|daily|/day\b", text):
        return "daily"

    if re.search(r"per\s+year|yearly|annually|/year\b", text):
        return "yearly"

    return "unspecified"


# --------------------------------------------------
# 3. Extract structured records
# --------------------------------------------------

rows = []

for record in records:
    if not isinstance(record, dict):
        continue

    title = clean_text(record.get("Title"))
    description = clean_text(record.get("Description"))
    property_type = clean_text(record.get("PropertyType"))
    city = clean_text(record.get("City"))
    status = clean_text(record.get("Status"))
    currency = clean_text(record.get("Currency")).upper()

    price = parse_price(record.get("Price"))

    # Combine text only for detecting an explicit price period.
    period_text = " ".join([
        title,
        description,
        clean_text(record.get("PriceType"))
    ])

    price_period = detect_period(period_text)

    # Extract the encoded attributes.
    attributes = {
        "bedrooms": parse_attribute(record.get("Attrib1")),
        "floor": parse_attribute(record.get("Attrib2")),
        "location_attribute": parse_attribute(record.get("Attrib3")),
        "bathrooms": parse_attribute(record.get("Attrib4")),
        "attribute_5": parse_attribute(record.get("Attrib5")),
        "attribute_6": parse_attribute(record.get("Attrib6")),
        "attribute_7": parse_attribute(record.get("Attrib7")),
        "attribute_8": parse_attribute(record.get("Attrib8")),
    }

    # Residential rental classification.
    property_type_lower = property_type.lower()

    residential_terms = [
        "apartment", "house", "villa",
        "residential", "condo", "room"
    ]

    excluded_terms = [
        "commercial", "office", "shop",
        "warehouse", "land", "hotel"
    ]

    is_residential = (
        any(term in property_type_lower for term in residential_terms)
        and not any(term in property_type_lower for term in excluded_terms)
    )

    is_rental = "rent" in property_type_lower

    is_addis = city.strip().lower() in [
        "addis ababa",
        "addis"
    ]

    is_available = status.strip().lower() == "available"

    # Monthly ETB rent is only populated when all conditions hold.
    valid_monthly_rent = (
        is_residential
        and is_rental
        and is_addis
        and is_available
        and currency == "ETB"
        and price_period == "monthly"
        and price is not None
        and price > 0
    )

    rows.append({
        "listing_id": clean_text(record.get("ListingID")),
        "title": title,
        "description": description,
        "property_type": property_type,
        "city": city,
        "status": status,
        "raw_price": clean_text(record.get("Price")),
        "price": price,
        "currency": currency,
        "price_type": clean_text(record.get("PriceType")),
        "price_period": price_period,
        "listing_date": clean_text(record.get("ListingDate")),
        "bedrooms": attributes["bedrooms"],
        "bathrooms": attributes["bathrooms"],
        "floor": attributes["floor"],
        "location_attribute": attributes["location_attribute"],
        "attribute_5": attributes["attribute_5"],
        "attribute_6": attributes["attribute_6"],
        "attribute_7": attributes["attribute_7"],
        "attribute_8": attributes["attribute_8"],
        "is_addis": is_addis,
        "is_rental": is_rental,
        "is_residential": is_residential,
        "is_available": is_available,
        "monthly_rent": price if valid_monthly_rent else None,
    })


# --------------------------------------------------
# 4. Create DataFrame and classify candidates
# --------------------------------------------------

df = pd.DataFrame(rows)

rental_candidates = df[
    df["is_rental"]
    & df["is_residential"]
    & df["is_addis"]
].copy()

monthly_candidates = rental_candidates[
    rental_candidates["monthly_rent"].notna()
].copy()

currency_review = rental_candidates[
    (rental_candidates["currency"] != "ETB")
    | (rental_candidates["price_period"] == "unspecified")
    | (
        rental_candidates["currency"] == "ETB"
    ) & (
        rental_candidates["price_period"] != "monthly"
    )
].copy()


# --------------------------------------------------
# 5. Duplicate audit
# --------------------------------------------------

duplicate_audit = df[
    df["listing_id"].duplicated(keep=False)
].sort_values("listing_id")


# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------

df.to_csv(
    output_dir / "engocha_all_extracted.csv",
    index=False
)

rental_candidates.to_csv(
    output_dir / "engocha_residential_rental_candidates.csv",
    index=False
)

monthly_candidates.to_csv(
    output_dir / "engocha_monthly_etb_candidates.csv",
    index=False
)

currency_review.to_csv(
    output_dir / "engocha_currency_review.csv",
    index=False
)

duplicate_audit.to_csv(
    output_dir / "engocha_duplicate_audit.csv",
    index=False
)


# --------------------------------------------------
# 7. Print extraction summary
# --------------------------------------------------

print("\n" + "=" * 60)
print("ENGOCHA EXTRACTION SUMMARY")
print("=" * 60)

print("Total extracted:", len(df))
print("Residential rental candidates:", len(rental_candidates))
print("Monthly ETB candidates:", len(monthly_candidates))
print("Currency/period review candidates:", len(currency_review))
print("Duplicate listing records:", len(duplicate_audit))

print("\nProperty type distribution:")
print(df["property_type"].value_counts(dropna=False).head(20))

print("\nRental candidate currency distribution:")
print(rental_candidates["currency"].value_counts(dropna=False))

print("\nRental candidate price period distribution:")
print(rental_candidates["price_period"].value_counts(dropna=False))

print("\nRental candidate status distribution:")
print(rental_candidates["status"].value_counts(dropna=False))

print("\nMissing values in rental candidates:")
print(rental_candidates.isna().sum())

print("\nMonthly rent statistics:")
if len(monthly_candidates):
    print(monthly_candidates["monthly_rent"].describe())
else:
    print("No confirmed monthly ETB candidates found.")

print("\nFiles saved to:", output_dir)


Raw records: 1793

ENGOCHA EXTRACTION SUMMARY
Total extracted: 1793
Residential rental candidates: 48
Monthly ETB candidates: 3
Currency/period review candidates: 45
Duplicate listing records: 230

Property type distribution:
property_type
Apartments for Sale           1017
Residential House for Sale     495
Shop & Office for Sale          86
Shop & Office for Rent          58
Condominium for Sale            41
Building for sale               25
Apartments for Rent             21
Land for sale                   21
Residential House for Rent      18
Condominium for Rent             9
Building for rent                2
Name: count, dtype: int64

Rental candidate currency distribution:
currency
ETB    42
USD     6
Name: count, dtype: int64

Rental candidate price period distribution:
price_period
unspecified    45
monthly         3
Name: count, dtype: int64

Rental candidate status distribution:
status
Available    48
Name: count, dtype: int64

Missing values in rental candidates:
listing

In [166]:

import json
from pathlib import Path
from collections import Counter

base_dir = Path("data/raw/real_estate/raw/afrotie")

files = [
    base_dir / "house_product-data.json",
    base_dir / "house_product-data_part-01.json",
]

# Fields we should not print because they may contain contact details.
sensitive_fields = {
    "phone", "telephone", "email", "contact",
    "contacts", "agent", "author", "password"
}

def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # Identify a list nested inside the top-level object.
        for key, value in data.items():
            if isinstance(value, list):
                print(f"Nested records found under key: {key}")
                return value

    return []


all_ids = {}

for path in files:
    print("\n" + "=" * 70)
    print("FILE:", path.name)
    print("=" * 70)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    records = load_records(path)

    valid_records = [
        r for r in records if isinstance(r, dict)
    ]

    invalid_count = len(records) - len(valid_records)

    print("Total records:", len(records))
    print("Valid records:", len(valid_records))
    print("Invalid records:", invalid_count)

    # Field frequency
    field_counts = Counter()

    for record in valid_records:
        field_counts.update(record.keys())

    print("\nFIELD FREQUENCY:")
    for field, count in field_counts.most_common():
        print(f"{field}: {count}/{len(valid_records)}")

    # Missing values
    print("\nMISSING VALUE COUNTS:")
    all_fields = sorted(field_counts.keys())

    for field in all_fields:
        missing = sum(
            record.get(field) is None
            or record.get(field) == ""
            for record in valid_records
        )
        print(f"{field}: {missing}/{len(valid_records)}")

    # Sample records
    print("\nSAMPLE RECORDS:")

    for i, record in enumerate(valid_records[:5], start=1):
        print(f"\nRECORD {i}")

        for key, value in record.items():
            if any(
                sensitive in key.lower()
                for sensitive in sensitive_fields
            ):
                continue

            if isinstance(value, (dict, list)):
                value = str(value)[:500]

            print(f"{key}: {value}")

    # Check likely identifier fields
    possible_ids = [
        "id", "ID", "product_id", "productId",
        "property_id", "PropertyID", "slug"
    ]

    for id_field in possible_ids:
        values = [
            str(r[id_field])
            for r in valid_records
            if r.get(id_field) not in (None, "")
        ]

        if values:
            duplicate_count = len(values) - len(set(values))

            print(
                f"\nIdentifier '{id_field}': "
                f"{len(values)} populated, "
                f"{duplicate_count} duplicate occurrences"
            )

            for value in values:
                all_ids.setdefault(id_field, {}).setdefault(
                    value, []
                ).append(path.name)

            break


# Cross-file overlap check
print("\n" + "=" * 70)
print("CROSS-FILE OVERLAP CHECK")
print("=" * 70)

for id_field, values in all_ids.items():
    overlapping = {
        identifier: filenames
        for identifier, filenames in values.items()
        if len(set(filenames)) > 1
    }

    print(f"\nIdentifier: {id_field}")
    print("Identifiers appearing in both files:", len(overlapping))

    for identifier, filenames in list(overlapping.items())[:10]:
        print(identifier, filenames)



FILE: house_product-data.json
Total records: 0
Valid records: 0
Invalid records: 0

FIELD FREQUENCY:

MISSING VALUE COUNTS:

SAMPLE RECORDS:

FILE: house_product-data_part-01.json
Total records: 0
Valid records: 0
Invalid records: 0

FIELD FREQUENCY:

MISSING VALUE COUNTS:

SAMPLE RECORDS:

CROSS-FILE OVERLAP CHECK


In [168]:

import json
from pathlib import Path

base_dir = Path("data/raw/real_estate/raw/afrotie")

files = [
    base_dir / "house_product-data.json",
    base_dir / "house_product-data_part-01.json",
]

def inspect_structure(obj, path="root", depth=0, max_depth=4):
    indent = "  " * depth

    if depth > max_depth:
        return

    if isinstance(obj, dict):
        print(f"{indent}{path}: dict ({len(obj)} keys)")

        for key, value in list(obj.items())[:15]:
            inspect_structure(
                value,
                path=f"{path}.{key}",
                depth=depth + 1,
                max_depth=max_depth
            )

    elif isinstance(obj, list):
        print(f"{indent}{path}: list ({len(obj)} items)")

        if obj:
            inspect_structure(
                obj[0],
                path=f"{path}[0]",
                depth=depth + 1,
                max_depth=max_depth
            )

    else:
        print(
            f"{indent}{path}: "
            f"{type(obj).__name__} = {str(obj)[:100]}"
        )


for file_path in files:
    print("\n" + "=" * 70)
    print("FILE:", file_path.name)
    print("=" * 70)

    if not file_path.exists():
        print("File not found")
        continue

    print("File size:", round(file_path.stat().st_size / (1024 * 1024), 2), "MB")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    inspect_structure(data)



FILE: house_product-data.json
File size: 53.27 MB
Top-level type: dict
root: dict (27534 keys)
  root.-NZ9xFMaiBekYeMf7p9a: dict (26 keys)
    root.-NZ9xFMaiBekYeMf7p9a.AREA: str = 72
    root.-NZ9xFMaiBekYeMf7p9a.BATHROOMS: str = 5
    root.-NZ9xFMaiBekYeMf7p9a.BEDROOMS: str = 5
    root.-NZ9xFMaiBekYeMf7p9a.DESCRIPTION: str = G+3 ብሎኬት ልስን የዉጭ ቀለም ስራ ላይ የደረሰ።
    root.-NZ9xFMaiBekYeMf7p9a.EIGHTPICTURE: str = EMPTY
    root.-NZ9xFMaiBekYeMf7p9a.FIFTHPICTURE: str = EMPTY
    root.-NZ9xFMaiBekYeMf7p9a.FIRSTPICTURE: str = https://firebasestorage.googleapis.com/v0/b/ewaywednesday.appspot.com/o/AlXM4sPByHc7bdJDuCxd5R9jq5z1
    root.-NZ9xFMaiBekYeMf7p9a.FOURTHPICTURE: str = EMPTY
    root.-NZ9xFMaiBekYeMf7p9a.HOUR: int = 1688105977714
    root.-NZ9xFMaiBekYeMf7p9a.KEYFORPROFILPICTURE: str = AlXM4sPByHc7bdJDuCxd5R9jq5z1
    root.-NZ9xFMaiBekYeMf7p9a.LIVE ON NOT: str = YES
    root.-NZ9xFMaiBekYeMf7p9a.LOCATION: str = ሰሚት 72
    root.-NZ9xFMaiBekYeMf7p9a.MEMBER SINCE: str = 12/17/2021
    roo

In [169]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

BASE = Path("data/raw/real_estate/raw/afrotie")

FILES = [
    BASE / "house_product-data.json",
    BASE / "house_product-data_part-01.json",
]

datasets = {}

for path in FILES:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise TypeError(f"{path.name} is not a dictionary")

    datasets[path.name] = data

    print(f"\n{'=' * 70}")
    print("FILE:", path.name)
    print("RECORDS:", len(data))

    # Count how frequently each field appears.
    field_counts = Counter(
        field
        for record in data.values()
        if isinstance(record, dict)
        for field in record
    )

    print("\nALL FIELD NAMES AND FREQUENCIES:")
    for field, count in sorted(field_counts.items()):
        print(f"{field}: {count}/{len(data)}")

    # Highlight fields that might describe price or listing status.
    print("\nPOTENTIAL PRICE / STATUS / CATEGORY FIELDS:")
    for field in sorted(field_counts):
        if any(word in field.lower() for word in [
            "price", "rent", "cost", "amount",
            "status", "type", "category", "sale",
            "currency", "period"
        ]):
            print(field, ":", field_counts[field])

    # Inspect distributions of important fields.
    for field in ["HD", "P", "LIVE ON NOT"]:
        values = Counter(
            str(record.get(field))
            for record in data.values()
            if isinstance(record, dict)
        )
        print(f"\n{field} DISTRIBUTION:")
        print(values)

# Check whether the files contain overlapping listing IDs.
ids_main = set(datasets[FILES[0].name])
ids_part = set(datasets[FILES[1].name])

print("\n" + "=" * 70)
print("CROSS-FILE DUPLICATE AUDIT")
print("=" * 70)
print("Main file IDs:", len(ids_main))
print("Part file IDs:", len(ids_part))
print("Overlapping IDs:", len(ids_main & ids_part))
print("Unique IDs across both:", len(ids_main | ids_part))


FILE: house_product-data.json
RECORDS: 27534

ALL FIELD NAMES AND FREQUENCIES:
AREA: 27534/27534
BATHROOMS: 27534/27534
BEDROOMS: 27534/27534
DESCRIPTION: 27534/27534
EIGHTPICTURE: 27534/27534
FIFTHPICTURE: 27534/27534
FIRSTPICTURE: 27534/27534
FOURTHPICTURE: 27534/27534
HD: 1413/27534
HOUR: 27534/27534
KEYFORPROFILPICTURE: 27534/27534
LIVE ON NOT: 27534/27534
LOCATION: 27534/27534
MEMBER SINCE: 27534/27534
P: 27532/27534
PHONE NUMBER: 27534/27534
POSTING CITY: 27534/27534
POSTING FIRST CATEGORY: 27534/27534
POSTING TIME: 27534/27534
PRICE: 27534/27534
SALEORRENT: 27534/27534
SECONDPICTURE: 27534/27534
SEVENTHPICTURE: 27534/27534
SIXTHPICTURE: 27534/27534
THE NAME: 27534/27534
THIRDPICTURE: 27534/27534
VIEW: 27534/27534

POTENTIAL PRICE / STATUS / CATEGORY FIELDS:
POSTING FIRST CATEGORY : 27534
PRICE : 27534
SALEORRENT : 27534

HD DISTRIBUTION:
Counter({'None': 26121, 'CON': 712, 'APA': 180, 'GPL': 177, 'GRO': 174, 'LAN': 104, 'OFF': 46, 'GUE': 14, 'BUI': 6})

P DISTRIBUTION:
Counter(

In [170]:
safe_fields = [
    "AREA",
    "BEDROOMS",
    "BATHROOMS",
    "DESCRIPTION",
    "LOCATION",
    "HOUR",
    "HD",
    "P",
    "LIVE ON NOT",
]

for filename, data in datasets.items():
    print(f"\n{'=' * 70}")
    print(filename)

    shown = 0

    for listing_id, record in data.items():
        if not isinstance(record, dict):
            continue

        print("\nLISTING ID:", listing_id)

        for field in safe_fields:
            if field in record:
                value = record[field]

                if field == "DESCRIPTION" and isinstance(value, str):
                    value = value[:500]

                print(f"{field}: {value}")

        shown += 1
        if shown == 5:
            break


house_product-data.json

LISTING ID: -NZ9xFMaiBekYeMf7p9a
AREA: 72
BEDROOMS: 5
BATHROOMS: 5
DESCRIPTION: G+3 ብሎኬት ልስን የዉጭ ቀለም ስራ ላይ የደረሰ።
LOCATION: ሰሚት 72
HOUR: 1688105977714
P: N
LIVE ON NOT: YES

LISTING ID: -NZ9xPQGg-1dJE6qmA7Y
AREA: 200
BEDROOMS: 6
BATHROOMS: 4
DESCRIPTION: 🏤👉👉አስቾካይ👈👈🏤
👉ጽድት ያለ ቤት ነው።
👉ብሩ በአስቾካይ ስለተፈለገ ነው።
እያንዳንዱ ክፍል 2 መኝታ፤ 1 ክችን፤ ሶፋ እና እቃ ማስቀመጫ ቦታዎች አሉ። 


ትንሽ  Commission አለበት
LOCATION: Akaki Kality
HOUR: 1688106019899
P: Y
LIVE ON NOT: YES

LISTING ID: -NZ9z2lnihGjxHZbBYIB
AREA: 250
BEDROOMS: 5
BATHROOMS: 4
DESCRIPTION: ⭐️ቪላ ቤቶች 🏠

በከተማች አዲስ አበባ  ያለውን በጥራት የተገነቡ ቪላ ቤቶች እጥረት የፈታው "እንይ ሪል እስቴት"  በተለያየ የካሬ አማራጮች ቪላ ቤቶችን አቅርቦሎታል!!!

➲120m² , 127m² , 250m²  +service room

➲30% Down payment only

 Price D.P - 3,300,000 ,  3,960,000 -  4,950,000

⭐️⭐️Call and Take your home!!!!
👉0922009504
#realestate  #villahomes  #ቪላቤት #enyirealestate
LOCATION: Jemo 2
HOUR: 1688106451664
P: Y
LIVE ON NOT: YES

LISTING ID: -NZ9z6B9bo8Yl0StcRMo
AREA: 110
BEDROOMS: 3
BATHROOMS: 1
DESCRIP

In [171]:
from collections import Counter
import pandas as pd

for filename, data in datasets.items():

    print("\n" + "=" * 75)
    print("FILE:", filename)
    print("=" * 75)

    df = pd.DataFrame.from_dict(data, orient="index")
    df.index.name = "listing_id"

    for field in ["SALEORRENT", "POSTING FIRST CATEGORY", "HD",
                  "POSTING CITY", "LIVE ON NOT", "P"]:

        if field not in df.columns:
            continue

        print(f"\n{field} DISTRIBUTION:")
        print(df[field].fillna("<MISSING>").astype(str).value_counts().head(40))

    print("\nPRICE SAMPLES:")
    print(df["PRICE"].dropna().astype(str).value_counts().head(30))

    print("\nSELECTED RECORDS:")
    columns = [
        "PRICE",
        "SALEORRENT",
        "POSTING FIRST CATEGORY",
        "HD",
        "LOCATION",
        "POSTING CITY",
        "DESCRIPTION",
    ]

    available = [c for c in columns if c in df.columns]

    print(df[available].head(12).to_string(max_colwidth=180))


FILE: house_product-data.json

SALEORRENT DISTRIBUTION:
SALEORRENT
SALE    25741
RENT     1793
Name: count, dtype: int64

POSTING FIRST CATEGORY DISTRIBUTION:
POSTING FIRST CATEGORY
HOUSE    27534
Name: count, dtype: int64

HD DISTRIBUTION:
HD
<MISSING>    26121
CON            712
APA            180
GPL            177
GRO            174
LAN            104
OFF             46
GUE             14
BUI              6
Name: count, dtype: int64

POSTING CITY DISTRIBUTION:
POSTING CITY
ADDIS ABABA    27534
Name: count, dtype: int64

LIVE ON NOT DISTRIBUTION:
LIVE ON NOT
YES    27529
NO         5
Name: count, dtype: int64

P DISTRIBUTION:
P
Y            25102
N             2430
<MISSING>        2
Name: count, dtype: int64

PRICE SAMPLES:
PRICE
6500000     502
5500000     408
8500000     407
11000000    404
3500000     397
14000000    392
15000000    350
13000000    349
7500000     349
9000000     343
5600000     332
6000000     329
12000000    319
8000000     315
5000000     314
7000000     305

In [172]:
import pandas as pd
from pathlib import Path

# Combine both files
rent_frames = []

for filename, data in datasets.items():
    df = pd.DataFrame.from_dict(data, orient="index")
    df.index.name = "listing_id"
    df = df.reset_index()
    df["source_file"] = filename

    # Keep only records explicitly marked as rentals
    df["SALEORRENT"] = df["SALEORRENT"].astype(str).str.strip().str.upper()
    df = df[df["SALEORRENT"] == "RENT"].copy()

    rent_frames.append(df)

afrotie_rent = pd.concat(rent_frames, ignore_index=True)

print("=" * 70)
print("AFROTIE RENTAL CANDIDATE AUDIT")
print("=" * 70)

print("Total rental candidates:", len(afrotie_rent))
print("Unique listing IDs:", afrotie_rent["listing_id"].nunique())
print("Duplicate listing IDs:", afrotie_rent["listing_id"].duplicated().sum())

# Price data types and missing values
print("\nPRICE DATA TYPES:")
print(afrotie_rent["PRICE"].map(type).value_counts())

print("\nMISSING PRICE VALUES:")
print(afrotie_rent["PRICE"].isna().sum())

print("\nPRICE VALUE SAMPLES:")
print(afrotie_rent["PRICE"].astype(str).value_counts().head(40))

# Convert prices for inspection only
afrotie_rent["price_numeric"] = pd.to_numeric(
    afrotie_rent["PRICE"], errors="coerce"
)

print("\nNUMERIC PRICE SUMMARY:")
print(afrotie_rent["price_numeric"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

print("\nNON-NUMERIC PRICE EXAMPLES:")
print(
    afrotie_rent.loc[
        afrotie_rent["price_numeric"].isna(),
        ["PRICE", "LOCATION", "DESCRIPTION"]
    ].head(10).to_string(index=False, max_colwidth=150)
)

# Location distribution
print("\nTOP 40 LOCATIONS:")
print(
    afrotie_rent["LOCATION"]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts()
    .head(40)
)

# HD distribution among rentals
print("\nRENTAL HD DISTRIBUTION:")
print(
    afrotie_rent["HD"]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts()
)

# Inspect examples without personal contact information
safe_columns = [
    "listing_id",
    "PRICE",
    "AREA",
    "BEDROOMS",
    "BATHROOMS",
    "LOCATION",
    "HD",
    "DESCRIPTION",
]

print("\nRENTAL RECORD SAMPLES:")
print(
    afrotie_rent[safe_columns]
    .head(20)
    .to_string(index=False, max_colwidth=200)
)

AFROTIE RENTAL CANDIDATE AUDIT
Total rental candidates: 2419
Unique listing IDs: 2419
Duplicate listing IDs: 0

PRICE DATA TYPES:
PRICE
<class 'str'>    2419
Name: count, dtype: int64

MISSING PRICE VALUES:
0

PRICE VALUE SAMPLES:
PRICE
25000     144
30000     143
35000     139
40000     137
60000     137
50000     120
20000      93
15000      89
45000      81
70000      74
80000      66
55000      55
18000      55
100000     53
22000      52
150000     50
16000      44
90000      43
13000      37
14000      35
65000      35
200000     33
23000      33
17000      29
27000      27
28000      25
130000     23
110000     22
120000     22
75000      22
160000     21
32000      21
140000     21
12000      19
400000     19
10000      19
26000      19
170000     17
8000       16
6000       13
Name: count, dtype: int64

NUMERIC PRICE SUMMARY:
count    2.419000e+03
mean     8.503299e+04
std      5.578915e+05
min      3.000000e+02
1%       6.000000e+03
5%       1.200000e+04
25%      2.300000e+04

In [173]:
import re
import numpy as np
import pandas as pd

audit = afrotie_rent.copy()

# Normalize price
audit["price_numeric"] = pd.to_numeric(
    audit["PRICE"], errors="coerce"
)

# Normalize description for text inspection
audit["description_text"] = (
    audit["DESCRIPTION"]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Currency indicators
audit["mentions_usd"] = audit["description_text"].str.contains(
    r"\busd\b|\b\d+\s*\$|\$\s*\d+|dollar",
    regex=True
)

audit["mentions_etb"] = audit["description_text"].str.contains(
    r"\betb\b|\bbirr\b|\bbr\b|ብር",
    regex=True
)

# Rental period indicators
audit["mentions_monthly"] = audit["description_text"].str.contains(
    r"per month|/month|monthly|a month|በወር",
    regex=True
)

audit["mentions_daily"] = audit["description_text"].str.contains(
    r"per day|/day|daily|a day|በቀን",
    regex=True
)

audit["mentions_weekly"] = audit["description_text"].str.contains(
    r"per week|/week|weekly",
    regex=True
)

audit["mentions_yearly"] = audit["description_text"].str.contains(
    r"per year|/year|yearly|annually",
    regex=True
)

# Property scope indicators
audit["commercial_terms"] = audit["description_text"].str.contains(
    r"warehouse|office|shop|store|factory|industrial|"
    r"commercial|restaurant|grocery|business",
    regex=True
)

audit["mixed_rent_sale"] = audit["description_text"].str.contains(
    r"for rent or sale|rent or sale|for sale and rent|"
    r"rent.*sale|sale.*rent",
    regex=True
)

# Inspect distributions
flags = [
    "mentions_usd",
    "mentions_etb",
    "mentions_monthly",
    "mentions_daily",
    "mentions_weekly",
    "mentions_yearly",
    "commercial_terms",
    "mixed_rent_sale",
]

print("=" * 70)
print("DESCRIPTION AUDIT")
print("=" * 70)

for col in flags:
    print(f"{col}: {audit[col].sum()} / {len(audit)}")

# Combined currency and period categories
print("\nCURRENCY INDICATOR COMBINATIONS:")
print(
    audit.groupby(["mentions_usd", "mentions_etb"])
    .size()
    .rename("count")
)

print("\nRENTAL PERIOD COMBINATIONS:")
print(
    audit.groupby([
        "mentions_monthly",
        "mentions_daily",
        "mentions_weekly",
        "mentions_yearly",
    ]).size().sort_values(ascending=False).head(20)
)

# Price and measurement anomalies
print("\nPRICE DISTRIBUTION BY RANGE:")
print(pd.cut(
    audit["price_numeric"],
    bins=[0, 5000, 10000, 50000, 100000, 500000, 1000000, float("inf")]
).value_counts().sort_index())

for col in ["AREA", "BEDROOMS", "BATHROOMS"]:
    audit[col] = pd.to_numeric(audit[col], errors="coerce")
    print(f"\n{col} SUMMARY:")
    print(audit[col].describe())

# Inspect selected records, excluding contact information
inspect_columns = [
    "listing_id",
    "PRICE",
    "AREA",
    "BEDROOMS",
    "BATHROOMS",
    "LOCATION",
    "HD",
    "DESCRIPTION",
]

print("\nCOMMERCIAL RENTAL CANDIDATES:")
print(
    audit.loc[audit["commercial_terms"], inspect_columns]
    .head(15)
    .to_string(index=False, max_colwidth=180)
)

print("\nRECORDS WITH USD MENTIONS:")
print(
    audit.loc[audit["mentions_usd"], inspect_columns]
    .head(15)
    .to_string(index=False, max_colwidth=180)
)

print("\nRECORDS WITH NO EXPLICIT RENTAL PERIOD:")
print(
    audit.loc[
        ~audit[[
            "mentions_monthly",
            "mentions_daily",
            "mentions_weekly",
            "mentions_yearly",
        ]].any(axis=1),
        inspect_columns
    ].head(15).to_string(index=False, max_colwidth=180)
)

print("\nEXTREME PRICE RECORDS:")
print(
    audit.nlargest(15, "price_numeric")[inspect_columns]
    .to_string(index=False, max_colwidth=180)
)

DESCRIPTION AUDIT
mentions_usd: 44 / 2419
mentions_etb: 357 / 2419
mentions_monthly: 59 / 2419
mentions_daily: 4 / 2419
mentions_weekly: 0 / 2419
mentions_yearly: 0 / 2419
commercial_terms: 115 / 2419
mixed_rent_sale: 4 / 2419

CURRENCY INDICATOR COMBINATIONS:
mentions_usd  mentions_etb
False         False           2026
              True             349
True          False             36
              True               8
Name: count, dtype: int64

RENTAL PERIOD COMBINATIONS:
mentions_monthly  mentions_daily  mentions_weekly  mentions_yearly
False             False           False            False              2358
True              False           False            False                57
False             True            False            False                 2
True              True            False            False                 2
dtype: int64

PRICE DISTRIBUTION BY RANGE:
price_numeric
(0.0, 5000.0]              18
(5000.0, 10000.0]          84
(10000.0, 50000.0]       1443
(50

In [175]:
import pandas as pd
import numpy as np

df = afrotie_rent.copy()

# Convert numeric fields safely
for col in ["PRICE", "AREA", "BEDROOMS", "BATHROOMS"]:
    df[col + "_numeric"] = pd.to_numeric(
        df[col], errors="coerce"
    )

print("=" * 70)
print("FEATURE QUALITY AUDIT")
print("=" * 70)

for col in ["PRICE", "AREA", "BEDROOMS", "BATHROOMS"]:
    s = df[col + "_numeric"]

    print(f"\n{col}")
    print("Missing:", s.isna().sum())
    print("Zero:", (s == 0).sum())
    print("Negative:", (s < 0).sum())
    print("Unique values:", s.nunique())
    print("Largest 15 values:")
    print(s.nlargest(15).to_list())

print("\n" + "=" * 70)
print("BEDROOM OUTLIERS")
print("=" * 70)

bedroom_outliers = df[
    (df["BEDROOMS_numeric"] > 15) |
    (df["BEDROOMS_numeric"] < 0)
]

print(
    bedroom_outliers[
        ["listing_id", "PRICE", "AREA", "BEDROOMS",
         "BATHROOMS", "LOCATION", "HD"]
    ].head(30).to_string(index=False)
)

print("\n" + "=" * 70)
print("AREA OUTLIERS")
print("=" * 70)

area_outliers = df[df["AREA_numeric"] > 2000]

print(
    area_outliers[
        ["listing_id", "PRICE", "AREA", "BEDROOMS",
         "BATHROOMS", "LOCATION", "HD"]
    ].head(30).to_string(index=False)
)

print("\n" + "=" * 70)
print("HD CATEGORY COUNTS")
print("=" * 70)

print(df["HD"].fillna("<MISSING>").value_counts().to_string())

FEATURE QUALITY AUDIT

PRICE
Missing: 0
Zero: 0
Negative: 0
Unique values: 130
Largest 15 values:
[26000000, 4800000, 4800000, 2000000, 2000000, 1600000, 1100000, 1100000, 1100000, 1100000, 1100000, 1000000, 1000000, 1000000, 800000]

AREA
Missing: 0
Zero: 14
Negative: 0
Unique values: 191
Largest 15 values:
[12000, 12000, 10000, 10000, 10000, 5000, 5000, 3600, 3600, 3000, 3000, 3000, 2500, 2500, 2300]

BEDROOMS
Missing: 0
Zero: 198
Negative: 0
Unique values: 29
Largest 15 values:
[920682278, 50, 50, 40, 40, 35, 35, 33, 33, 30, 30, 27, 27, 27, 25]

BATHROOMS
Missing: 0
Zero: 120
Negative: 0
Unique values: 19
Largest 15 values:
[45, 35, 35, 27, 27, 27, 27, 22, 22, 20, 18, 18, 13, 13, 13]

BEDROOM OUTLIERS
          listing_id   PRICE AREA   BEDROOMS BATHROOMS                    LOCATION       HD
-NZAeKrSoygZRx6nMOpN   40000  160         20         1       shiromeda addis abeba      NaN
-NYJNfzwwLYyc2B90ITJ 1600000 1000         40        45                     bole ቦሌ      NaN
-NXo93v_la

In [176]:
# 1. Inspect records with suspicious bedroom counts
bedroom_suspicious = df[
    df["BEDROOMS_numeric"] > 15
].copy()

print("=" * 70)
print("SUSPICIOUS BEDROOM RECORDS")
print("=" * 70)

for _, row in bedroom_suspicious.iterrows():
    print("\nListing ID:", row["listing_id"])
    print("Price:", row["PRICE"])
    print("Area:", row["AREA"])
    print("Bedrooms:", row["BEDROOMS"])
    print("Bathrooms:", row["BATHROOMS"])
    print("Location:", row["LOCATION"])
    print("Description:", str(row["DESCRIPTION"])[:500])


# 2. Inspect records with unusually high bathroom counts
bathroom_suspicious = df[
    df["BATHROOMS_numeric"] > 10
].copy()

print("\n" + "=" * 70)
print("SUSPICIOUS BATHROOM RECORDS")
print("=" * 70)

print(
    bathroom_suspicious[
        ["listing_id", "PRICE", "AREA", "BEDROOMS",
         "BATHROOMS", "LOCATION", "HD"]
    ].to_string(index=False)
)


# 3. Find repeated descriptions
# Normalize text so differences in capitalization and whitespace
# do not prevent matching.
df["description_normalized"] = (
    df["DESCRIPTION"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

repeated_descriptions = df[
    df["description_normalized"].duplicated(keep=False)
].sort_values("description_normalized")

print("\n" + "=" * 70)
print("REPEATED DESCRIPTION GROUPS")
print("=" * 70)

print(
    repeated_descriptions[
        ["listing_id", "PRICE", "AREA", "BEDROOMS",
         "BATHROOMS", "LOCATION", "HD"]
    ].head(60).to_string(index=False)
)

print(
    "\nRecords with repeated descriptions:",
    len(repeated_descriptions)
)

print(
    "Distinct repeated descriptions:",
    repeated_descriptions["description_normalized"].nunique()
)


# 4. Check whether repeated descriptions have conflicting values
print("\n" + "=" * 70)
print("REPEATED DESCRIPTION CONSISTENCY")
print("=" * 70)

if not repeated_descriptions.empty:
    consistency = (
        repeated_descriptions
        .groupby("description_normalized")
        .agg(
            record_count=("listing_id", "size"),
            distinct_prices=("PRICE", "nunique"),
            distinct_areas=("AREA", "nunique"),
            distinct_bedrooms=("BEDROOMS", "nunique"),
            distinct_locations=("LOCATION", "nunique")
        )
    )

    print(consistency.head(30).to_string())

SUSPICIOUS BEDROOM RECORDS

Listing ID: -NZAeKrSoygZRx6nMOpN
Price: 40000
Area: 160
Bedrooms: 20
Bathrooms: 1
Location: shiromeda addis abeba
Description: location shiromeda
2 bed rooms 
2 services 
1 car parking
40.000.fixed 
call 0930537331
       0910158948

Listing ID: -NYJNfzwwLYyc2B90ITJ
Price: 1600000
Area: 1000
Bedrooms: 40
Bathrooms: 45
Location: bole ቦሌ
Description: # G+ 8 + basement and terrace modern and standard bulinding. 
# 1000m2
# for guest house service and hotel service 
# 5th floor 500 m2 hail room
#  40 furnished modern rooms including bath room & cupboard
# ground floor 500 m2 (including Mezzanine) for cafe and restaurant and lounge Burger and pizza shop 
# 600 care basement for night club or for store .
# የመሬት ወለል ላይ 500 ካሬ +  300 ሜዛኒን ጨምሮ ለሆቴል ስራ እና ለካፌ ሬስቶራንት እና ላዉንጅ የሚሆን 
# 600 ካሬ ከመሬት ስር ለናይት ክለብና ለፈለጉት ስራ ቦታ የሚሆን ።
 5ኛ ወለል ላይ 500 ካሬ ትልቅ አዳ

Listing ID: -NXo93v_laVMebXwPgsh
Price: 750000
Area: 3000
Bedrooms: 40
Bathrooms: 3
Location: እሥቴዲየም አካባቢ
Description: 

In [177]:

import pandas as pd
import numpy as np
import re

def extract_afrotie_candidates(source_df):
    df = source_df.copy()

    # Standardize identifiers and source fields
    df["listing_id"] = df["listing_id"].astype(str).str.strip()

    for col in ["PRICE", "AREA", "BEDROOMS", "BATHROOMS"]:
        df[f"raw_{col.lower()}"] = df[col]
        df[col.lower()] = pd.to_numeric(df[col], errors="coerce")

    df["description_text"] = (
        df["DESCRIPTION"].fillna("").astype(str).str.lower()
    )

    description = df["description_text"]

    # Currency evidence
    df["mentions_usd"] = description.str.contains(
        r"\busd\b|\$\s*\d|\d\s*\$|dollars?",
        regex=True
    )

    df["mentions_etb"] = description.str.contains(
        r"\betb\b|\bbirr\b|\bbr\b|ብር",
        regex=True
    )

    df["currency_conflict"] = (
        df["mentions_usd"] & df["mentions_etb"]
    )

    df["currency_evidence"] = np.select(
        [
            df["currency_conflict"],
            df["mentions_etb"],
            df["mentions_usd"]
        ],
        [
            "conflicting",
            "ETB mentioned",
            "USD mentioned"
        ],
        default="unspecified"
    )

    # Rental-period evidence
    df["mentions_monthly"] = description.str.contains(
        r"per month|/month|monthly|a month|በወር",
        regex=True
    )

    df["mentions_daily"] = description.str.contains(
        r"per day|/day|daily|a day|በቀን",
        regex=True
    )

    df["mentions_weekly"] = description.str.contains(
        r"per week|/week|weekly",
        regex=True
    )

    df["mentions_yearly"] = description.str.contains(
        r"per year|/year|yearly|annually",
        regex=True
    )

    df["period_evidence"] = np.select(
        [
            df["mentions_monthly"],
            df["mentions_daily"],
            df["mentions_weekly"],
            df["mentions_yearly"]
        ],
        [
            "monthly",
            "daily",
            "weekly",
            "yearly"
        ],
        default="unspecified"
    )

    # Property-use indicators
    df["commercial_terms"] = description.str.contains(
        r"warehouse|office|shop|store|factory|industrial|"
        r"commercial|restaurant|grocery|business|college|school",
        regex=True
    )

    df["hospitality_terms"] = description.str.contains(
        r"hotel|guest.?house|pension|hostel|night.?club",
        regex=True
    )

    df["mixed_use_terms"] = description.str.contains(
        r"for office|for living|for guest|for hotel|"
        r"office.*living|living.*office",
        regex=True
    )

    # Basic feature-quality flags
    df["invalid_price"] = (
        df["price"].isna() | (df["price"] <= 0)
    )

    df["invalid_area"] = (
        df["area"].isna() |
        (df["area"] <= 0) |
        (df["area"] > 1000)
    )

    # Preserve raw bedroom/bathroom values.
    # Values above 10 are flagged for review, not silently corrected.
    df["bedroom_outlier"] = (
        df["bedrooms"].isna() |
        (df["bedrooms"] < 0) |
        (df["bedrooms"] > 10)
    )

    df["bathroom_outlier"] = (
        df["bathrooms"].isna() |
        (df["bathrooms"] < 0) |
        (df["bathrooms"] > 10)
    )

    # Potential duplicate descriptions.
    # This flags repeated descriptions; it does not delete records.
    normalized_description = (
        description
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df["description_duplicate_count"] = (
        normalized_description
        .map(normalized_description.value_counts())
    )

    # Conservative target extraction
    valid_monthly_etb = (
        ~df["invalid_price"] &
        df["mentions_etb"] &
        ~df["mentions_usd"] &
        df["mentions_monthly"] &
        ~df["mentions_daily"] &
        ~df["mentions_weekly"] &
        ~df["mentions_yearly"]
    )

    df["monthly_rent_etb"] = np.where(
        valid_monthly_etb,
        df["price"],
        np.nan
    )

    # Overall review status
    df["review_required"] = (
        ~valid_monthly_etb |
        df["invalid_area"] |
        df["bedroom_outlier"] |
        df["bathroom_outlier"] |
        df["commercial_terms"] |
        df["hospitality_terms"] |
        df["mixed_use_terms"] |
        (df["description_duplicate_count"] > 1)
    )

    # Keep a standardized, non-contact dataset.
    # Do not include phone numbers, profile data, or raw descriptions.
    output_columns = [
        "listing_id",
        "source_file",
        "location",
        "hd",
        "price",
        "area",
        "bedrooms",
        "bathrooms",
        "raw_price",
        "raw_area",
        "raw_bedrooms",
        "raw_bathrooms",
        "currency_evidence",
        "period_evidence",
        "mentions_usd",
        "mentions_etb",
        "currency_conflict",
        "commercial_terms",
        "hospitality_terms",
        "mixed_use_terms",
        "invalid_price",
        "invalid_area",
        "bedroom_outlier",
        "bathroom_outlier",
        "description_duplicate_count",
        "monthly_rent_etb",
        "review_required"
    ]

    df.columns = [
        col.lower() if col in ["LOCATION", "HD"] else col
        for col in df.columns
    ]

    return df[output_columns].copy()


# Create the Afrotie extraction dataset
afrotie_candidates = extract_afrotie_candidates(afrotie_rent)

print("Afrotie extraction completed.")
print("Total records:", len(afrotie_candidates))
print("Unique listing IDs:", afrotie_candidates["listing_id"].nunique())
print(
    "Monthly ETB candidates:",
    afrotie_candidates["monthly_rent_etb"].notna().sum()
)
print(
    "Records requiring review:",
    afrotie_candidates["review_required"].sum()
)


Afrotie extraction completed.
Total records: 2419
Unique listing IDs: 2419
Monthly ETB candidates: 28
Records requiring review: 2418


In [178]:

print("=" * 65)
print("AFROTIE EXTRACTION SUMMARY")
print("=" * 65)

print("\nCurrency evidence:")
print(afrotie_candidates["currency_evidence"].value_counts(dropna=False))

print("\nRental period evidence:")
print(afrotie_candidates["period_evidence"].value_counts(dropna=False))

print("\nMonthly ETB target:")
print("Available:", afrotie_candidates["monthly_rent_etb"].notna().sum())
print("Missing:", afrotie_candidates["monthly_rent_etb"].isna().sum())

print("\nReview flags:")
for col in [
    "currency_conflict",
    "commercial_terms",
    "hospitality_terms",
    "mixed_use_terms",
    "invalid_area",
    "bedroom_outlier",
    "bathroom_outlier"
]:
    print(f"{col}: {afrotie_candidates[col].sum()}")

print("\nMonthly ETB candidate price summary:")
print(
    afrotie_candidates["monthly_rent_etb"]
    .dropna()
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)


AFROTIE EXTRACTION SUMMARY

Currency evidence:
currency_evidence
unspecified      2026
ETB mentioned     349
USD mentioned      36
conflicting         8
Name: count, dtype: int64

Rental period evidence:
period_evidence
unspecified    2358
monthly          59
daily             2
Name: count, dtype: int64

Monthly ETB target:
Available: 28
Missing: 2391

Review flags:
currency_conflict: 8
commercial_terms: 131
hospitality_terms: 89
mixed_use_terms: 22
invalid_area: 69
bedroom_outlier: 64
bathroom_outlier: 18

Monthly ETB candidate price summary:
count        28.000000
mean     108928.571429
std      149580.383090
min       10000.000000
1%        10000.000000
5%        17700.000000
25%       40000.000000
50%       61000.000000
75%       95000.000000
95%      477500.000000
99%      600000.000000
max      600000.000000
Name: monthly_rent_etb, dtype: float64


In [179]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Complete extracted rental dataset
afrotie_candidates.to_csv(
    processed_dir / "afrotie_all_extracted.csv",
    index=False
)

# Records with a monthly ETB target
afrotie_candidates[
    afrotie_candidates["monthly_rent_etb"].notna()
].to_csv(
    processed_dir / "afrotie_monthly_etb_candidates.csv",
    index=False
)

# Records flagged for review
afrotie_candidates[
    afrotie_candidates["review_required"]
].to_csv(
    processed_dir / "afrotie_review.csv",
    index=False
)

print("Afrotie files saved successfully.")

Afrotie files saved successfully.


In [180]:
import json
from pathlib import Path
from collections import Counter

beten_path = Path(
    "data/raw/real_estate/raw/beten/beten_data.json"
)

with open(beten_path, "r", encoding="utf-8") as f:
    beten_data = json.load(f)

print("=" * 65)
print("BETEN SCHEMA INSPECTION")
print("=" * 65)

print("\nTop-level type:", type(beten_data).__name__)

if isinstance(beten_data, list):
    print("Total records:", len(beten_data))

    valid_records = [
        record for record in beten_data
        if isinstance(record, dict)
    ]

    print("Valid records:", len(valid_records))
    print("Invalid records:", len(beten_data) - len(valid_records))

elif isinstance(beten_data, dict):
    print("Total records:", len(beten_data))

    valid_records = [
        record for record in beten_data.values()
        if isinstance(record, dict)
    ]

    print("Valid records:", len(valid_records))
    print("Invalid records:", len(beten_data) - len(valid_records))

else:
    valid_records = []

if valid_records:
    field_counts = Counter(
        key
        for record in valid_records
        for key in record.keys()
    )

    print("\nFIELD FREQUENCY")
    for field, count in field_counts.most_common():
        print(f"{field}: {count}/{len(valid_records)}")

    print("\nSAMPLE RECORD KEYS")
    print(list(valid_records[0].keys()))

    print("\nSAMPLE VALUES (excluding contact and personal data)")
    sensitive_terms = [
        "phone", "mobile", "email", "contact",
        "password", "author", "member", "profile"
    ]

    for key, value in valid_records[0].items():
        if any(term in key.lower() for term in sensitive_terms):
            continue

        if isinstance(value, dict):
            print(f"{key}: nested keys = {list(value.keys())[:20]}")
        elif isinstance(value, list):
            print(f"{key}: list length = {len(value)}")
        else:
            print(f"{key}: {str(value)[:250]}")

BETEN SCHEMA INSPECTION

Top-level type: list
Total records: 1213
Valid records: 1213
Invalid records: 0

FIELD FREQUENCY
id: 1213/1213
title: 1213/1213
description: 1213/1213
latitude: 1213/1213
longitude: 1213/1213
location_name: 1213/1213
number_of_bed_rooms: 1213/1213
number_of_baths: 1213/1213
number_of_saloons: 1213/1213
number_of_living_rooms: 1213/1213
total_number_of_rooms: 1213/1213
area: 1213/1213
price: 1213/1213
old_price: 1213/1213
car_parking: 1213/1213
fire_place: 1213/1213
air_conditioning: 1213/1213
free_wifi: 1213/1213
spa: 1213/1213
pets: 1213/1213
owners_living_there: 1213/1213
swimming_pool: 1213/1213
garden: 1213/1213
property_images: 1213/1213
property_video: 1213/1213
floor_plan: 1213/1213
city: 1213/1213
subcity: 1213/1213
wereda: 1213/1213
kebele: 1213/1213
village_name: 1213/1213
owner_id: 1213/1213
order_number: 1213/1213
owner_phone_number: 1213/1213
owner_name: 1213/1213
property_category_id: 1213/1213
property_type_id: 1213/1213
status: 1213/1213
created

In [181]:
import pandas as pd
import numpy as np

beten_df = pd.DataFrame(beten_data)

# Extract nested category and type names
beten_df["category_name"] = beten_df[
    "property_category_id"
].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)

beten_df["property_type_name"] = beten_df[
    "property_type_id"
].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)

print("=" * 65)
print("BETEN VALUE AUDIT")
print("=" * 65)

for col in [
    "city",
    "subcity",
    "location_name",
    "category_name",
    "property_type_name",
    "status"
]:
    print(f"\n{col.upper()} DISTRIBUTION")
    print(beten_df[col].value_counts(dropna=False).head(30))

print("\nPRICE SUMMARY")
print(
    pd.to_numeric(beten_df["price"], errors="coerce")
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

print("\nPRICE DATA TYPES")
print(beten_df["price"].map(type).value_counts())

print("\nMISSING VALUES")
for col in [
    "price",
    "area",
    "number_of_bed_rooms",
    "number_of_baths",
    "city",
    "location_name"
]:
    print(f"{col}: {beten_df[col].isna().sum()}")

print("\nSELECTED LISTING SAMPLES")

sample_columns = [
    "id",
    "description",
    "price",
    "area",
    "number_of_bed_rooms",
    "number_of_baths",
    "location_name",
    "city",
    "category_name",
    "property_type_name",
    "status"
]

for _, row in beten_df[sample_columns].head(8).iterrows():
    print("\n" + "-" * 60)
    for col in sample_columns:
        value = row[col]
        if col == "description" and isinstance(value, str):
            value = value[:500]
        print(f"{col}: {value}")

BETEN VALUE AUDIT

CITY DISTRIBUTION
city
Addis Ababa    1169
Sheger City      20
Sheger city       6
Adama             6
Hawassa           2
addis ababa       2
ለሚ ኩራ             2
sheger city       1
Addis ababa       1
አዲስ አበባ           1
Addis abeba       1
addis abeba       1
Addisababa        1
Name: count, dtype: int64

SUBCITY DISTRIBUTION
subcity
NaN                 1149
Lemikura              13
addis ababa           10
Lami Kura              7
ለሚኩራ                   6
Nefas Silk-Lafto       4
lemikura               4
Addis Ababa            4
bole                   3
Yeka                   3
BOLE                   2
Lemi Kura              2
Bole                   1
yeka                   1
ኮልፌ ቀራንዮ               1
kolefe keranio         1
Gelan                  1
addisababa             1
Name: count, dtype: int64

LOCATION_NAME DISTRIBUTION
location_name
Ayat             165
ayat              64
semit             55
Semit             27
አያት               22
figa              1

In [182]:

import re
import pandas as pd
import numpy as np
from pathlib import Path

def extract_beten_candidates(source_df):

    df = source_df.copy()

    # Extract nested category and property type names
    df["category_name"] = df["property_category_id"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else None
    )

    df["property_type_name"] = df["property_type_id"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else None
    )

    # Preserve raw values before conversion
    for col in [
        "price",
        "area",
        "number_of_bed_rooms",
        "number_of_baths"
    ]:
        df[f"raw_{col}"] = df[col]

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["area"] = pd.to_numeric(df["area"], errors="coerce")
    df["bedrooms"] = pd.to_numeric(
        df["number_of_bed_rooms"], errors="coerce"
    )
    df["bathrooms"] = pd.to_numeric(
        df["number_of_baths"], errors="coerce"
    )

    # Normalize city labels
    city = df["city"].fillna("").astype(str).str.strip().str.lower()

    addis_variants = {
        "addis ababa",
        "addis abeba",
        "addisababa",
        "አዲስ አበባ"
    }

    df["city_scope"] = np.select(
        [
            city.isin(addis_variants),
            city.str.replace(" ", "", regex=False).eq("shegercity")
        ],
        [
            "addis_ababa",
            "sheger_city"
        ],
        default="other"
    )

    # Normalize transaction type
    df["transaction_type"] = (
        df["property_type_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Keep only records explicitly marked as rentals
    df = df[df["transaction_type"] == "rent"].copy()

    # Description evidence
    description = (
        df["description"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    # Currency evidence
    usd_pattern = (
        r"\busd\b|"
        r"\$\s*\d|"
        r"\d\s*\$|"
        r"\bdollars?\b"
    )

    etb_pattern = (
        r"\betb\b|"
        r"\bbirr\b|"
        r"\bbr\b|"
        r"ብር"
    )

    df["mentions_usd"] = description.str.contains(
        usd_pattern, regex=True, na=False
    )

    df["mentions_etb"] = description.str.contains(
        etb_pattern, regex=True, na=False
    )

    df["currency_conflict"] = (
        df["mentions_usd"] & df["mentions_etb"]
    )

    df["currency_evidence"] = np.select(
        [
            df["currency_conflict"],
            df["mentions_usd"],
            df["mentions_etb"]
        ],
        [
            "conflicting",
            "USD mentioned",
            "ETB mentioned"
        ],
        default="unspecified"
    )

    # Rental-period evidence
    monthly_pattern = (
        r"\bper month\b|"
        r"/month\b|"
        r"/mo\b|"
        r"\bmonthly\b|"
        r"\ba month\b|"
        r"በወር|ወርሃዊ"
    )

    daily_pattern = (
        r"\bper day\b|"
        r"/day\b|"
        r"\bdaily\b|"
        r"\ba day\b|"
        r"በቀን"
    )

    weekly_pattern = (
        r"\bper week\b|"
        r"/week\b|"
        r"\bweekly\b"
    )

    yearly_pattern = (
        r"\bper year\b|"
        r"/year\b|"
        r"\bannually\b|"
        r"\byearly\b"
    )

    df["mentions_monthly"] = description.str.contains(
        monthly_pattern, regex=True, na=False
    )

    df["mentions_daily"] = description.str.contains(
        daily_pattern, regex=True, na=False
    )

    df["mentions_weekly"] = description.str.contains(
        weekly_pattern, regex=True, na=False
    )

    df["mentions_yearly"] = description.str.contains(
        yearly_pattern, regex=True, na=False
    )

    period_count = (
        df["mentions_monthly"].astype(int)
        + df["mentions_daily"].astype(int)
        + df["mentions_weekly"].astype(int)
        + df["mentions_yearly"].astype(int)
    )

    df["period_evidence"] = np.select(
        [
            period_count > 1,
            df["mentions_monthly"],
            df["mentions_daily"],
            df["mentions_weekly"],
            df["mentions_yearly"]
        ],
        [
            "multiple periods",
            "monthly",
            "daily",
            "weekly",
            "yearly"
        ],
        default="unspecified"
    )

    # Property category scope
    residential_categories = {
        "villa houses",
        "apartment",
        "residence building",
        "condominium",
        "service house"
    }

    df["residential_category"] = (
        df["category_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(residential_categories)
    )

    # Data quality checks
    df["invalid_price"] = (
        df["price"].isna() | (df["price"] <= 0)
    )

    df["invalid_area"] = (
        df["area"].isna() | (df["area"] <= 0)
    )

    df["bedroom_outlier"] = (
        df["bedrooms"].isna()
        | (df["bedrooms"] < 0)
        | (df["bedrooms"] > 10)
    )

    df["bathroom_outlier"] = (
        df["bathrooms"].isna()
        | (df["bathrooms"] < 0)
        | (df["bathrooms"] > 10)
    )

    # Conservative target:
    # Addis Ababa + residential category + explicit ETB
    # + explicit monthly period + no currency conflict.
    valid_target = (
        (df["city_scope"] == "addis_ababa")
        & df["residential_category"]
        & df["price"].notna()
        & (df["price"] > 0)
        & df["mentions_etb"]
        & ~df["mentions_usd"]
        & df["mentions_monthly"]
        & ~df["mentions_daily"]
        & ~df["mentions_weekly"]
        & ~df["mentions_yearly"]
    )

    df["monthly_rent_etb"] = np.where(
        valid_target,
        df["price"],
        np.nan
    )

    # Review flags are not necessarily errors.
    df["review_required"] = (
        df["monthly_rent_etb"].isna()
        | df["currency_conflict"]
        | df["invalid_price"]
        | df["invalid_area"]
        | df["bedroom_outlier"]
        | df["bathroom_outlier"]
        | ~df["residential_category"]
        | (df["city_scope"] != "addis_ababa")
    )

    # Preserve useful fields only.
    output_columns = [
        "id",
        "title",
        "city",
        "city_scope",
        "subcity",
        "location_name",
        "category_name",
        "property_type_name",
        "price",
        "raw_price",
        "area",
        "raw_area",
        "bedrooms",
        "raw_number_of_bed_rooms",
        "bathrooms",
        "raw_number_of_baths",
        "currency_evidence",
        "period_evidence",
        "mentions_usd",
        "mentions_etb",
        "mentions_monthly",
        "mentions_daily",
        "mentions_weekly",
        "mentions_yearly",
        "currency_conflict",
        "residential_category",
        "invalid_price",
        "invalid_area",
        "bedroom_outlier",
        "bathroom_outlier",
        "monthly_rent_etb",
        "review_required"
    ]

    return df[output_columns].copy()


# Extract Beten rental records
beten_candidates = extract_beten_candidates(beten_df)

print("Beten extraction completed.")
print("Total rental records:", len(beten_candidates))
print("Unique listing IDs:", beten_candidates["id"].nunique())
print(
    "Monthly ETB candidates:",
    beten_candidates["monthly_rent_etb"].notna().sum()
)
print(
    "Records requiring review:",
    beten_candidates["review_required"].sum()
)


Beten extraction completed.
Total rental records: 41
Unique listing IDs: 41
Monthly ETB candidates: 0
Records requiring review: 41


In [183]:

print("=" * 65)
print("BETEN EXTRACTION SUMMARY")
print("=" * 65)

print("\nCITY SCOPE")
print(beten_candidates["city_scope"].value_counts(dropna=False))

print("\nPROPERTY CATEGORY")
print(beten_candidates["category_name"].value_counts(dropna=False))

print("\nCURRENCY EVIDENCE")
print(beten_candidates["currency_evidence"].value_counts())

print("\nRENTAL PERIOD EVIDENCE")
print(beten_candidates["period_evidence"].value_counts())

print("\nMONTHLY ETB TARGET")
target = beten_candidates["monthly_rent_etb"].dropna()

print("Available:", len(target))
print("Missing:", beten_candidates["monthly_rent_etb"].isna().sum())

if not target.empty:
    print(
        target.describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
    )

print("\nDATA QUALITY FLAGS")

for col in [
    "currency_conflict",
    "invalid_price",
    "invalid_area",
    "bedroom_outlier",
    "bathroom_outlier",
    "residential_category"
]:
    print(f"{col}: {beten_candidates[col].sum()}")

print("\nDUPLICATE LISTING IDS")
print(beten_candidates["id"].duplicated().sum())


BETEN EXTRACTION SUMMARY

CITY SCOPE
city_scope
addis_ababa    40
other           1
Name: count, dtype: int64

PROPERTY CATEGORY
category_name
Apartment              20
Villa Houses            7
Service House           3
Guest Houses            3
Warehouses              3
Residence Building      2
Others                  1
Condominium             1
Commercial Building     1
Name: count, dtype: int64

CURRENCY EVIDENCE
currency_evidence
unspecified      34
ETB mentioned     7
Name: count, dtype: int64

RENTAL PERIOD EVIDENCE
period_evidence
unspecified    40
monthly         1
Name: count, dtype: int64

MONTHLY ETB TARGET
Available: 0
Missing: 41

DATA QUALITY FLAGS
currency_conflict: 0
invalid_price: 0
invalid_area: 0
bedroom_outlier: 1
bathroom_outlier: 1
residential_category: 33

DUPLICATE LISTING IDS
0


In [184]:

from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# All 41 rental records
beten_candidates.to_csv(
    processed_dir / "beten_all_extracted.csv",
    index=False
)

# Records with a supported monthly ETB target
beten_candidates[
    beten_candidates["monthly_rent_etb"].notna()
].to_csv(
    processed_dir / "beten_monthly_etb_candidates.csv",
    index=False
)

# Records requiring further review
beten_candidates[
    beten_candidates["review_required"]
].to_csv(
    processed_dir / "beten_review.csv",
    index=False
)

# Addis Ababa records only, preserving all rental records
beten_candidates[
    beten_candidates["city_scope"] == "addis_ababa"
].to_csv(
    processed_dir / "beten_addis_rentals.csv",
    index=False
)

print("Beten files saved successfully.")

for file in sorted(processed_dir.glob("beten_*.csv")):
    print(file.name, "-", file.stat().st_size, "bytes")


Beten files saved successfully.
beten_addis_rentals.csv - 8528 bytes
beten_all_extracted.csv - 8708 bytes
beten_monthly_etb_candidates.csv - 429 bytes
beten_review.csv - 8708 bytes


In [185]:

import json
from pathlib import Path
from collections import Counter

loozap_path = Path(
    "data/raw/real_estate/raw/loozap/loozap_data.json"
)

with open(loozap_path, "r", encoding="utf-8") as f:
    loozap_data = json.load(f)

print("=" * 65)
print("LOOZAP SCHEMA INSPECTION")
print("=" * 65)

print("\nTOP-LEVEL TYPE:", type(loozap_data).__name__)

if isinstance(loozap_data, list):
    records = [
        item for item in loozap_data
        if isinstance(item, dict)
    ]
    print("TOTAL RECORDS:", len(loozap_data))
    print("VALID RECORDS:", len(records))
    print("INVALID RECORDS:", len(loozap_data) - len(records))

elif isinstance(loozap_data, dict):
    records = [
        item for item in loozap_data.values()
        if isinstance(item, dict)
    ]
    print("TOTAL RECORDS:", len(loozap_data))
    print("VALID RECORDS:", len(records))
    print("INVALID RECORDS:", len(loozap_data) - len(records))

else:
    records = []
    print("Unsupported top-level structure.")

if records:
    field_counts = Counter(
        key
        for record in records
        for key in record.keys()
    )

    print("\nFIELD FREQUENCY")
    for field, count in field_counts.most_common():
        print(f"{field}: {count}/{len(records)}")

    print("\nSAMPLE RECORD KEYS")
    print(list(records[0].keys()))

    print("\nSAMPLE VALUES")
    sensitive_terms = [
        "phone", "mobile", "email", "contact",
        "password", "owner", "author", "user"
    ]

    for key, value in records[0].items():

        if any(term in key.lower() for term in sensitive_terms):
            continue

        if isinstance(value, dict):
            print(f"{key}: nested keys = {list(value.keys())[:20]}")

        elif isinstance(value, list):
            print(f"{key}: list length = {len(value)}")

        else:
            print(f"{key}: {str(value)[:300]}")


LOOZAP SCHEMA INSPECTION

TOP-LEVEL TYPE: list
TOTAL RECORDS: 75213
VALID RECORDS: 75213
INVALID RECORDS: 0

FIELD FREQUENCY
url: 75213/75213
listing_type: 75213/75213
title: 75213/75213
date_published: 75213/75213
reference_number: 75213/75213
image_urls: 75213/75213
Location:: 75213/75213
Price:: 75213/75213
description: 75213/75213
additional_details: 75213/75213
features: 75213/75213
seller: 75213/75213

SAMPLE RECORD KEYS
['url', 'listing_type', 'title', 'date_published', 'reference_number', 'image_urls', 'Location:', 'Price:', 'description', 'additional_details', 'features', 'seller']

SAMPLE VALUES
url: https://et.loozap.com/ads/bg2-main-modern-house-and-bg2-service-house-for-sale-in-cmc-addis-ababa-zz/2713859.html
listing_type: House, Villas, Apartment, and Property for sale
title: B+G+2 Main Modern House and B+G+2 Service House for Sale in CMC - Addis Ababa
date_published: 2019-04-19 09:00
reference_number: 2713859
image_urls: list length = 4
Location:: Addis Ababa
Price:: Br1

In [186]:

import pandas as pd
import re

loozap_df = pd.DataFrame(loozap_data)

print("=" * 65)
print("LOOZAP VALUE AUDIT")
print("=" * 65)

# Transaction categories
print("\nLISTING TYPE DISTRIBUTION")
print(
    loozap_df["listing_type"]
    .value_counts(dropna=False)
    .head(40)
)

# Geographic distribution
print("\nLOCATION DISTRIBUTION")
print(
    loozap_df["Location:"]
    .value_counts(dropna=False)
    .head(30)
)

# Price formats
print("\nPRICE DATA TYPES")
print(loozap_df["Price:"].map(type).value_counts())

print("\nPRICE EXAMPLES")
print(
    loozap_df["Price:"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .head(40)
    .to_list()
)

# Additional details structure
print("\nADDITIONAL DETAILS TYPES")
print(
    loozap_df["additional_details"]
    .map(type)
    .value_counts()
)

# Feature structure
print("\nFEATURES TYPES")
print(
    loozap_df["features"]
    .map(type)
    .value_counts()
)

# Rental candidates based on listing type
rental_mask = (
    loozap_df["listing_type"]
    .fillna("")
    .astype(str)
    .str.contains(r"\brent(?:al)?\b", case=False, regex=True)
)

loozap_rentals = loozap_df[rental_mask].copy()

print("\nRENTAL CANDIDATES")
print("Total:", len(loozap_rentals))

print("\nRENTAL LISTING TYPE DISTRIBUTION")
print(loozap_rentals["listing_type"].value_counts())

print("\nRENTAL PRICE EXAMPLES")
print(
    loozap_rentals["Price:"]
    .value_counts(dropna=False)
    .head(30)
)

# Inspect selected rental records without seller/contact data
sample_columns = [
    "reference_number",
    "listing_type",
    "title",
    "Location:",
    "Price:",
    "description",
    "additional_details",
    "date_published"
]

print("\nSAMPLE RENTAL RECORDS")

for _, row in loozap_rentals[sample_columns].head(8).iterrows():
    print("\n" + "-" * 60)

    for col in sample_columns:
        value = row[col]

        if isinstance(value, dict):
            value = str(value)[:700]
        elif isinstance(value, str):
            value = value[:700]

        print(f"{col}: {value}")


LOOZAP VALUE AUDIT

LISTING TYPE DISTRIBUTION
listing_type
House, Villas, Apartment, and Property for sale                       47861
House, Villas, Apartment, and Property for Rent                       16059
Commercial Property, Office Space, and Premises for sale and lease     5596
Land & Lots for sale and lease                                         4773
New & Used Caravans, Campers, and Motorhomes for sale                   694
Furnished Apartments and Hotels for rent                                164
Vacation house, apartment, villas, cabins, beach houses for rent         58
Farms and Ranches for sale and lease                                      7
Rooms and Studio Apartments for Rent                                      1
Name: count, dtype: int64

LOCATION DISTRIBUTION
Location:
Addis Ababa                      27796
Bole,                            23493
Nifas Silk Lafto,                 3909
Other location,                   3462
Yeka,                             3149
Kir

In [187]:

import re
import numpy as np
import pandas as pd
from pathlib import Path

def extract_loozap_candidates(source_df):

    df = source_df.copy()

    # 1. Keep only listings explicitly classified as rentals.
    listing_type = (
        df["listing_type"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    df = df[
        listing_type.str.contains(r"\brent\b", regex=True)
    ].copy()

    # 2. Preserve raw price and parse numeric value.
    df["raw_price"] = df["Price:"].fillna("").astype(str).str.strip()

    def parse_price(value):
        cleaned = re.sub(
            r"(?i)\b(?:br|etb|birr)\b|\$|€|£|\s",
            "",
            str(value)
        )
        cleaned = cleaned.replace(",", "")

        match = re.fullmatch(r"\d+(?:\.\d+)?", cleaned)

        if not match:
            return np.nan

        return float(cleaned)

    def valid_comma_grouping(value):
        cleaned = re.sub(
            r"(?i)\b(?:br|etb|birr)\b|\$|€|£|\s",
            "",
            str(value)
        )

        if "," not in cleaned:
            return bool(re.fullmatch(r"\d+(?:\.\d+)?", cleaned))

        return bool(
            re.fullmatch(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?", cleaned)
        )

    df["price"] = df["raw_price"].apply(parse_price)

    df["price_format_valid"] = (
        df["raw_price"].apply(valid_comma_grouping)
    )

    # 3. Determine currency from the structured price field.
    raw_lower = df["raw_price"].str.lower()

    df["price_currency"] = np.select(
        [
            raw_lower.str.contains(r"\b(?:br|birr|etb)\b"),
            raw_lower.str.contains(r"\busd\b|\$")
        ],
        [
            "ETB",
            "USD"
        ],
        default="unspecified"
    )

    # 4. Description evidence.
    description = (
        df["description"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    df["description_mentions_usd"] = description.str.contains(
        r"\busd\b|\$\s*\d|\d\s*\$|\bdollars?\b",
        regex=True,
        na=False
    )

    df["description_mentions_etb"] = description.str.contains(
        r"\betb\b|\bbirr\b|\bbr\b|ብር",
        regex=True,
        na=False
    )

    df["currency_conflict"] = (
        (df["price_currency"] == "ETB")
        & df["description_mentions_usd"]
    ) | (
        (df["price_currency"] == "USD")
        & df["description_mentions_etb"]
    )

    # 5. Detect rental periods.
    monthly_pattern = (
        r"\bper month\b|"
        r"/month\b|"
        r"/mo\b|"
        r"\bmonthly\b|"
        r"\ba month\b|"
        r"በወር|ወርሃዊ"
    )

    daily_pattern = (
        r"\bper day\b|"
        r"/day\b|"
        r"\bdaily\b|"
        r"\ba day\b|"
        r"በቀን"
    )

    weekly_pattern = (
        r"\bper week\b|"
        r"/week\b|"
        r"\bweekly\b"
    )

    yearly_pattern = (
        r"\bper year\b|"
        r"/year\b|"
        r"\byearly\b|"
        r"\bannually\b"
    )

    df["mentions_monthly"] = description.str.contains(
        monthly_pattern, regex=True, na=False
    )

    df["mentions_daily"] = description.str.contains(
        daily_pattern, regex=True, na=False
    )

    df["mentions_weekly"] = description.str.contains(
        weekly_pattern, regex=True, na=False
    )

    df["mentions_yearly"] = description.str.contains(
        yearly_pattern, regex=True, na=False
    )

    period_count = (
        df["mentions_monthly"].astype(int)
        + df["mentions_daily"].astype(int)
        + df["mentions_weekly"].astype(int)
        + df["mentions_yearly"].astype(int)
    )

    df["period_evidence"] = np.select(
        [
            period_count > 1,
            df["mentions_monthly"],
            df["mentions_daily"],
            df["mentions_weekly"],
            df["mentions_yearly"]
        ],
        [
            "multiple periods",
            "monthly",
            "daily",
            "weekly",
            "yearly"
        ],
        default="unspecified"
    )

    # 6. Classify listing category.
    df["listing_group"] = np.select(
        [
            df["listing_type"].str.contains(
                "Vacation", case=False, na=False
            ),
            df["listing_type"].str.contains(
                "Furnished Apartments and Hotels",
                case=False, na=False
            ),
            df["listing_type"].str.contains(
                "Rooms and Studio",
                case=False, na=False
            )
        ],
        [
            "vacation_rental",
            "furnished_apartment_or_hotel",
            "room_or_studio"
        ],
        default="general_residential_rental"
    )

    # 7. Geographic classification.
    location = (
        df["Location:"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    addis_neighborhoods = [
        "bole", "yeka", "kirkos", "arada",
        "lideta", "gulele", "kolfe", "keranio",
        "nifas", "silk", "lafto", "akaki",
        "kaliti", "addis ketema", "megenagna",
        "ayat", "cmc", "gerji", "sarbet",
        "old airport", "piassa", "piaza",
        "mexico", "kality", "summit", "kotebe",
        "semit", "lebu", "jemo", "goro"
    ]

    has_addis_name = location.str.contains(
        r"addis ababa|addis abeba",
        regex=True,
        na=False
    )

    has_known_neighborhood = location.apply(
        lambda value: any(
            name in value for name in addis_neighborhoods
        )
    )

    is_other_location = location.str.contains(
        "other location", regex=False
    )

    df["geographic_scope"] = np.select(
        [
            has_addis_name,
            has_known_neighborhood,
            is_other_location
        ],
        [
            "addis_ababa_confirmed",
            "addis_neighborhood_inferred",
            "unknown"
        ],
        default="outside_or_unverified"
    )

    # 8. Extract bedroom and bathroom counts from descriptions.
    def extract_number(pattern, text):
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            try:
                return float(match.group(1).replace(",", ""))
            except ValueError:
                return np.nan
        return np.nan

    df["bedrooms"] = description.apply(
        lambda text: extract_number(
            r"(?:no\.?\s*of\s*bedrooms|bedrooms?)\s*:\s*(\d+(?:\.\d+)?)",
            text
        )
    )

    df["bathrooms"] = description.apply(
        lambda text: extract_number(
            r"(?:bathrooms?|toilet\s+and\s+bath)\s*:\s*(\d+(?:\.\d+)?)",
            text
        )
    )

    # Extract area while preserving its stated unit.
    def extract_area(text):
        patterns = [
            (r"square\s*feet\s*:\s*([\d,.]+)", "sq_ft"),
            (r"([\d,.]+)\s*sq\.?\s*ft\b", "sq_ft"),
            (r"([\d,.]+)\s*sqm\b", "sqm"),
            (r"([\d,.]+)\s*sq\.?\s*m\b", "sqm"),
            (r"([\d,.]+)\s*square\s*meters?\b", "sqm")
        ]

        for pattern, unit in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                try:
                    return float(match.group(1).replace(",", "")), unit
                except ValueError:
                    pass

        return np.nan, "unspecified"

    area_results = description.apply(extract_area)

    df["area"] = area_results.apply(lambda x: x[0])
    df["area_unit"] = area_results.apply(lambda x: x[1])

    # 9. Data quality flags.
    df["invalid_price"] = (
        df["price"].isna() | (df["price"] <= 0)
    )

    df["invalid_area"] = (
        df["area"].notna() & (df["area"] <= 0)
    )

    df["bedroom_outlier"] = (
        df["bedrooms"].notna()
        & ((df["bedrooms"] < 0) | (df["bedrooms"] > 10))
    )

    df["bathroom_outlier"] = (
        df["bathrooms"].notna()
        & ((df["bathrooms"] < 0) | (df["bathrooms"] > 10))
    )

    # 10. Conservative monthly ETB target.
    valid_target = (
        (df["price_currency"] == "ETB")
        & ~df["currency_conflict"]
        & df["price"].notna()
        & (df["price"] > 0)
        & df["mentions_monthly"]
        & ~df["mentions_daily"]
        & ~df["mentions_weekly"]
        & ~df["mentions_yearly"]
        & (df["geographic_scope"] != "outside_or_unverified")
        & (df["listing_group"] != "vacation_rental")
        & df["price_format_valid"]
    )

    df["monthly_rent_etb"] = np.where(
        valid_target,
        df["price"],
        np.nan
    )

    df["review_required"] = (
        df["monthly_rent_etb"].isna()
        | df["currency_conflict"]
        | df["invalid_price"]
        | ~df["price_format_valid"]
        | df["invalid_area"]
        | df["bedroom_outlier"]
        | df["bathroom_outlier"]
        | (df["geographic_scope"] == "unknown")
        | (df["geographic_scope"] == "outside_or_unverified")
        | (df["listing_group"] == "vacation_rental")
    )

    # 11. Output only non-sensitive fields.
    output_columns = [
        "reference_number",
        "url",
        "title",
        "listing_type",
        "listing_group",
        "date_published",
        "Location:",
        "geographic_scope",
        "raw_price",
        "price",
        "price_currency",
        "price_format_valid",
        "description_mentions_usd",
        "description_mentions_etb",
        "currency_conflict",
        "period_evidence",
        "mentions_monthly",
        "mentions_daily",
        "mentions_weekly",
        "mentions_yearly",
        "bedrooms",
        "bathrooms",
        "area",
        "area_unit",
        "invalid_price",
        "invalid_area",
        "bedroom_outlier",
        "bathroom_outlier",
        "monthly_rent_etb",
        "review_required"
    ]

    return df[output_columns].copy()


# Run extraction
loozap_candidates = extract_loozap_candidates(loozap_df)

print("Loozap extraction completed.")
print("Total rental records:", len(loozap_candidates))
print("Unique listing IDs:", loozap_candidates["reference_number"].nunique())
print(
    "Monthly ETB candidates:",
    loozap_candidates["monthly_rent_etb"].notna().sum()
)
print(
    "Records requiring review:",
    loozap_candidates["review_required"].sum()
)


Loozap extraction completed.
Total rental records: 16282
Unique listing IDs: 16282
Monthly ETB candidates: 0
Records requiring review: 16282


In [188]:

print("=" * 65)
print("LOOZAP EXTRACTION SUMMARY")
print("=" * 65)

print("\nTOTAL RENTAL RECORDS:", len(loozap_candidates))

print("\nLISTING GROUPS")
print(loozap_candidates["listing_group"].value_counts(dropna=False))

print("\nGEOGRAPHIC SCOPE")
print(loozap_candidates["geographic_scope"].value_counts(dropna=False))

print("\nPRICE CURRENCY")
print(loozap_candidates["price_currency"].value_counts(dropna=False))

print("\nPRICE FORMAT")
print(loozap_candidates["price_format_valid"].value_counts(dropna=False))

print("\nRENTAL PERIOD")
print(loozap_candidates["period_evidence"].value_counts(dropna=False))

print("\nCURRENCY CONFLICTS")
print(loozap_candidates["currency_conflict"].sum())

print("\nMONTHLY ETB TARGET")
target = loozap_candidates["monthly_rent_etb"].dropna()

print("Available:", len(target))
print("Missing:", loozap_candidates["monthly_rent_etb"].isna().sum())

if not target.empty:
    print(
        target.describe(
            percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
        )
    )

print("\nDATA QUALITY FLAGS")

for col in [
    "invalid_price",
    "price_format_valid",
    "invalid_area",
    "bedroom_outlier",
    "bathroom_outlier"
]:
    print(f"{col}: {loozap_candidates[col].value_counts(dropna=False).to_dict()}")

print("\nDUPLICATE REFERENCE NUMBERS")
print(loozap_candidates["reference_number"].duplicated().sum())


LOOZAP EXTRACTION SUMMARY

TOTAL RENTAL RECORDS: 16282

LISTING GROUPS
listing_group
general_residential_rental      16059
furnished_apartment_or_hotel      164
vacation_rental                    58
room_or_studio                      1
Name: count, dtype: int64

GEOGRAPHIC SCOPE
geographic_scope
addis_neighborhood_inferred    10350
addis_ababa_confirmed           5484
unknown                          285
outside_or_unverified            163
Name: count, dtype: int64

PRICE CURRENCY
price_currency
unspecified    16282
Name: count, dtype: int64

PRICE FORMAT
price_format_valid
False    16282
Name: count, dtype: int64

RENTAL PERIOD
period_evidence
unspecified         15277
monthly               749
multiple periods      139
daily                 112
yearly                  3
weekly                  2
Name: count, dtype: int64

CURRENCY CONFLICTS
0

MONTHLY ETB TARGET
Available: 0
Missing: 16282

DATA QUALITY FLAGS
invalid_price: {True: 16282}
price_format_valid: {False: 16282}
invalid_a

In [189]:

from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# All extracted rental records
loozap_candidates.to_csv(
    processed_dir / "loozap_all_extracted.csv",
    index=False
)

# Monthly ETB candidates
loozap_candidates[
    loozap_candidates["monthly_rent_etb"].notna()
].to_csv(
    processed_dir / "loozap_monthly_etb_candidates.csv",
    index=False
)

# Records requiring review
loozap_candidates[
    loozap_candidates["review_required"]
].to_csv(
    processed_dir / "loozap_review.csv",
    index=False
)

# Duplicate reference numbers for investigation
loozap_candidates[
    loozap_candidates["reference_number"].duplicated(keep=False)
].sort_values("reference_number").to_csv(
    processed_dir / "loozap_duplicate_audit.csv",
    index=False
)

print("Loozap files saved successfully.")

for file in sorted(processed_dir.glob("loozap_*.csv")):
    print(f"{file.name}: {file.stat().st_size:,} bytes")


Loozap files saved successfully.
loozap_all_extracted.csv: 6,758,478 bytes
loozap_duplicate_audit.csv: 421 bytes
loozap_monthly_etb_candidates.csv: 421 bytes
loozap_review.csv: 6,758,478 bytes


In [190]:

from pathlib import Path
import pandas as pd

processed_dir = Path("data/processed")

csv_files = sorted(processed_dir.glob("*.csv"))

print(f"Total CSV files: {len(csv_files)}")

for file in csv_files:
    try:
        df = pd.read_csv(file)
        print("\n" + "=" * 65)
        print("FILE:", file.name)
        print("ROWS:", len(df))
        print("COLUMNS:", list(df.columns))
    except Exception as e:
        print(f"\nCould not read {file.name}: {e}")


Total CSV files: 33

FILE: afrotie_all_extracted.csv
ROWS: 2419
COLUMNS: ['listing_id', 'source_file', 'location', 'hd', 'price', 'area', 'bedrooms', 'bathrooms', 'raw_price', 'raw_area', 'raw_bedrooms', 'raw_bathrooms', 'currency_evidence', 'period_evidence', 'mentions_usd', 'mentions_etb', 'currency_conflict', 'commercial_terms', 'hospitality_terms', 'mixed_use_terms', 'invalid_price', 'invalid_area', 'bedroom_outlier', 'bathroom_outlier', 'description_duplicate_count', 'monthly_rent_etb', 'review_required']

FILE: afrotie_monthly_etb_candidates.csv
ROWS: 28
COLUMNS: ['listing_id', 'source_file', 'location', 'hd', 'price', 'area', 'bedrooms', 'bathrooms', 'raw_price', 'raw_area', 'raw_bedrooms', 'raw_bathrooms', 'currency_evidence', 'period_evidence', 'mentions_usd', 'mentions_etb', 'currency_conflict', 'commercial_terms', 'hospitality_terms', 'mixed_use_terms', 'invalid_price', 'invalid_area', 'bedroom_outlier', 'bathroom_outlier', 'description_duplicate_count', 'monthly_rent_etb', 

In [191]:

from pathlib import Path
import pandas as pd

processed_dir = Path("data/processed")

candidate_files = {
    "afrotie": "afrotie_monthly_etb_candidates.csv",
    "livingethio": "livingethio_monthly_etb_candidates.csv",
    "qefira": "qefira_monthly_etb_candidates.csv",
    "realethio": "realethio_monthly_etb_candidates.csv",
    "zegebeya": "zegebeya_monthly_etb_candidates.csv",
}

audit_results = []

for source, filename in candidate_files.items():
    path = processed_dir / filename
    df = pd.read_csv(path)

    target_col = next(
        (c for c in ["monthly_rent_etb", "monthly_rent"] if c in df.columns),
        None
    )

    id_col = next(
        (c for c in ["listing_id", "id", "reference_number"] if c in df.columns),
        None
    )

    target = pd.to_numeric(df[target_col], errors="coerce") if target_col else pd.Series(dtype=float)

    audit_results.append({
        "source": source,
        "rows": len(df),
        "target_column": target_col,
        "valid_targets": int((target > 0).sum()),
        "missing_targets": int(target.isna().sum()),
        "nonpositive_targets": int((target <= 0).sum()),
        "duplicate_ids": int(df[id_col].duplicated().sum()) if id_col else None,
        "id_column": id_col
    })

audit_df = pd.DataFrame(audit_results)
display(audit_df)

# Check whether the validated Jiji dataset still exists in memory.
if "jiji_model" in globals():
    print("\nJIJI MODEL FOUND")
    print("Rows:", len(jiji_model))
    print("Columns:", list(jiji_model.columns))
    print("\nTarget summary:")
    print(jiji_model["monthly_rent"].describe())
else:
    print("\nJIJI MODEL NOT FOUND IN MEMORY.")
    print("We will need to reload or recreate it from the saved Jiji data.")


,source,rows,target_column,valid_targets,missing_targets,nonpositive_targets,duplicate_ids,id_column
0,afrotie,28,monthly_rent_etb,28,0,0,0,listing_id
1,livingethio,14,monthly_rent,14,0,0,0,listing_id
2,qefira,59,monthly_rent,59,0,0,0,listing_id
3,realethio,142,monthly_rent,142,0,0,0,listing_id
4,zegebeya,106,monthly_rent,106,0,0,0,listing_id



JIJI MODEL FOUND
Rows: 1939
Columns: ['source', 'listing_id', 'title', 'property_type', 'address', 'size_sqm', 'bedrooms', 'bathrooms', 'condition', 'furnishing', 'raw_price', 'raw_price_title', 'monthly_rent', 'currency', 'price_period', 'description_x', 'date_published', 'description_mentions_usd', 'description_mentions_etb', 'usd_mentions', 'etb_mentions', 'description_y', 'region_name', 'region_text', 'estate_name']

Target summary:
count      1939.000000
mean      90893.765859
std       74559.658135
min         900.000000
25%       40000.000000
50%       70000.000000
75%      120000.000000
max      840000.000000
Name: monthly_rent, dtype: float64


In [192]:

from pathlib import Path
import pandas as pd
import numpy as np

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Helper functions
# --------------------------------------------------

def get_col(df, possible_names, default=None):
    """Return the first available column from a list of names."""
    for col in possible_names:
        if col in df.columns:
            return df[col]
    return pd.Series(default, index=df.index)


def standardize_source(df, source):
    """Convert a source-specific DataFrame to the common schema."""

    result = pd.DataFrame(index=df.index)

    result["source"] = source

    result["listing_id"] = get_col(
        df, ["listing_id", "id", "reference_number"]
    ).astype("string")

    result["title"] = get_col(df, ["title"])

    result["property_type"] = get_col(
        df, ["property_type", "category_name", "hd"]
    )

    result["location"] = get_col(
        df, [
            "address",
            "location",
            "location_name",
            "Location:",
            "region_text"
        ]
    )

    # Numeric property features
    result["bedrooms"] = pd.to_numeric(
        get_col(df, ["bedrooms", "raw_bedrooms", "raw_number_of_bed_rooms"]),
        errors="coerce"
    )

    result["bathrooms"] = pd.to_numeric(
        get_col(df, ["bathrooms", "raw_bathrooms", "raw_number_of_baths"]),
        errors="coerce"
    )

    # Area: preserve the original unit
    result["area"] = pd.to_numeric(
        get_col(df, ["size_sqm", "size_value", "area"]),
        errors="coerce"
    )

    result["area_unit"] = get_col(
        df, ["size_unit", "area_unit"]
    )

    # Explicit unit handling for sources whose area is known to be sqm
    sqm_sources = ["jiji", "livingethio", "realethio"]
    if source in sqm_sources:
        result["area_unit"] = "sqm"

    # Afrotie area unit was not reliably established
    if source == "afrotie":
        result["area_unit"] = "unknown"

    result["area_unit"] = result["area_unit"].fillna("unknown")

    # Target
    target_col = (
        "monthly_rent_etb"
        if "monthly_rent_etb" in df.columns
        else "monthly_rent"
    )

    result["monthly_rent_etb"] = pd.to_numeric(
        get_col(df, [target_col]),
        errors="coerce"
    )

    # Optional additional features
    result["furnished"] = get_col(df, ["furnished"])
    result["condition"] = get_col(df, ["condition"])
    result["date_published"] = get_col(df, ["date_published"])

    result["latitude"] = pd.to_numeric(
        get_col(df, ["latitude"]), errors="coerce"
    )

    result["longitude"] = pd.to_numeric(
        get_col(df, ["longitude"]), errors="coerce"
    )

    return result.reset_index(drop=True)


# --------------------------------------------------
# 2. Load and standardize Jiji
# --------------------------------------------------

if "jiji_model" not in globals():
    raise ValueError(
        "jiji_model is not available. Reload or recreate it first."
    )

datasets = [
    standardize_source(jiji_model, "jiji")
]


# --------------------------------------------------
# 3. Load and standardize other sources
# --------------------------------------------------

candidate_files = {
    "afrotie": "afrotie_monthly_etb_candidates.csv",
    "livingethio": "livingethio_monthly_etb_candidates.csv",
    "qefira": "qefira_monthly_etb_candidates.csv",
    "realethio": "realethio_monthly_etb_candidates.csv",
    "zegebeya": "zegebeya_monthly_etb_candidates.csv",
}

for source, filename in candidate_files.items():

    path = processed_dir / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    source_df = pd.read_csv(path)

    standardized = standardize_source(source_df, source)
    datasets.append(standardized)


# --------------------------------------------------
# 4. Combine datasets
# --------------------------------------------------

training_df = pd.concat(datasets, ignore_index=True)

# Remove rows without valid target values
training_df["monthly_rent_etb"] = pd.to_numeric(
    training_df["monthly_rent_etb"], errors="coerce"
)

training_df = training_df[
    training_df["monthly_rent_etb"].notna()
    & (training_df["monthly_rent_etb"] > 0)
].copy()

# Remove records without a source-specific listing ID
training_df = training_df[
    training_df["listing_id"].notna()
    & (training_df["listing_id"].str.strip() != "")
].copy()

# Create a globally unique identifier.
training_df["record_id"] = (
    training_df["source"] + "_" + training_df["listing_id"].astype(str)
)

# Keep a consistent column order
columns = [
    "record_id",
    "source",
    "listing_id",
    "title",
    "property_type",
    "location",
    "bedrooms",
    "bathrooms",
    "area",
    "area_unit",
    "furnished",
    "condition",
    "date_published",
    "latitude",
    "longitude",
    "monthly_rent_etb"
]

training_df = training_df[columns]

# --------------------------------------------------
# 5. Initial validation
# --------------------------------------------------

print("=" * 65)
print("CONSOLIDATED DATASET SUMMARY")
print("=" * 65)

print("Total records:", len(training_df))
print("Total columns:", len(training_df.columns))
print("Duplicate record IDs:", training_df["record_id"].duplicated().sum())
print("Missing target values:", training_df["monthly_rent_etb"].isna().sum())
print("Non-positive targets:", (training_df["monthly_rent_etb"] <= 0).sum())

print("\nRECORDS PER SOURCE")
print(training_df["source"].value_counts())

print("\nTARGET SUMMARY")
print(training_df["monthly_rent_etb"].describe())

print("\nMISSING VALUES")
print(training_df.isna().sum())

# --------------------------------------------------
# 6. Save consolidated dataset
# --------------------------------------------------

output_path = processed_dir / "addis_ababa_rental_training_dataset.csv"

training_df.to_csv(output_path, index=False)

print("\nSaved to:", output_path)


CONSOLIDATED DATASET SUMMARY
Total records: 2288
Total columns: 16
Duplicate record IDs: 0
Missing target values: 0
Non-positive targets: 0

RECORDS PER SOURCE
source
jiji           1939
realethio       142
zegebeya        106
qefira           59
afrotie          28
livingethio      14
Name: count, dtype: int64

TARGET SUMMARY
count    2.288000e+03
mean     9.825724e+04
std      8.404023e+04
min      9.000000e+02
25%      4.200000e+04
50%      8.000000e+04
75%      1.300000e+05
max      1.300000e+06
Name: monthly_rent_etb, dtype: float64

MISSING VALUES
record_id              0
source                 0
listing_id             0
title                 28
property_type         20
location              12
bedrooms               2
bathrooms             15
area                  60
area_unit              0
furnished           2132
condition            349
date_published        28
latitude            2040
longitude           2040
monthly_rent_etb       0
dtype: int64

Saved to: data/processed/a

In [194]:

import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher
from itertools import combinations

df = training_df.copy().reset_index(drop=True)

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    return re.sub(r"\s+", " ", value).strip()

df["normalized_title"] = df["title"].map(normalize_text)
df["normalized_location"] = df["location"].map(normalize_text)

# Only compare records from different sources
pairs = []

for source_1, source_2 in combinations(df["source"].unique(), 2):

    first_group = df[df["source"] == source_1]
    second_group = df[df["source"] == source_2]

    for i, first in first_group.iterrows():

        if not first["normalized_title"]:
            continue

        for j, second in second_group.iterrows():

            if not second["normalized_title"]:
                continue

            title_similarity = SequenceMatcher(
                None,
                first["normalized_title"],
                second["normalized_title"]
            ).ratio()

            location_similarity = SequenceMatcher(
                None,
                first["normalized_location"],
                second["normalized_location"]
            ).ratio()

            if title_similarity < 0.75:
                continue

            price_1 = first["monthly_rent_etb"]
            price_2 = second["monthly_rent_etb"]

            price_ratio = min(price_1, price_2) / max(price_1, price_2)

            bedrooms_match = (
                pd.notna(first["bedrooms"])
                and pd.notna(second["bedrooms"])
                and first["bedrooms"] == second["bedrooms"]
            )

            area_match = False
            area_ratio = np.nan

            if (
                pd.notna(first["area"])
                and pd.notna(second["area"])
                and first["area_unit"] == second["area_unit"]
                and first["area"] > 0
                and second["area"] > 0
            ):
                area_ratio = min(
                    first["area"], second["area"]
                ) / max(first["area"], second["area"])

                area_match = area_ratio >= 0.85

            pairs.append({
                "source_1": source_1,
                "listing_id_1": first["listing_id"],
                "title_1": first["title"],
                "location_1": first["location"],
                "rent_1": price_1,

                "source_2": source_2,
                "listing_id_2": second["listing_id"],
                "title_2": second["title"],
                "location_2": second["location"],
                "rent_2": price_2,

                "title_similarity": round(title_similarity, 3),
                "location_similarity": round(location_similarity, 3),
                "price_ratio": round(price_ratio, 3),
                "bedrooms_match": bedrooms_match,
                "area_ratio": round(area_ratio, 3)
                    if pd.notna(area_ratio) else np.nan,
                "area_match": area_match
            })

duplicate_candidates = pd.DataFrame(pairs)

if not duplicate_candidates.empty:

    duplicate_candidates["review_priority"] = (
        duplicate_candidates["bedrooms_match"].astype(int)
        + duplicate_candidates["area_match"].astype(int)
        + (duplicate_candidates["price_ratio"] >= 0.90).astype(int)
        + (duplicate_candidates["location_similarity"] >= 0.80).astype(int)
    )

    duplicate_candidates = duplicate_candidates.sort_values(
        ["review_priority", "title_similarity"],
        ascending=False
    )

    duplicate_candidates.to_csv(
        "data/processed/cross_source_duplicate_candidates.csv",
        index=False
    )

print("CROSS-SOURCE DUPLICATE AUDIT")
print("=" * 60)

print("Potential matching pairs:", len(duplicate_candidates))

if not duplicate_candidates.empty:
    print("\nSource combinations:")
    print(
        duplicate_candidates.groupby(
            ["source_1", "source_2"]
        ).size().sort_values(ascending=False)
    )

    print("\nHighest-priority candidates:")
    display(duplicate_candidates.head(30))

else:
    print("No potential matches found.")


CROSS-SOURCE DUPLICATE AUDIT
Potential matching pairs: 1095

Source combinations:
source_1     source_2   
jiji         livingethio    390
             qefira         345
realethio    zegebeya       261
jiji         zegebeya        49
livingethio  qefira          39
qefira       zegebeya        11
dtype: int64

Highest-priority candidates:


,source_1,listing_id_1,title_1,location_1,rent_1,source_2,listing_id_2,title_2,location_2,rent_2,title_similarity,location_similarity,price_ratio,bedrooms_match,area_ratio,area_match,review_priority
1065,realethio,RE5845,"Hayahulet,Three Bedroom Furnished Apartment fo...","Hayahulet 1, Equatorial Guinea Street, Megenag...",120000.0,zegebeya,ZG-13974-property,3 Bedroom Furnished Apartment For Rent In Addi...,"Addis Ababa, Bole",118000.0,0.803,0.280,0.983,True,0.964,True,3
47,jiji,750018,"Furnished 3bdrm Apartment in አድስ አበባ, Yeka for...",መገናኛ አከባቢ,80000.0,livingethio,1124,Furnished Apartment for Rent,NaN,85000.0,0.800,1.000,0.941,True,NaN,False,3
111,jiji,1192952,Furnished 3bdrm Apartment in Bole for rent,መመገናኛ,90000.0,livingethio,1124,Furnished Apartment for Rent,NaN,85000.0,0.800,1.000,0.944,True,NaN,False,3
131,jiji,483673,"Furnished 3bdrm Apartment in አድስ አበባ, Yeka for...",ሲግናል,90000.0,livingethio,1124,Furnished Apartment for Rent,NaN,85000.0,0.800,1.000,0.944,True,NaN,False,3
144,jiji,302715,"Furnished 3bdrm Apartment in አአ, Bole for Rent",ሰሚት ፍየል ቤት,80000.0,livingethio,1124,Furnished Apartment for Rent,NaN,85000.0,0.800,1.000,0.941,True,NaN,False,3
945,realethio,RE6533,"Megenagna, Three bedrooms Furnished Apartment ...","Megenagna, Yeka, Addis Ababa, 2310, Ethiopia",120000.0,zegebeya,ZG-13974-property,3 Bedroom Furnished Apartment For Rent In Addi...,"Addis Ababa, Bole",118000.0,0.797,0.464,0.983,True,1.000,True,3
860,realethio,RE6788,"Bole Olympia, Two bedrooms Furnished Apartmen...","Africa Avenue / Bole Road, Olympia, Kirkos, Ad...",80000.0,zegebeya,ZG-13047-property,2 Bedroom Furnished Apartment For Rent In Addi...,"Atlas, ጋና መንገድ, Ghiliffalegn Stream, Bole, Add...",80000.0,0.790,0.550,1.000,True,1.000,True,3
861,realethio,RE6788,"Bole Olympia, Two bedrooms Furnished Apartmen...","Africa Avenue / Bole Road, Olympia, Kirkos, Ad...",80000.0,zegebeya,ZG-13619-property,2 Bedroom Furnished Apartment For Rent In Addi...,"Addis Ababa, Bole",80000.0,0.790,0.325,1.000,True,1.000,True,3
28,jiji,582788,"Furnished 3bdrm Apartment in አድስ አበባ, Kirkos f...",ካሳቺስ አከባቢ,83000.0,livingethio,1124,Furnished Apartment for Rent,NaN,85000.0,0.778,1.000,0.976,True,NaN,False,3
844,realethio,RE6805,"3 bedrooms Apartment for Rent, Mekanisa, Addis...","Mekanisa, Nefas Silk, Addis Ababa, 2506, Ethiopia",70000.0,zegebeya,ZG-14181-property,3 Bedroom Furnished Apartment For Rent In Addi...,"ቤለር, ንግስት ኤልሳቤጥ መንገድ, Amist Kilo, Arat Kilo, A...",75000.0,0.775,0.619,0.933,True,0.907,True,3
